In [ ]:
# ============================================================
# 05.0 独立恢复研究上下文
# 动态 CSI 800 + 月末信号日 + 无前视研究掩码
# ============================================================

from pathlib import Path
import json

import pandas as pd
import matplotlib.pyplot as plt

from rqdatac import (
    index_components,
    get_trading_dates,
    get_next_trading_date,
    instruments,
    is_st_stock,
    is_suspended,
)


# ---------- 1. 项目路径 ----------
PROJECT_NAME = "a-share-profitability-factor-replication"

project_candidates = [
    path
    for path in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    if path.name == PROJECT_NAME
]

if not project_candidates:
    raise RuntimeError(
        "未找到项目根目录。请确认当前 Notebook 位于 "
        "a-share-profitability-factor-replication/notebooks/ 下。"
    )

PROJECT_DIR = project_candidates[-1]
TABLE_DIR = PROJECT_DIR / "artifacts" / "tables"
META_DIR = PROJECT_DIR / "artifacts" / "metadata"
FIG_DIR = PROJECT_DIR / "artifacts" / "figures"

for directory in [TABLE_DIR, META_DIR, FIG_DIR]:
    directory.mkdir(parents=True, exist_ok=True)


# ---------- 2. 固定研究设定 ----------
INDEX_CODE = "000906.XSHG"
START_DATE = "2016-01-04"
END_DATE = "2025-12-31"
NEW_STOCK_TRADING_DAYS = 252


# ---------- 3. 动态 CSI 800 成分股 ----------
daily_dates = pd.DatetimeIndex(
    pd.to_datetime(
        get_trading_dates(START_DATE, END_DATE)
    )
)

component_map = index_components(
    INDEX_CODE,
    start_date=START_DATE,
    end_date=END_DATE,
)

membership_on_change_dates = pd.DataFrame(
    {
        pd.Timestamp(date): dict.fromkeys(stocks, True)
        for date, stocks in component_map.items()
    }
).T

membership_on_change_dates.index = pd.to_datetime(
    membership_on_change_dates.index
)

membership_on_change_dates = (
    membership_on_change_dates
    .sort_index()
    .fillna(False)
    .astype(bool)
)

# 只把已知成分变更向未来交易日延续，不使用未来成分信息
index_fix = (
    membership_on_change_dates
    .reindex(daily_dates)
    .ffill()
    .fillna(False)
    .astype(bool)
)

stock_list = index_fix.columns.tolist()


# ---------- 4. 每月最后一个交易日作为信号日 ----------
signal_dates = pd.DatetimeIndex(
    pd.Series(daily_dates)
    .groupby(daily_dates.to_period("M"))
    .max()
    .tolist()
)

membership_on_signal = (
    index_fix
    .reindex(index=signal_dates, columns=stock_list)
    .fillna(False)
    .astype(bool)
)


# ---------- 5. 新股过滤：信号日已上市满 252 个交易日 ----------
listing_eligible_date = {}

for sequence, stock in enumerate(stock_list, start=1):
    listed_date = pd.Timestamp(instruments(stock).listed_date)

    listing_eligible_date[stock] = pd.Timestamp(
        get_next_trading_date(
            listed_date,
            n=NEW_STOCK_TRADING_DAYS,
        )
    )

    if sequence % 250 == 0 or sequence == len(stock_list):
        print(
            "已处理 {}/{} 只股票的上市日期".format(
                sequence,
                len(stock_list),
            )
        )

new_stock_ready = pd.DataFrame(
    {
        stock: signal_dates >= eligible_date
        for stock, eligible_date
        in listing_eligible_date.items()
    },
    index=signal_dates,
).reindex(columns=stock_list).fillna(False).astype(bool)


# ---------- 6. 仅按信号日 T 的 ST、停牌状态过滤 ----------
st_on_t = (
    is_st_stock(stock_list, START_DATE, END_DATE)
    .reindex(index=signal_dates, columns=stock_list)
    .fillna(False)
    .astype(bool)
)

suspended_on_t = (
    is_suspended(stock_list, START_DATE, END_DATE)
    .reindex(index=signal_dates, columns=stock_list)
    .fillna(False)
    .astype(bool)
)

research_mask = (
    membership_on_signal
    & new_stock_ready
    & (~st_on_t)
    & (~suspended_on_t)
)


# ---------- 7. 只保存汇总审计，不保存股票名单或掩码 ----------
context_summary = pd.DataFrame(
    {
        "csi800_members": membership_on_signal.sum(axis=1),
        "after_new_stock_filter": (
            membership_on_signal & new_stock_ready
        ).sum(axis=1),
        "st_excluded": (
            membership_on_signal & st_on_t
        ).sum(axis=1),
        "suspended_excluded": (
            membership_on_signal & suspended_on_t
        ).sum(axis=1),
        "research_eligible": research_mask.sum(axis=1),
    }
)

context_summary.index.name = "signal_date"

table_path = TABLE_DIR / "05_research_context_summary.csv"
context_summary.to_csv(
    table_path,
    encoding="utf-8-sig",
)

metadata = {
    "index_code": INDEX_CODE,
    "index_name": "CSI 800",
    "research_period": {
        "start_date": START_DATE,
        "end_date": END_DATE,
    },
    "signal_frequency": "monthly last trading day",
    "new_stock_rule": (
        "Exclude stocks listed for fewer than 252 trading days "
        "at signal date T."
    ),
    "st_rule": "Exclude stocks that are ST at signal date T.",
    "suspension_rule": (
        "Exclude stocks suspended at signal date T."
    ),
    "future_execution_filter": (
        "T+1 tradability, limit-up and other execution constraints "
        "are not used in research-stage factor testing."
    ),
    "raw_membership_mask_saved_to_repository": False,
}

metadata_path = META_DIR / "05_research_context_rules.json"

with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(
        metadata,
        file,
        ensure_ascii=False,
        indent=2,
    )


# ---------- 8. 图 ----------
plt.figure(figsize=(12, 5))

plt.plot(
    context_summary.index,
    context_summary["research_eligible"],
    linewidth=1.6,
    label="research_eligible",
)

plt.plot(
    context_summary.index,
    context_summary["csi800_members"],
    linewidth=1.2,
    label="csi800_members",
)

plt.title("CSI 800 Research Universe for TTM Profitability Factors")
plt.xlabel("Signal Date")
plt.ylabel("Stock Count")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()

figure_path = FIG_DIR / "05_research_context_coverage.png"
plt.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()


# ---------- 9. 输出 ----------
print("\n项目根目录：", PROJECT_DIR)
print("信号日期数：", len(signal_dates))
print(
    "动态 CSI 800 成分股范围：{} - {}".format(
        int(membership_on_signal.sum(axis=1).min()),
        int(membership_on_signal.sum(axis=1).max()),
    )
)
print(
    "可研究股票范围：{} - {}".format(
        int(research_mask.sum(axis=1).min()),
        int(research_mask.sum(axis=1).max()),
    )
)
print("已保存：", table_path)
print("已保存：", metadata_path)
print("已保存：", figure_path)

In [ ]:
# 05.0 Notebook 启动单元：每次重启后先运行
from pathlib import Path
import importlib
import sys

PROJECT_ROOT = Path("/home/rice/notebook/a-share-profitability-factor-replication")
LIB_DIR = PROJECT_ROOT / "lib"

if str(LIB_DIR) not in sys.path:
    sys.path.insert(0, str(LIB_DIR))

import profitability_research_utils as pru
import profitability_diagnostics as pdiag

pru = importlib.reload(pru)
pdiag = importlib.reload(pdiag)

PATHS = pru.ProjectPaths.from_root(PROJECT_ROOT)
PATHS.ensure_directories()
PATHS.ensure_cache_ignored()

print("05 Notebook 基础库已就绪。")
print("通用库版本：", pru.LIBRARY_VERSION)
print("盈利诊断库版本：", pdiag.DIAGNOSTICS_VERSION)

In [ ]:
# ============================================================
# 05.1 提取 ROA、ROIC、RONOA、FCFFIC 的 TTM 输入
# 不保存股票级原始数值，只保存覆盖率与字段审计
# ============================================================

import numpy as np

from rqfactor import Factor, execute_factor


# ---------- 1. 研报公式所需输入 ----------
INPUT_SPECS = [
    {
        "input_name": "market_cap",
        "component_name": "总市值",
        "factor_modules": "市值中性化",
        "necessity": "required",
        "candidates": ["market_cap_3"],
    },
    {
        "input_name": "roa_vendor",
        "component_name": "供应商 ROA TTM",
        "factor_modules": "ROA 交叉核验",
        "necessity": "reference",
        "candidates": ["return_on_asset_ttm"],
    },
    {
        "input_name": "net_profit_ttm",
        "component_name": "归母净利润 TTM",
        "factor_modules": "ROA 重构",
        "necessity": "required",
        "candidates": ["net_profit_parent_company_ttm_0"],
    },
    {
        "input_name": "total_assets_mrq",
        "component_name": "总资产 MRQ",
        "factor_modules": "ROA 重构",
        "necessity": "required",
        "candidates": ["total_assets_mrq_0"],
    },
    {
        "input_name": "roic_vendor",
        "component_name": "供应商 ROIC TTM",
        "factor_modules": "ROIC 交叉核验",
        "necessity": "reference",
        "candidates": ["return_on_invested_capital_ttm"],
    },
    {
        "input_name": "ebit_ttm",
        "component_name": "EBIT TTM",
        "factor_modules": "ROIC 重构",
        "necessity": "required",
        "candidates": ["ebit_ttm_0", "ebit_ttm", "ebit"],
    },
    {
        "input_name": "income_tax_ttm",
        "component_name": "所得税费用 TTM",
        "factor_modules": "ROIC 重构",
        "necessity": "required",
        "candidates": ["income_tax_ttm_0"],
    },
    {
        "input_name": "total_profit_ttm",
        "component_name": "利润总额 TTM",
        "factor_modules": "ROIC 重构",
        "necessity": "required",
        "candidates": ["total_profit_ttm_0", "profit_before_tax_ttm_0"],
    },
    {
        "input_name": "total_equity_mrq",
        "component_name": "总权益 MRQ",
        "factor_modules": "ROIC、RONOA、FCFFIC",
        "necessity": "required",
        "candidates": ["total_equity_mrq_0"],
    },
    {
        "input_name": "interest_bearing_debt",
        "component_name": "有息负债",
        "factor_modules": "ROIC、RONOA、FCFFIC",
        "necessity": "required",
        "candidates": ["interest_bearing_debt"],
    },
    {
        "input_name": "cash_equivalent",
        "component_name": "现金及现金等价物",
        "factor_modules": "ROIC、RONOA、FCFFIC",
        "necessity": "required",
        "candidates": ["cash_equivalent"],
    },
    {
        "input_name": "operating_profit_ttm",
        "component_name": "营业利润 TTM",
        "factor_modules": "RONOA 重构",
        "necessity": "required",
        "candidates": ["profit_from_operation_ttm_0"],
    },
    {
        "input_name": "financial_expense_ttm",
        "component_name": "财务费用 TTM",
        "factor_modules": "RONOA 重构",
        "necessity": "required",
        "candidates": ["financial_expenseTTM", "financial_expense_ttm_0"],
    },
    {
        "input_name": "investment_income_ttm",
        "component_name": "投资收益 TTM",
        "factor_modules": "RONOA 重构",
        "necessity": "required",
        "candidates": ["investment_income_ttm_0"],
    },
    {
        "input_name": "fair_value_change_ttm",
        "component_name": "公允价值变动收益 TTM",
        "factor_modules": "RONOA 重构",
        "necessity": "required",
        "candidates": ["fair_value_change_income_ttm_0"],
    },
    {
        "input_name": "fixed_deposits_mrq",
        "component_name": "定期存款 MRQ",
        "factor_modules": "RONOA 扩展金融资产",
        "necessity": "optional",
        "candidates": ["fixed_deposits_mrq_0"],
    },
    {
        "input_name": "trading_financial_assets_mrq",
        "component_name": "交易性金融资产 MRQ",
        "factor_modules": "RONOA 扩展金融资产",
        "necessity": "optional",
        "candidates": ["financial_asset_held_for_trading_mrq_0"],
    },
    {
        "input_name": "available_for_sale_assets_mrq",
        "component_name": "可供出售金融资产 MRQ",
        "factor_modules": "RONOA 扩展金融资产",
        "necessity": "optional",
        "candidates": ["financial_asset_available_for_sale_mrq_0"],
    },
    {
        "input_name": "hold_to_maturity_assets_mrq",
        "component_name": "持有至到期投资 MRQ",
        "factor_modules": "RONOA 扩展金融资产",
        "necessity": "optional",
        "candidates": ["financial_asset_hold_to_maturity_mrq_0"],
    },
    {
        "input_name": "dividend_receivable_mrq",
        "component_name": "应收股利 MRQ",
        "factor_modules": "RONOA 扩展金融资产",
        "necessity": "optional",
        "candidates": ["dividend_receivable_mrq_0"],
    },
    {
        "input_name": "cfo_ttm",
        "component_name": "经营活动现金流 TTM",
        "factor_modules": "FCFFIC 重构",
        "necessity": "required",
        "candidates": ["cash_flow_from_operating_activities_ttm_0"],
    },
    {
        "input_name": "capex_ttm",
        "component_name": "资本开支 TTM",
        "factor_modules": "FCFFIC 重构",
        "necessity": "required",
        "candidates": ["cash_paid_for_asset_ttm_0"],
    },
]


# ---------- 2. 单字段提取与信号日对齐 ----------
def fetch_factor_panel(rqfactor_field):
    raw_panel = execute_factor(
        Factor(rqfactor_field),
        stock_list,
        START_DATE,
        END_DATE,
    ).copy()

    raw_panel.index = pd.to_datetime(raw_panel.index)
    raw_panel.columns = raw_panel.columns.astype(str)

    return raw_panel.reindex(
        index=signal_dates,
        columns=stock_list,
    )


# ---------- 3. 逐字段查询 ----------
factor_inputs = {}
selected_input_fields = {}
coverage_rows = []
query_rows = []

for sequence, spec in enumerate(INPUT_SPECS, start=1):
    input_name = spec["input_name"]

    print(
        "正在提取 {}/{}：{} ({})".format(
            sequence,
            len(INPUT_SPECS),
            input_name,
            spec["component_name"],
        )
    )

    selected_panel = None
    selected_field = None
    selected_valid_count = 0

    for candidate_rank, rqfactor_field in enumerate(
        spec["candidates"],
        start=1,
    ):
        try:
            candidate_panel = fetch_factor_panel(rqfactor_field)

            valid_count = int(
                (research_mask & candidate_panel.notna())
                .sum(axis=1)
                .sum()
            )

            total_count = int(research_mask.sum(axis=1).sum())

            query_rows.append(
                {
                    "input_name": input_name,
                    "component_name": spec["component_name"],
                    "candidate_rank": candidate_rank,
                    "rqfactor_field": rqfactor_field,
                    "query_success": True,
                    "research_valid_observations": valid_count,
                    "research_total_observations": total_count,
                    "research_coverage": round(
                        valid_count / total_count,
                        4,
                    ) if total_count else 0.0,
                    "query_result": "available"
                    if valid_count > 0
                    else "no_valid_observation",
                }
            )

            if selected_panel is None and valid_count > 0:
                selected_panel = candidate_panel
                selected_field = rqfactor_field
                selected_valid_count = valid_count

        except Exception as error:
            query_rows.append(
                {
                    "input_name": input_name,
                    "component_name": spec["component_name"],
                    "candidate_rank": candidate_rank,
                    "rqfactor_field": rqfactor_field,
                    "query_success": False,
                    "research_valid_observations": 0,
                    "research_total_observations": int(
                        research_mask.sum(axis=1).sum()
                    ),
                    "research_coverage": 0.0,
                    "query_result": "query_error: {}".format(
                        str(error)[:160]
                    ),
                }
            )

    if selected_panel is None:
        selected_panel = pd.DataFrame(
            np.nan,
            index=signal_dates,
            columns=stock_list,
        )
        input_status = "missing_or_no_valid_observation"
    else:
        input_status = "selected"

    factor_inputs[input_name] = selected_panel
    selected_input_fields[input_name] = selected_field

    valid_by_date = (
        research_mask & selected_panel.notna()
    ).sum(axis=1)

    eligible_by_date = research_mask.sum(axis=1)

    coverage_rows.append(
        {
            "input_name": input_name,
            "component_name": spec["component_name"],
            "factor_modules": spec["factor_modules"],
            "necessity": spec["necessity"],
            "selected_rqfactor_field": selected_field,
            "input_status": input_status,
            "mean_valid_stock_count": round(
                float(valid_by_date.mean()),
                1,
            ),
            "min_valid_stock_count": int(valid_by_date.min()),
            "mean_coverage_pct": round(
                float(
                    (valid_by_date / eligible_by_date)
                    .replace([np.inf, -np.inf], np.nan)
                    .mean()
                    * 100
                ),
                2,
            ),
            "min_coverage_pct": round(
                float(
                    (valid_by_date / eligible_by_date)
                    .replace([np.inf, -np.inf], np.nan)
                    .min()
                    * 100
                ),
                2,
            ),
            "research_valid_observations": selected_valid_count,
        }
    )

ttm_input_coverage = pd.DataFrame(coverage_rows)
candidate_query_log = pd.DataFrame(query_rows)


# ---------- 4. 保存可开源审计结果 ----------
coverage_path = TABLE_DIR / "05_ttm_raw_input_coverage.csv"
query_log_path = TABLE_DIR / "05_ttm_input_candidate_query_log.csv"
metadata_path = META_DIR / "05_ttm_input_fields.json"
figure_path = FIG_DIR / "05_ttm_raw_input_coverage.png"

ttm_input_coverage.to_csv(
    coverage_path,
    index=False,
    encoding="utf-8-sig",
)

candidate_query_log.to_csv(
    query_log_path,
    index=False,
    encoding="utf-8-sig",
)

metadata = {
    "purpose": (
        "TTM raw-input coverage audit for ROA, report-formula ROIC, "
        "RONOA core proxy and FCFFIC."
    ),
    "selected_input_fields": selected_input_fields,
    "raw_panels_saved_to_repository": False,
    "roa_formula": (
        "net_profit_parent_company_ttm_0 / total_assets_mrq_0"
    ),
    "roic_formula_planned": (
        "EBIT * (1 - income_tax / total_profit) / "
        "(equity + interest_bearing_debt - cash_equivalent)"
    ),
    "ronoa_core_proxy_formula": (
        "(operating_profit + financial_expense - investment_income "
        "- fair_value_change_income) / "
        "(equity + interest_bearing_debt - cash_equivalent)"
    ),
    "fcffic_formula": (
        "(cash_flow_from_operating_activities - cash_paid_for_asset) / "
        "(equity + interest_bearing_debt - cash_equivalent)"
    ),
    "ronoa_note": (
        "Extended financial assets are audited separately. Missing "
        "components will never be silently imputed as zero."
    ),
}

with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(
        metadata,
        file,
        ensure_ascii=False,
        indent=2,
    )


# ---------- 5. 覆盖率图 ----------
plot_data = ttm_input_coverage.sort_values(
    "mean_coverage_pct",
    ascending=True,
).copy()

colors = [
    "#F4A261" if value == "optional" else "#00A6A6"
    for value in plot_data["necessity"]
]

fig, ax = plt.subplots(figsize=(11, 8))

ax.barh(
    plot_data["input_name"],
    plot_data["mean_coverage_pct"],
    color=colors,
)

ax.set_xlim(0, 100)
ax.set_xlabel("Mean coverage in research universe (%)")
ax.set_title("TTM Profitability Factor Input Coverage")
ax.grid(axis="x", alpha=0.25)

fig.tight_layout()
fig.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)


# ---------- 6. 输出 ----------
print("\n===== TTM 输入覆盖率 =====")
display(
    ttm_input_coverage.sort_values(
        ["necessity", "mean_coverage_pct"],
        ascending=[True, False],
    )
)

print("\n===== 未成功查询的候选字段 =====")
display(
    candidate_query_log[
        candidate_query_log["query_result"] != "available"
    ][
        [
            "input_name",
            "component_name",
            "rqfactor_field",
            "candidate_rank",
            "query_result",
        ]
    ]
)

print("\n内存中可供下一格使用的输入数量：", len(factor_inputs))
print("已保存：", coverage_path)
print("已保存：", query_log_path)
print("已保存：", metadata_path)
print("已保存：", figure_path)

In [ ]:
# 05.0000 本地缓存与断点续跑设置
# cache/ 内包含米筐原始个股数据和中间面板：
# 仅供实验室本地恢复使用，绝不进入 GitHub 导出包。

from pathlib import Path
import json
import hashlib
from datetime import datetime
import pandas as pd

PROJECT_ROOT = Path("/home/rice/notebook/a-share-profitability-factor-replication")

ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
CACHE_DIR = PROJECT_ROOT / "cache"
CACHE_DATA_DIR = CACHE_DIR / "data"
CACHE_META_DIR = CACHE_DIR / "metadata"

for folder in [
    ARTIFACT_DIR,
    CACHE_DIR,
    CACHE_DATA_DIR,
    CACHE_META_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

# 防止 Git 追踪任何本地缓存数据。
gitignore_path = PROJECT_ROOT / ".gitignore"
gitignore_rule = "cache/\n"

if gitignore_path.exists():
    existing_gitignore = gitignore_path.read_text(encoding="utf-8")
else:
    existing_gitignore = ""

if "cache/" not in existing_gitignore:
    with open(gitignore_path, "a", encoding="utf-8") as file:
        if existing_gitignore and not existing_gitignore.endswith("\n"):
            file.write("\n")
        file.write("\n# RiceQuant local-only cached data\n")
        file.write(gitignore_rule)

CACHE_SCHEMA_VERSION = "v1"

def cache_fingerprint(config):
    config_text = json.dumps(
        config,
        ensure_ascii=False,
        sort_keys=True,
        default=str,
    )

    return hashlib.sha256(
        config_text.encode("utf-8")
    ).hexdigest()

def load_or_build_cache(cache_name, config, build_function):
    """
    缓存存在且配置哈希一致：直接读取。
    不存在、配置变化或读取失败：重新计算。
    只有 build_function 成功返回后才写入缓存。
    """
    data_path = CACHE_DATA_DIR / f"{cache_name}.pkl"
    meta_path = CACHE_META_DIR / f"{cache_name}.json"

    expected_fingerprint = cache_fingerprint(config)

    if data_path.exists() and meta_path.exists():
        try:
            saved_meta = json.loads(
                meta_path.read_text(encoding="utf-8")
            )

            if (
                saved_meta.get("fingerprint")
                == expected_fingerprint
            ):
                cached_object = pd.read_pickle(data_path)

                print(
                    f"[缓存命中] {cache_name}："
                    "直接读取本地结果，不重新查询米筐。"
                )

                return cached_object

            print(
                f"[缓存失效] {cache_name}：参数或研究范围发生变化，重新计算。"
            )

        except Exception as error:
            print(
                f"[缓存读取失败] {cache_name}：{error}；重新计算。"
            )

    print(
        f"[首次计算] {cache_name}：计算成功后会自动保存本地缓存。"
    )

    built_object = build_function()

    # 只有成功得到结果才写入；报错时不会留下假缓存。
    pd.to_pickle(built_object, data_path)

    saved_meta = {
        "cache_name": cache_name,
        "fingerprint": expected_fingerprint,
        "config": config,
        "created_at": datetime.now().isoformat(
            timespec="seconds"
        ),
        "local_only": True,
        "github_export_allowed": False,
    }

    meta_path.write_text(
        json.dumps(saved_meta, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )

    print(f"[缓存已保存] {data_path}")

    return built_object

print("本地缓存目录：", CACHE_DIR)
print("GitHub 可公开结果目录：", ARTIFACT_DIR)
print("缓存规则已启用：cache/ 不会进入 Git。")

In [ ]:
# ============================================================
# 05.2 严格构造 TTM 原始因子
# 不填补缺失值；不保存股票级原始因子值
# ============================================================

required_variables = [
    "factor_inputs",
    "research_mask",
    "signal_dates",
    "TABLE_DIR",
    "META_DIR",
    "FIG_DIR",
]

missing_variables = [
    variable
    for variable in required_variables
    if variable not in globals()
]

if missing_variables:
    raise RuntimeError(
        "缺少变量：{}\n请先重新运行 05.0 和 05.1。".format(
            ", ".join(missing_variables)
        )
    )


# ---------- 1. 读取内存中的原始输入 ----------
net_profit = factor_inputs["net_profit_ttm"]
total_assets = factor_inputs["total_assets_mrq"]

ebit = factor_inputs["ebit_ttm"]
income_tax = factor_inputs["income_tax_ttm"]
total_profit = factor_inputs["total_profit_ttm"]

total_equity = factor_inputs["total_equity_mrq"]
interest_debt = factor_inputs["interest_bearing_debt"]
cash_equivalent = factor_inputs["cash_equivalent"]

operating_profit = factor_inputs["operating_profit_ttm"]
financial_expense = factor_inputs["financial_expense_ttm"]
investment_income = factor_inputs["investment_income_ttm"]
fair_value_change = factor_inputs["fair_value_change_ttm"]

cfo_ttm = factor_inputs["cfo_ttm"]
capex_ttm = factor_inputs["capex_ttm"]


# ---------- 2. 辅助函数 ----------
def replace_nonfinite(panel):
    return panel.replace([np.inf, -np.inf], np.nan)


def valid_values(panel):
    return panel.where(research_mask).stack(dropna=True)


def panel_audit(panel_name, panel):
    values = valid_values(panel)
    valid_by_date = (
        research_mask & panel.notna()
    ).sum(axis=1)

    eligible_by_date = research_mask.sum(axis=1)

    if len(values) == 0:
        return {
            "panel_name": panel_name,
            "mean_valid_stock_count": 0.0,
            "min_valid_stock_count": 0,
            "mean_coverage_pct": 0.0,
            "min_coverage_pct": 0.0,
            "median": np.nan,
            "p01": np.nan,
            "p99": np.nan,
            "positive_ratio": np.nan,
            "negative_ratio": np.nan,
        }

    return {
        "panel_name": panel_name,
        "mean_valid_stock_count": round(
            float(valid_by_date.mean()),
            1,
        ),
        "min_valid_stock_count": int(valid_by_date.min()),
        "mean_coverage_pct": round(
            float(
                (valid_by_date / eligible_by_date)
                .replace([np.inf, -np.inf], np.nan)
                .mean()
                * 100
            ),
            2,
        ),
        "min_coverage_pct": round(
            float(
                (valid_by_date / eligible_by_date)
                .replace([np.inf, -np.inf], np.nan)
                .min()
                * 100
            ),
            2,
        ),
        "median": float(values.median()),
        "p01": float(values.quantile(0.01)),
        "p99": float(values.quantile(0.99)),
        "positive_ratio": round(float((values > 0).mean()), 4),
        "negative_ratio": round(float((values < 0).mean()), 4),
    }


# ---------- 3. 严格公式：缺一个输入即为缺失 ----------
invested_capital = replace_nonfinite(
    total_equity + interest_debt - cash_equivalent
)

effective_tax_rate = replace_nonfinite(
    income_tax / total_profit.replace(0, np.nan)
)

noplat = replace_nonfinite(
    ebit * (1 - effective_tax_rate)
)

operating_profit_core = replace_nonfinite(
    operating_profit
    + financial_expense
    - investment_income
    - fair_value_change
)

# RONOA 扩展版：五项金融资产全部有值时才计算。
# 不将缺失字段默认为 0。
extended_financial_assets = (
    factor_inputs["fixed_deposits_mrq"]
    + factor_inputs["trading_financial_assets_mrq"]
    + factor_inputs["available_for_sale_assets_mrq"]
    + factor_inputs["hold_to_maturity_assets_mrq"]
    + factor_inputs["dividend_receivable_mrq"]
)

operating_net_assets_extended = replace_nonfinite(
    invested_capital - extended_financial_assets
)


# ---------- 4. 原始因子面板 ----------
raw_factor_panels = {
    "roa_reconstructed_strict": replace_nonfinite(
        net_profit / total_assets.where(total_assets > 0)
    ),
    "roa_vendor_reference": replace_nonfinite(
        factor_inputs["roa_vendor"]
    ),
    "roic_reconstructed_strict": replace_nonfinite(
        noplat / invested_capital.where(invested_capital > 0)
    ),
    "roic_vendor_reference": replace_nonfinite(
        factor_inputs["roic_vendor"]
    ),
    "ronoa_core_strict": replace_nonfinite(
        operating_profit_core
        / invested_capital.where(invested_capital > 0)
    ),
    "ronoa_extended_strict": replace_nonfinite(
        operating_profit_core
        / operating_net_assets_extended.where(
            operating_net_assets_extended > 0
        )
    ),
    "fcffic_reconstructed_strict": replace_nonfinite(
        (cfo_ttm - capex_ttm)
        / invested_capital.where(invested_capital > 0)
    ),
}


# ---------- 5. 覆盖率、符号与公式中间项审计 ----------
construction_audit = pd.DataFrame(
    [
        panel_audit(panel_name, panel)
        for panel_name, panel in raw_factor_panels.items()
    ]
)

intermediate_panels = {
    "invested_capital": invested_capital,
    "effective_tax_rate": effective_tax_rate,
    "noplat": noplat,
    "operating_profit_core": operating_profit_core,
    "extended_financial_assets_strict": extended_financial_assets,
    "operating_net_assets_extended": operating_net_assets_extended,
}

intermediate_audit = pd.DataFrame(
    [
        panel_audit(panel_name, panel)
        for panel_name, panel in intermediate_panels.items()
    ]
)


# ---------- 6. 保存可开源审计结果 ----------
coverage_path = (
    TABLE_DIR / "05_ttm_raw_factor_construction_coverage.csv"
)
intermediate_path = (
    TABLE_DIR / "05_ttm_formula_intermediate_audit.csv"
)
metadata_path = (
    META_DIR / "05_ttm_raw_factor_construction_rules.json"
)
figure_path = (
    FIG_DIR / "05_ttm_raw_factor_coverage.png"
)

construction_audit.to_csv(
    coverage_path,
    index=False,
    encoding="utf-8-sig",
)

intermediate_audit.to_csv(
    intermediate_path,
    index=False,
    encoding="utf-8-sig",
)

metadata = {
    "raw_factor_values_saved_to_repository": False,
    "missing_value_policy": (
        "Strict construction: no missing financial-statement input "
        "is silently imputed as zero."
    ),
    "roa_reconstructed_strict": (
        "net_profit_parent_company_ttm_0 / total_assets_mrq_0"
    ),
    "roic_reconstructed_strict": (
        "ebit_ttm_0 * (1 - income_tax_ttm_0 / profit_before_tax_ttm_0) "
        "/ (total_equity_mrq_0 + interest_bearing_debt - cash_equivalent)"
    ),
    "ronoa_core_strict": (
        "(profit_from_operation_ttm_0 + financial_expenseTTM "
        "- investment_income_ttm_0 - fair_value_change_income_ttm_0) "
        "/ (total_equity_mrq_0 + interest_bearing_debt - cash_equivalent)"
    ),
    "ronoa_extended_strict": (
        "RONOA core numerator / "
        "(invested capital - five disclosed financial-asset fields). "
        "This version is only calculated when all five fields are present."
    ),
    "fcffic_reconstructed_strict": (
        "(cash_flow_from_operating_activities_ttm_0 "
        "- cash_paid_for_asset_ttm_0) / "
        "(total_equity_mrq_0 + interest_bearing_debt - cash_equivalent)"
    ),
    "decision_pending": (
        "RONOA variants are not tested or included in a composite "
        "until strict coverage and missingness are reviewed."
    ),
}

with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(
        metadata,
        file,
        ensure_ascii=False,
        indent=2,
    )


# ---------- 7. 覆盖率图 ----------
plot_data = construction_audit.sort_values(
    "mean_coverage_pct",
    ascending=True,
).copy()

colors = [
    "#E76F51" if "extended" in name else "#00A6A6"
    for name in plot_data["panel_name"]
]

fig, ax = plt.subplots(figsize=(10, 5))

ax.barh(
    plot_data["panel_name"],
    plot_data["mean_coverage_pct"],
    color=colors,
)

ax.set_xlim(0, 100)
ax.set_xlabel("Mean coverage in research universe (%)")
ax.set_title("Strict TTM Raw Factor Construction Coverage")
ax.grid(axis="x", alpha=0.25)

for y_position, value in enumerate(plot_data["mean_coverage_pct"]):
    ax.text(
        min(value + 1.5, 96),
        y_position,
        "{:.1f}%".format(value),
        va="center",
        fontsize=9,
    )

fig.tight_layout()
fig.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()
plt.close(fig)


# ---------- 8. 输出 ----------
print("\n===== 严格原始因子覆盖率 =====")
display(construction_audit)

print("\n===== 公式中间项审计 =====")
display(intermediate_audit)

print("\n内存中待后续中性化的原始因子：")
print(list(raw_factor_panels.keys()))

print("\n已保存：", coverage_path)
print("已保存：", intermediate_path)
print("已保存：", metadata_path)
print("已保存：", figure_path)

In [ ]:
# 05.3 重构值—参考值一致性诊断，并冻结 TTM 因子池范围
# 不使用未来收益；只依据公式、覆盖率与参考字段一致性决定后续范围。

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path("/home/rice/notebook/a-share-profitability-factor-replication")
TABLE_DIR = PROJECT_ROOT / "artifacts" / "tables"
META_DIR = PROJECT_ROOT / "artifacts" / "metadata"
FIG_DIR = PROJECT_ROOT / "artifacts" / "figures"

for folder in [TABLE_DIR, META_DIR, FIG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

required_variables = ["raw_factor_panels", "signal_dates"]
missing_variables = [name for name in required_variables if name not in globals()]

if missing_variables:
    raise RuntimeError(
        "请先按顺序运行 05.0、05.1、05.2。缺少变量："
        + ", ".join(missing_variables)
    )

signal_dates = pd.DatetimeIndex(signal_dates)
# 严格限制到每个信号日的动态研究股票池。
# research_mask 的 True 数应当位于 726—800 之间。
if "research_mask" not in globals():
    raise RuntimeError(
        "缺少 research_mask。请先运行 05.0，再运行 05.1、05.2。"
    )

research_mask = (
    research_mask
    .reindex(
        index=signal_dates,
        columns=raw_factor_panels["roa_reconstructed_strict"].columns,
    )
    .fillna(False)
    .astype(bool)
)

mask_count = research_mask.sum(axis=1)

if mask_count.min() < 700 or mask_count.max() > 800:
    raise RuntimeError(
        "research_mask 股票数异常："
        f"最小 {mask_count.min()}，最大 {mask_count.max()}。"
    )

print(
    "研究股票池确认："
    f"{int(mask_count.min())} - {int(mask_count.max())} 只；"
    f"平均 {mask_count.mean():.2f} 只"
)

required_panels = [
    "roa_reconstructed_strict",
    "roa_vendor_reference",
    "roic_reconstructed_strict",
    "roic_vendor_reference",
    "ronoa_core_strict",
    "ronoa_extended_strict",
    "fcffic_reconstructed_strict",
]

missing_panels = [
    name for name in required_panels
    if name not in raw_factor_panels
]

if missing_panels:
    raise RuntimeError(
        "raw_factor_panels 中缺少："
        + ", ".join(missing_panels)
    )

# 直接读取 05.2 已保存的严格覆盖率。
# 这样不依赖 research_eligible 变量是否还存在。
construction_coverage_path = (
    TABLE_DIR / "05_ttm_raw_factor_construction_coverage.csv"
)

if not construction_coverage_path.exists():
    raise RuntimeError(
        "找不到 05.2 的覆盖率文件："
        + str(construction_coverage_path)
        + "。请重新运行 05.2。"
    )

factor_coverage = pd.read_csv(construction_coverage_path)

if "panel_name" in factor_coverage.columns:
    factor_coverage = factor_coverage.rename(
        columns={"panel_name": "factor_name"}
    )

if "factor_name" not in factor_coverage.columns:
    raise RuntimeError(
        "05.2 覆盖率文件中找不到 factor_name 或 panel_name 字段。"
    )

coverage_map = factor_coverage.set_index("factor_name").to_dict("index")

def get_panel(name):
    # 先对齐全部股票列，再用当期 research_mask 剔除非研究股票。
    # 这样每期配对股票数绝不会超过动态 CSI 800 研究券池。
    panel = raw_factor_panels[name].reindex(
        index=signal_dates,
        columns=research_mask.columns,
    )

    return panel.where(research_mask)

# ---------------------------------------------------------------
# 1. 重构值与米筐直接字段的一致性
# ---------------------------------------------------------------
comparison_pairs = {
    "roa_reconstructed_vs_vendor": (
        "roa_reconstructed_strict",
        "roa_vendor_reference",
    ),
    "roic_reconstructed_vs_vendor": (
        "roic_reconstructed_strict",
        "roic_vendor_reference",
    ),
}

MIN_PAIR_STOCKS = 100
monthly_diagnosis_rows = []

for comparison_name, (left_name, right_name) in comparison_pairs.items():
    left_panel = get_panel(left_name)
    right_panel = get_panel(right_name)

    for signal_date in signal_dates:
        paired = pd.concat(
            [
                left_panel.loc[signal_date].rename("left"),
                right_panel.loc[signal_date].rename("right"),
            ],
            axis=1,
        ).dropna()

        pair_stock_count = len(paired)

        if pair_stock_count < MIN_PAIR_STOCKS:
            pearson_corr = np.nan
            rank_corr = np.nan
            top_quintile_jaccard = np.nan
        else:
            pearson_corr = paired["left"].corr(paired["right"])

            rank_corr = (
                paired["left"].rank(method="average")
                .corr(paired["right"].rank(method="average"))
            )

            left_rank_pct = paired["left"].rank(
                method="average",
                pct=True,
            )
            right_rank_pct = paired["right"].rank(
                method="average",
                pct=True,
            )

            left_top = set(paired.index[left_rank_pct >= 0.80])
            right_top = set(paired.index[right_rank_pct >= 0.80])
            union_count = len(left_top | right_top)

            top_quintile_jaccard = (
                len(left_top & right_top) / union_count
                if union_count > 0 else np.nan
            )

        monthly_diagnosis_rows.append({
            "signal_date": signal_date,
            "comparison": comparison_name,
            "pair_stock_count": pair_stock_count,
            "pearson_correlation": pearson_corr,
            "rank_correlation": rank_corr,
            "top_quintile_jaccard": top_quintile_jaccard,
        })

monthly_diagnosis = pd.DataFrame(monthly_diagnosis_rows)

comparison_summary = (
    monthly_diagnosis
    .groupby("comparison")
    .agg(
        observations=("rank_correlation", "count"),
        mean_pair_stock_count=("pair_stock_count", "mean"),
        min_pair_stock_count=("pair_stock_count", "min"),
        mean_pearson_correlation=("pearson_correlation", "mean"),
        median_pearson_correlation=("pearson_correlation", "median"),
        mean_rank_correlation=("rank_correlation", "mean"),
        median_rank_correlation=("rank_correlation", "median"),
        mean_top_quintile_jaccard=("top_quintile_jaccard", "mean"),
    )
    .reset_index()
    .round(4)
)

# ---------------------------------------------------------------
# 2. 冻结本阶段的因子范围
# 此决定不参考未来收益或样本外表现。
# ---------------------------------------------------------------
scope_definition = [
    {
        "factor_name": "roa_reconstructed_strict",
        "role": "main_ttm_candidate",
        "decision": "enter_neutralization_and_return_test",
        "reason": "严格重构 ROA 覆盖率充足。",
    },
    {
        "factor_name": "roa_vendor_reference",
        "role": "reference_benchmark",
        "decision": "test_in_parallel_but_not_used_in_composite",
        "reason": "仅用于比较米筐直接字段与重构口径，不与重构版本混用。",
    },
    {
        "factor_name": "roic_reconstructed_strict",
        "role": "main_ttm_candidate",
        "decision": "enter_neutralization_and_return_test",
        "reason": "按 NOPLAT / Invested Capital 重构，覆盖率足以支持检验。",
    },
    {
        "factor_name": "roic_vendor_reference",
        "role": "reference_benchmark",
        "decision": "test_in_parallel_but_not_used_in_composite",
        "reason": "仅用于诊断米筐直接 ROIC 与重构公式的差异。",
    },
    {
        "factor_name": "ronoa_core_strict",
        "role": "low_coverage_sensitivity_only",
        "decision": "do_not_enter_main_pool",
        "reason": "严格口径覆盖率偏低，仅可作附录敏感性分析。",
    },
    {
        "factor_name": "ronoa_extended_strict",
        "role": "unavailable",
        "decision": "exclude",
        "reason": "扩展金融资产字段无法同时满足，覆盖率近乎为零。",
    },
    {
        "factor_name": "fcffic_reconstructed_strict",
        "role": "main_ttm_candidate_after_industry_exclusion",
        "decision": "enter_neutralization_after_report_industry_exclusion",
        "reason": "严格重构覆盖率充足；后续先按研报排除指定行业。",
    },
]

scope_rows = []

for row in scope_definition:
    info = coverage_map[row["factor_name"]]

    scope_rows.append({
        **row,
        "mean_coverage_pct": info["mean_coverage_pct"],
        "min_coverage_pct": info["min_coverage_pct"],
        "mean_valid_stock_count": info["mean_valid_stock_count"],
        "min_valid_stock_count": info["min_valid_stock_count"],
    })

factor_pool_scope = pd.DataFrame(scope_rows).round(4)

# ---------------------------------------------------------------
# 3. 保存开源所需的汇总产物，不保存个股级原始数据
# ---------------------------------------------------------------
monthly_path = TABLE_DIR / "05_reconstruction_reference_monthly_diagnosis.csv"
summary_path = TABLE_DIR / "05_reconstruction_reference_summary.csv"
scope_path = TABLE_DIR / "05_ttm_factor_pool_scope.csv"
metadata_path = META_DIR / "05_ttm_factor_pool_scope_rules.json"
figure_path = FIG_DIR / "05_reconstruction_reference_diagnosis.png"

monthly_diagnosis.to_csv(monthly_path, index=False, encoding="utf-8-sig")
comparison_summary.to_csv(summary_path, index=False, encoding="utf-8-sig")
factor_pool_scope.to_csv(scope_path, index=False, encoding="utf-8-sig")

metadata = {
    "purpose": "重构 ROA/ROIC 与米筐参考字段一致性诊断，并冻结 TTM 因子池。",
    "coverage_source": str(construction_coverage_path),
    "minimum_pair_stocks_for_correlation": MIN_PAIR_STOCKS,
    "main_ttm_candidates": [
        "roa_reconstructed_strict",
        "roic_reconstructed_strict",
        "fcffic_reconstructed_strict",
    ],
    "reference_benchmarks": [
        "roa_vendor_reference",
        "roic_vendor_reference",
    ],
    "sensitivity_only": [
        "ronoa_core_strict",
    ],
    "excluded_from_main_pool": [
        "ronoa_extended_strict",
    ],
    "restrictions": [
        "不将缺失财务字段填充为零。",
        "RONOA 扩展口径因联合字段覆盖率不足而停用。",
        "FCFFIC 在有效性检验前必须执行研报行业剔除。",
        "本范围决定不使用未来收益和样本外表现。",
    ],
}

with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)

# ---------------------------------------------------------------
# 4. 可开源图表
# ---------------------------------------------------------------
try:
    plt.style.use("seaborn-whitegrid")
except OSError:
    plt.style.use("default")
    plt.rcParams["axes.grid"] = True
    plt.rcParams["grid.alpha"] = 0.25

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for comparison_name, group in monthly_diagnosis.groupby("comparison"):
    axes[0].plot(
        group["signal_date"],
        group["rank_correlation"],
        label=comparison_name,
        linewidth=1.6,
    )

axes[0].axhline(0, color="gray", linestyle="--", linewidth=1)
axes[0].set_title("Monthly Cross-sectional Rank Correlation")
axes[0].set_xlabel("Signal Date")
axes[0].set_ylabel("Reconstructed vs Reference Rank Correlation")
axes[0].legend()

plot_summary = comparison_summary.set_index("comparison")

axes[1].bar(
    plot_summary.index,
    plot_summary["mean_top_quintile_jaccard"],
    color=["#4C78A8", "#F58518"],
)
axes[1].set_ylim(0, 1)
axes[1].set_title("Mean Top-Quintile Set Overlap")
axes[1].set_xlabel("Comparison")
axes[1].set_ylabel("Jaccard Overlap")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()

print("\n===== 重构值与米筐参考字段一致性 =====")
print(comparison_summary.to_string(index=False))

print("\n===== 已冻结的 TTM 因子池范围 =====")
print(
    factor_pool_scope[
        [
            "factor_name",
            "role",
            "decision",
            "mean_coverage_pct",
            "min_coverage_pct",
        ]
    ].to_string(index=False)
)

print("\n已保存：", monthly_path)
print("已保存：", summary_path)
print("已保存：", scope_path)
print("已保存：", metadata_path)
print("已保存：", figure_path)

In [ ]:
# 05.3a 补画图：兼容 RiceQuant 的旧版 Matplotlib

import matplotlib.pyplot as plt

try:
    plt.style.use("seaborn-whitegrid")
except OSError:
    plt.style.use("default")
    plt.rcParams["axes.grid"] = True
    plt.rcParams["grid.alpha"] = 0.25

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

for comparison_name, group in monthly_diagnosis.groupby("comparison"):
    axes[0].plot(
        group["signal_date"],
        group["rank_correlation"],
        label=comparison_name,
        linewidth=1.6,
    )

axes[0].axhline(0, color="gray", linestyle="--", linewidth=1)
axes[0].set_title("Monthly Cross-sectional Rank Correlation")
axes[0].set_xlabel("Signal Date")
axes[0].set_ylabel("Reconstructed vs Reference Rank Correlation")
axes[0].legend()

plot_summary = comparison_summary.set_index("comparison")

axes[1].bar(
    plot_summary.index,
    plot_summary["mean_top_quintile_jaccard"],
    color=["#4C78A8", "#F58518"],
)
axes[1].set_ylim(0, 1)
axes[1].set_title("Mean Top-Quintile Set Overlap")
axes[1].set_xlabel("Comparison")
axes[1].set_ylabel("Jaccard Overlap")
axes[1].tick_params(axis="x", rotation=15)

plt.tight_layout()
plt.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()

print("\n===== 重构值与米筐参考字段一致性 =====")
print(comparison_summary.to_string(index=False))

print("\n===== 已冻结的 TTM 因子池范围 =====")
print(
    factor_pool_scope[
        [
            "factor_name",
            "role",
            "decision",
            "mean_coverage_pct",
            "min_coverage_pct",
        ]
    ].to_string(index=False)
)

print("\n图已保存：", figure_path)

In [ ]:
# 05.3b 临时诊断：定位 05.0 中仍存在的研究券池变量
# 不保存文件，不修改任何已有结果。

import numpy as np
import pandas as pd

if "raw_factor_panels" not in globals() or "signal_dates" not in globals():
    raise RuntimeError("当前内核缺少 raw_factor_panels 或 signal_dates，请先运行 05.0—05.2。")

signal_dates = pd.DatetimeIndex(signal_dates)
factor_columns = raw_factor_panels["roa_reconstructed_strict"].columns

def is_boolean_like(frame):
    values = frame.to_numpy().ravel()
    values = values[pd.notna(values)]

    if len(values) == 0:
        return False

    unique_values = pd.unique(values)

    return all(
        value in [True, False, 0, 1, 0.0, 1.0]
        for value in unique_values
    )

frame_rows = []

for variable_name, variable_value in list(globals().items()):
    if not isinstance(variable_value, pd.DataFrame):
        continue

    if len(variable_value.index) != len(signal_dates):
        continue

    if not variable_value.index.equals(signal_dates):
        continue

    common_columns = variable_value.columns.intersection(factor_columns)

    if len(common_columns) < 500:
        continue

    if not is_boolean_like(variable_value):
        continue

    candidate_mask = (
        variable_value
        .reindex(index=signal_dates, columns=common_columns)
        .fillna(False)
        .astype(bool)
    )

    count_by_date = candidate_mask.sum(axis=1)

    frame_rows.append({
        "variable_name": variable_name,
        "shape": str(variable_value.shape),
        "common_stock_columns": len(common_columns),
        "min_true_count": int(count_by_date.min()),
        "mean_true_count": round(float(count_by_date.mean()), 2),
        "max_true_count": int(count_by_date.max()),
    })

mask_candidates = pd.DataFrame(frame_rows)

if len(mask_candidates) == 0:
    print("没有找到布尔型 DataFrame 券池掩码。")
else:
    print("===== 可能的研究券池掩码 =====")
    print(
        mask_candidates.sort_values(
            ["mean_true_count", "common_stock_columns"],
            ascending=[False, False],
        ).to_string(index=False)
    )

print("\n===== 名称中可能包含券池含义的变量 =====")
keywords = ["mask", "eligible", "universe", "pool", "stock"]

for variable_name, variable_value in list(globals().items()):
    name_lower = variable_name.lower()

    if any(keyword in name_lower for keyword in keywords):
        if isinstance(variable_value, pd.DataFrame):
            print(variable_name, "-> DataFrame", variable_value.shape)
        elif isinstance(variable_value, dict):
            print(variable_name, "-> dict, 长度", len(variable_value))
        elif isinstance(variable_value, (list, tuple, set, pd.Index)):
            print(variable_name, "->", type(variable_value).__name__, "长度", len(variable_value))

In [ ]:
# 05.4 动态行业信息、FCFFIC 行业剔除与中性化输入审计
# 本单元不测试收益，不保存个股原始数据。

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import rqdatac
from rqfactor import Factor, execute_factor

PROJECT_ROOT = Path("/home/rice/notebook/a-share-profitability-factor-replication")
TABLE_DIR = PROJECT_ROOT / "artifacts" / "tables"
META_DIR = PROJECT_ROOT / "artifacts" / "metadata"
FIG_DIR = PROJECT_ROOT / "artifacts" / "figures"

for folder in [TABLE_DIR, META_DIR, FIG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

required_variables = [
    "stock_list",
    "signal_dates",
    "research_mask",
    "raw_factor_panels",
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "请先运行 05.0—05.3。缺少变量："
        + ", ".join(missing_variables)
    )

signal_dates = pd.DatetimeIndex(signal_dates)
stock_list = list(stock_list)

research_mask = (
    research_mask
    .reindex(index=signal_dates, columns=stock_list)
    .fillna(False)
    .astype(bool)
)

research_stock_count = research_mask.sum(axis=1)

if research_stock_count.min() < 700 or research_stock_count.max() > 800:
    raise RuntimeError(
        "research_mask 范围异常："
        f"{research_stock_count.min()} - {research_stock_count.max()}"
    )

# ---------------------------------------------------------------
# 1. 读取动态市值：中性化需要 log 市值
# ---------------------------------------------------------------
MARKET_CAP_FACTOR = "market_cap_3"

market_cap_panel = execute_factor(
    Factor(MARKET_CAP_FACTOR),
    stock_list,
    signal_dates.min(),
    signal_dates.max(),
)

market_cap_panel.index = pd.to_datetime(market_cap_panel.index)

market_cap_panel = (
    market_cap_panel
    .reindex(index=signal_dates, columns=stock_list)
    .astype(float)
    .where(research_mask)
)

log_market_cap_panel = np.log(
    market_cap_panel.where(market_cap_panel > 0)
)

# ---------------------------------------------------------------
# 2. 逐月获取信号日当时的中信 2019 一级行业
# ---------------------------------------------------------------
INDUSTRY_SOURCE = "citics_2019"
INDUSTRY_LEVEL = 1
INDUSTRY_FIELD = "first_industry_name"

industry_panel_raw = pd.DataFrame(
    index=signal_dates,
    columns=stock_list,
    dtype="object",
)

for sequence, as_of_date in enumerate(signal_dates, start=1):
    industry_df = rqdatac.get_instrument_industry(
        order_book_ids=stock_list,
        source=INDUSTRY_SOURCE,
        level=INDUSTRY_LEVEL,
        date=pd.Timestamp(as_of_date),
        market="cn",
    ).copy()

    if "order_book_id" in industry_df.columns:
        industry_df = industry_df.set_index("order_book_id")

    industry_df.index = industry_df.index.astype(str)

    if INDUSTRY_FIELD not in industry_df.columns:
        raise ValueError(
            "未找到行业字段 "
            + INDUSTRY_FIELD
            + "，实际字段为："
            + str(industry_df.columns.tolist())
        )

    industry_panel_raw.loc[as_of_date] = (
        industry_df[INDUSTRY_FIELD]
        .reindex(stock_list)
        .values
    )

    if sequence % 12 == 0 or sequence == len(signal_dates):
        print(
            f"已完成 {sequence}/{len(signal_dates)} 个信号日的行业查询"
        )

industry_panel = industry_panel_raw.where(research_mask)

# ---------------------------------------------------------------
# 3. 严格执行研报对 FCFFIC 的行业排除
# ---------------------------------------------------------------
FCFFIC_EXCLUDED_INDUSTRIES = {
    "银行",
    "非银行金融",
    "综合金融",
    "房地产",
}

fcffic_industry_excluded = (
    research_mask
    & industry_panel.isin(FCFFIC_EXCLUDED_INDUSTRIES)
)

fcffic_industry_allowed = (
    research_mask
    & industry_panel.notna()
    & (~industry_panel.isin(FCFFIC_EXCLUDED_INDUSTRIES))
)

def restrict_to_research_universe(raw_panel):
    return (
        raw_panel
        .reindex(index=signal_dates, columns=stock_list)
        .astype(float)
        .where(research_mask)
    )

# 这些面板将留在内存中，供 05.5 清洗和中性化使用。
neutralization_input_panels = {
    "roa_reconstructed_strict": restrict_to_research_universe(
        raw_factor_panels["roa_reconstructed_strict"]
    ),
    "roa_vendor_reference": restrict_to_research_universe(
        raw_factor_panels["roa_vendor_reference"]
    ),
    "roic_reconstructed_strict": restrict_to_research_universe(
        raw_factor_panels["roic_reconstructed_strict"]
    ),
    "roic_vendor_reference": restrict_to_research_universe(
        raw_factor_panels["roic_vendor_reference"]
    ),
    "fcffic_reconstructed_strict": restrict_to_research_universe(
        raw_factor_panels["fcffic_reconstructed_strict"]
    ).where(fcffic_industry_allowed),
}

# ---------------------------------------------------------------
# 4. 覆盖率与行业排除审计
# ---------------------------------------------------------------
coverage_by_date = pd.DataFrame(index=signal_dates)

coverage_by_date["research_stock_count"] = research_mask.sum(axis=1)
coverage_by_date["market_cap_valid_count"] = (
    log_market_cap_panel.notna().sum(axis=1)
)
coverage_by_date["industry_valid_count"] = (
    industry_panel.notna().sum(axis=1)
)
coverage_by_date["fcffic_industry_excluded_count"] = (
    fcffic_industry_excluded.sum(axis=1)
)
coverage_by_date["fcffic_industry_allowed_count"] = (
    fcffic_industry_allowed.sum(axis=1)
)

for factor_name, factor_panel in neutralization_input_panels.items():
    coverage_by_date[
        factor_name + "_raw_valid_count"
    ] = factor_panel.notna().sum(axis=1)

coverage_by_date.index.name = "signal_date"

coverage_summary = (
    coverage_by_date
    .agg(["mean", "min", "max"])
    .T
    .reset_index()
    .rename(columns={"index": "metric"})
    .round(4)
)

industry_distribution = (
    industry_panel
    .stack()
    .value_counts()
    .rename_axis("industry_name")
    .reset_index(name="stock_date_observations")
)

excluded_industry_distribution = (
    industry_panel
    .where(fcffic_industry_excluded)
    .stack()
    .value_counts()
    .rename_axis("industry_name")
    .reset_index(name="excluded_stock_date_observations")
)

# ---------------------------------------------------------------
# 5. 保存可开源汇总产物
# ---------------------------------------------------------------
coverage_path = TABLE_DIR / "05_neutralization_context_coverage.csv"
summary_path = TABLE_DIR / "05_neutralization_context_summary.csv"
industry_path = TABLE_DIR / "05_industry_distribution.csv"
exclusion_path = TABLE_DIR / "05_fcffic_industry_exclusion_audit.csv"
metadata_path = META_DIR / "05_neutralization_context_rules.json"
figure_path = FIG_DIR / "05_neutralization_context_coverage.png"

coverage_by_date.reset_index().to_csv(
    coverage_path,
    index=False,
    encoding="utf-8-sig",
)

coverage_summary.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig",
)

industry_distribution.to_csv(
    industry_path,
    index=False,
    encoding="utf-8-sig",
)

excluded_industry_distribution.to_csv(
    exclusion_path,
    index=False,
    encoding="utf-8-sig",
)

metadata = {
    "industry_source": INDUSTRY_SOURCE,
    "industry_level": INDUSTRY_LEVEL,
    "industry_field": INDUSTRY_FIELD,
    "market_cap_factor": MARKET_CAP_FACTOR,
    "fcffic_excluded_industries": sorted(
        FCFFIC_EXCLUDED_INDUSTRIES
    ),
    "research_universe": (
        "Dynamic CSI 800 after new-stock, ST and suspension filters."
    ),
    "raw_factor_values_saved_to_repository": False,
    "next_step": (
        "MAD winsorization, CITIC 2019 level-1 industry "
        "neutralization, log-market-cap neutralization and z-score."
    ),
}

with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)

# ---------------------------------------------------------------
# 6. 覆盖率图
# ---------------------------------------------------------------
plt.style.use("default")
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

plot_columns = [
    "research_stock_count",
    "market_cap_valid_count",
    "industry_valid_count",
    "fcffic_industry_allowed_count",
    "fcffic_reconstructed_strict_raw_valid_count",
]

plt.figure(figsize=(13, 5))

for column_name in plot_columns:
    plt.plot(
        coverage_by_date.index,
        coverage_by_date[column_name],
        label=column_name,
        linewidth=1.5,
    )

plt.title("Neutralization Context and FCFFIC Industry Filter Coverage")
plt.xlabel("Signal Date")
plt.ylabel("Stock Count")
plt.legend()
plt.tight_layout()
plt.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()

print("\n===== 中性化输入覆盖率摘要 =====")
print(coverage_summary.to_string(index=False))

print("\n===== FCFFIC 排除行业的累计观测数 =====")
print(excluded_industry_distribution.to_string(index=False))

print(
    "\nFCFFIC 每期平均因行业剔除股票数："
    f"{coverage_by_date['fcffic_industry_excluded_count'].mean():.2f}"
)

print(
    "FCFFIC 每期平均允许测试股票数："
    f"{coverage_by_date['fcffic_industry_allowed_count'].mean():.2f}"
)

print("\n已保存：", coverage_path)
print("已保存：", summary_path)
print("已保存：", industry_path)
print("已保存：", exclusion_path)
print("已保存：", metadata_path)
print("已保存：", figure_path)

In [ ]:
# 05.5 TTM 盈利因子的统一清洗、中性化与标准化
# 因子：重构 ROA、米筐 ROA 参考、重构 ROIC、米筐 ROIC 参考、重构 FCFFIC
# 流程：MAD 去极值 -> 中信2019一级行业 + log市值中性化 -> z-score

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path("/home/rice/notebook/a-share-profitability-factor-replication")
TABLE_DIR = PROJECT_ROOT / "artifacts" / "tables"
META_DIR = PROJECT_ROOT / "artifacts" / "metadata"
FIG_DIR = PROJECT_ROOT / "artifacts" / "figures"

for folder in [TABLE_DIR, META_DIR, FIG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

required_variables = [
    "signal_dates",
    "research_mask",
    "industry_panel",
    "log_market_cap_panel",
    "neutralization_input_panels",
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "请先运行 05.0—05.4。缺少变量："
        + ", ".join(missing_variables)
    )

signal_dates = pd.DatetimeIndex(signal_dates)

FACTOR_TEST_POOL = [
    "roa_reconstructed_strict",
    "roa_vendor_reference",
    "roic_reconstructed_strict",
    "roic_vendor_reference",
    "fcffic_reconstructed_strict",
]

missing_factors = [
    factor_name for factor_name in FACTOR_TEST_POOL
    if factor_name not in neutralization_input_panels
]

if missing_factors:
    raise RuntimeError(
        "neutralization_input_panels 中缺少："
        + ", ".join(missing_factors)
    )

MAD_N = 3.5
MIN_OBSERVATIONS = 100
MIN_INDUSTRY_GROUPS = 5

def mad_winsorize(series, n=MAD_N):
    """横截面 MAD 去极值；不填补缺失值。"""
    clean = (
        series
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .astype(float)
    )

    if len(clean) == 0:
        return clean

    median_value = clean.median()
    mad_value = (clean - median_value).abs().median()

    if (not np.isfinite(mad_value)) or mad_value == 0:
        return clean

    lower_bound = median_value - n * 1.4826 * mad_value
    upper_bound = median_value + n * 1.4826 * mad_value

    return clean.clip(lower=lower_bound, upper=upper_bound)

def zscore(series):
    """横截面标准化。"""
    std_value = series.std(ddof=0)

    if (not np.isfinite(std_value)) or std_value == 0:
        return pd.Series(np.nan, index=series.index, dtype=float)

    return (series - series.mean()) / std_value

def neutralize_one_cross_section(raw_factor, log_market_cap, industry):
    """
    对单个信号日做：
    factor ~ 常数项 + log市值 + 行业虚拟变量
    返回回归残差的 z-score。
    """
    raw_valid_count = int(raw_factor.notna().sum())

    data = pd.DataFrame({
        "factor": raw_factor,
        "log_market_cap": log_market_cap,
        "industry": industry,
    })

    data = (
        data
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )

    base_audit = {
        "raw_valid_count": raw_valid_count,
        "model_input_count": int(len(data)),
        "industry_groups": int(data["industry"].nunique()),
    }

    if len(data) < MIN_OBSERVATIONS:
        base_audit.update({
            "status": "insufficient_observations",
            "final_valid_count": 0,
            "raw_cap_corr": np.nan,
            "final_cap_corr": np.nan,
            "raw_industry_mean_dispersion": np.nan,
            "final_industry_mean_dispersion": np.nan,
            "design_matrix_rank": np.nan,
        })
        return pd.Series(dtype=float), base_audit

    if data["industry"].nunique() < MIN_INDUSTRY_GROUPS:
        base_audit.update({
            "status": "insufficient_industry_groups",
            "final_valid_count": 0,
            "raw_cap_corr": np.nan,
            "final_cap_corr": np.nan,
            "raw_industry_mean_dispersion": np.nan,
            "final_industry_mean_dispersion": np.nan,
            "design_matrix_rank": np.nan,
        })
        return pd.Series(dtype=float), base_audit

    factor_winsorized = mad_winsorize(data["factor"])
    data = data.loc[factor_winsorized.index].copy()
    data["factor"] = factor_winsorized

    industry_dummy = pd.get_dummies(
        data["industry"],
        prefix="industry",
    ).astype(float)

    # 留下一类作为基准行业，避免完全共线性。
    industry_dummy = industry_dummy.iloc[:, 1:]

    x_matrix = np.column_stack([
        np.ones(len(data)),
        data["log_market_cap"].to_numpy(dtype=float),
        industry_dummy.to_numpy(dtype=float),
    ])

    y_vector = data["factor"].to_numpy(dtype=float)

    beta, _, matrix_rank, _ = np.linalg.lstsq(
        x_matrix,
        y_vector,
        rcond=None,
    )

    residual = pd.Series(
        y_vector - x_matrix @ beta,
        index=data.index,
        dtype=float,
    )

    final_factor = zscore(residual)

    raw_industry_dispersion = (
        data.groupby("industry")["factor"].mean().std(ddof=0)
    )

    final_industry_dispersion = (
        final_factor
        .groupby(data.loc[final_factor.index, "industry"])
        .mean()
        .std(ddof=0)
    )

    base_audit.update({
        "status": "success",
        "final_valid_count": int(final_factor.notna().sum()),
        "raw_cap_corr": float(
            data["factor"].corr(data["log_market_cap"])
        ),
        "final_cap_corr": float(
            final_factor.corr(
                data.loc[final_factor.index, "log_market_cap"]
            )
        ),
        "raw_industry_mean_dispersion": float(raw_industry_dispersion),
        "final_industry_mean_dispersion": float(
            final_industry_dispersion
        ),
        "design_matrix_rank": int(matrix_rank),
    })

    return final_factor, base_audit

def construct_neutralized_panel(factor_name, raw_panel):
    final_panel = pd.DataFrame(
        np.nan,
        index=signal_dates,
        columns=raw_panel.columns,
        dtype=float,
    )

    audit_records = []

    for as_of_date in signal_dates:
        final_factor, audit = neutralize_one_cross_section(
            raw_factor=raw_panel.loc[as_of_date],
            log_market_cap=log_market_cap_panel.loc[as_of_date],
            industry=industry_panel.loc[as_of_date],
        )

        if len(final_factor) > 0:
            final_panel.loc[
                as_of_date,
                final_factor.index,
            ] = final_factor.values

        audit["signal_date"] = as_of_date
        audit["factor_name"] = factor_name
        audit_records.append(audit)

    return final_panel, pd.DataFrame(audit_records)

# ---------------------------------------------------------------
# 构建全部最终因子面板：仅保存在当前实验室内存
# ---------------------------------------------------------------
neutralized_ttm_factor_panels = {}
neutralization_audits = []

for factor_name in FACTOR_TEST_POOL:
    final_panel, factor_audit = construct_neutralized_panel(
        factor_name=factor_name,
        raw_panel=neutralization_input_panels[factor_name],
    )

    neutralized_ttm_factor_panels[factor_name] = final_panel
    neutralization_audits.append(factor_audit)

neutralization_audit = pd.concat(
    neutralization_audits,
    ignore_index=True,
)

# ---------------------------------------------------------------
# 汇总质量审计
# ---------------------------------------------------------------
neutralization_summary = (
    neutralization_audit
    .groupby("factor_name")
    .agg(
        successful_periods=(
            "status",
            lambda values: int((values == "success").sum()),
        ),
        raw_valid_mean=("raw_valid_count", "mean"),
        raw_valid_min=("raw_valid_count", "min"),
        model_input_mean=("model_input_count", "mean"),
        final_valid_mean=("final_valid_count", "mean"),
        final_valid_min=("final_valid_count", "min"),
        industry_groups_mean=("industry_groups", "mean"),
        industry_groups_min=("industry_groups", "min"),
        raw_cap_corr_abs_mean=(
            "raw_cap_corr",
            lambda values: values.abs().mean(),
        ),
        final_cap_corr_abs_mean=(
            "final_cap_corr",
            lambda values: values.abs().mean(),
        ),
        raw_industry_dispersion_mean=(
            "raw_industry_mean_dispersion",
            "mean",
        ),
        final_industry_dispersion_mean=(
            "final_industry_mean_dispersion",
            "mean",
        ),
    )
    .reset_index()
    .round(6)
)

failure_records = neutralization_audit[
    neutralization_audit["status"] != "success"
].copy()

# ---------------------------------------------------------------
# 保存可开源的汇总结果；不保存个股级最终因子
# ---------------------------------------------------------------
audit_path = TABLE_DIR / "05_ttm_factor_neutralization_audit.csv"
summary_path = TABLE_DIR / "05_ttm_factor_neutralization_summary.csv"
failure_path = TABLE_DIR / "05_ttm_factor_neutralization_failures.csv"
metadata_path = META_DIR / "05_ttm_factor_neutralization_rules.json"
figure_path = FIG_DIR / "05_ttm_factor_neutralization_quality.png"

neutralization_audit.to_csv(
    audit_path,
    index=False,
    encoding="utf-8-sig",
)

neutralization_summary.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig",
)

failure_records.to_csv(
    failure_path,
    index=False,
    encoding="utf-8-sig",
)

metadata = {
    "factor_test_pool": FACTOR_TEST_POOL,
    "treatment_order": [
        "raw factor",
        "MAD winsorization, n=3.5",
        "CITIC 2019 first-level industry neutralization",
        "log market-cap neutralization",
        "cross-sectional z-score",
    ],
    "minimum_observations": MIN_OBSERVATIONS,
    "minimum_industry_groups": MIN_INDUSTRY_GROUPS,
    "fcffic_specific_rule": (
        "Banks, non-bank financials, comprehensive finance and "
        "real estate are excluded before neutralization."
    ),
    "ronoa_status": (
        "Not in main factor pool because strict coverage is insufficient; "
        "it will be retained only as a clearly labeled sensitivity analysis."
    ),
    "raw_or_security_level_factor_values_saved_to_repository": False,
}

with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)

# ---------------------------------------------------------------
# 可开源图表
# ---------------------------------------------------------------
plt.style.use("default")
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

coverage_plot = neutralization_audit.pivot(
    index="signal_date",
    columns="factor_name",
    values="final_valid_count",
)

for factor_name in coverage_plot.columns:
    axes[0].plot(
        coverage_plot.index,
        coverage_plot[factor_name],
        label=factor_name,
        linewidth=1.4,
    )

axes[0].set_title("Final Factor Coverage After Neutralization")
axes[0].set_xlabel("Signal Date")
axes[0].set_ylabel("Valid Stock Count")
axes[0].legend(fontsize=8)

quality_plot = neutralization_summary.set_index("factor_name")

x_position = np.arange(len(quality_plot))
bar_width = 0.36

axes[1].bar(
    x_position - bar_width / 2,
    quality_plot["raw_cap_corr_abs_mean"],
    width=bar_width,
    label="Before neutralization",
)

axes[1].bar(
    x_position + bar_width / 2,
    quality_plot["final_cap_corr_abs_mean"],
    width=bar_width,
    label="After neutralization",
)

axes[1].set_title("Absolute Correlation with Log Market Cap")
axes[1].set_xlabel("Factor")
axes[1].set_ylabel("Mean Absolute Correlation")
axes[1].set_xticks(x_position)
axes[1].set_xticklabels(
    quality_plot.index,
    rotation=25,
    ha="right",
)
axes[1].legend()

plt.tight_layout()
plt.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()

print("\n===== 中性化质量摘要 =====")
print(neutralization_summary.to_string(index=False))

print("\n===== 非成功期数 =====")
print(len(failure_records))

if len(failure_records) > 0:
    print(
        failure_records[
            [
                "signal_date",
                "factor_name",
                "status",
                "model_input_count",
                "industry_groups",
            ]
        ].to_string(index=False)
    )

print("\n内存中可进入后续收益检验的因子：")
print(list(neutralized_ttm_factor_panels.keys()))

print("\n已保存：", audit_path)
print("已保存：", summary_path)
print("已保存：", failure_path)
print("已保存：", metadata_path)
print("已保存：", figure_path)

In [ ]:
# 05.6 统一生成未来 20 个交易日收益标签
# 信号 T 收盘后形成因子；
# T+1 开盘买入，T+21 开盘卖出，持有 20 个交易日。

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rqdatac import get_price, get_trading_dates

PROJECT_ROOT = Path("/home/rice/notebook/a-share-profitability-factor-replication")
TABLE_DIR = PROJECT_ROOT / "artifacts" / "tables"
META_DIR = PROJECT_ROOT / "artifacts" / "metadata"
FIG_DIR = PROJECT_ROOT / "artifacts" / "figures"

for folder in [TABLE_DIR, META_DIR, FIG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

required_variables = [
    "stock_list",
    "signal_dates",
    "research_mask",
    "neutralized_ttm_factor_panels",
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "请先运行 05.0—05.5。缺少变量："
        + ", ".join(missing_variables)
    )

signal_dates = pd.DatetimeIndex(signal_dates)
stock_list = list(stock_list)

research_mask = (
    research_mask
    .reindex(index=signal_dates, columns=stock_list)
    .fillna(False)
    .astype(bool)
)

FACTOR_TEST_POOL = [
    "roa_reconstructed_strict",
    "roa_vendor_reference",
    "roic_reconstructed_strict",
    "roic_vendor_reference",
    "fcffic_reconstructed_strict",
]

missing_factors = [
    factor_name for factor_name in FACTOR_TEST_POOL
    if factor_name not in neutralized_ttm_factor_panels
]

if missing_factors:
    raise RuntimeError(
        "neutralized_ttm_factor_panels 中缺少："
        + ", ".join(missing_factors)
    )

RETURN_HORIZON = 20
RETURN_CALENDAR_END = "2026-03-31"
PRICE_FIELD = "open"
PRICE_ADJUSTMENT = "pre"

# ---------------------------------------------------------------
# 1. 为每个信号日建立严格的交易时间表
# ---------------------------------------------------------------
trading_calendar = pd.DatetimeIndex(
    pd.to_datetime(
        get_trading_dates(
            signal_dates.min(),
            RETURN_CALENDAR_END,
        )
    )
)

schedule_records = []

for signal_date in signal_dates:
    signal_position = trading_calendar.get_loc(signal_date)

    entry_position = signal_position + 1
    exit_position = entry_position + RETURN_HORIZON

    if exit_position >= len(trading_calendar):
        entry_date = pd.NaT
        exit_date = pd.NaT
        status = "insufficient_future_calendar"
    else:
        entry_date = trading_calendar[entry_position]
        exit_date = trading_calendar[exit_position]
        status = "ok"

    schedule_records.append({
        "signal_date": signal_date,
        "entry_date_t_plus_1_open": entry_date,
        "exit_date_t_plus_21_open": exit_date,
        "holding_intervals": RETURN_HORIZON,
        "status": status,
    })

return_schedule = pd.DataFrame(schedule_records)

if (return_schedule["status"] != "ok").any():
    raise RuntimeError(
        "部分信号日没有足够未来交易日。请将 "
        "RETURN_CALENDAR_END 调整得更晚。"
    )

# ---------------------------------------------------------------
# 2. 一次性获取全部所需开盘价
# ---------------------------------------------------------------
first_entry_date = return_schedule[
    "entry_date_t_plus_1_open"
].min()

last_exit_date = return_schedule[
    "exit_date_t_plus_21_open"
].max()

open_price_long = get_price(
    order_book_ids=stock_list,
    start_date=first_entry_date,
    end_date=last_exit_date,
    frequency="1d",
    fields=PRICE_FIELD,
    adjust_type=PRICE_ADJUSTMENT,
)

open_price = (
    open_price_long["open"]
    .unstack("order_book_id")
)

open_price.index = pd.to_datetime(open_price.index)
open_price = open_price.reindex(columns=stock_list)

# ---------------------------------------------------------------
# 3. 生成未来收益标签：仅存内存，不导出个股收益
# ---------------------------------------------------------------
forward_return_20d = pd.DataFrame(
    np.nan,
    index=signal_dates,
    columns=stock_list,
    dtype=float,
)

for _, row in return_schedule.iterrows():
    signal_date = pd.Timestamp(row["signal_date"])
    entry_date = pd.Timestamp(row["entry_date_t_plus_1_open"])
    exit_date = pd.Timestamp(row["exit_date_t_plus_21_open"])

    entry_open = open_price.loc[entry_date]
    exit_open = open_price.loc[exit_date]

    valid_price_mask = (
        entry_open.notna()
        & exit_open.notna()
        & (entry_open > 0)
        & (exit_open > 0)
    )

    one_period_return = (
        exit_open / entry_open - 1
    ).where(valid_price_mask)

    forward_return_20d.loc[signal_date] = one_period_return

# ---------------------------------------------------------------
# 4. 逐因子审计收益配对覆盖率
# ---------------------------------------------------------------
return_coverage = (
    return_schedule
    .set_index("signal_date")
    .reindex(signal_dates)
    .copy()
)

return_coverage["research_stock_count"] = research_mask.sum(axis=1)

return_coverage["forward_return_valid_in_research_universe"] = (
    forward_return_20d.notna()
    & research_mask
).sum(axis=1)

for factor_name in FACTOR_TEST_POOL:
    factor_panel = (
        neutralized_ttm_factor_panels[factor_name]
        .reindex(index=signal_dates, columns=stock_list)
    )

    factor_valid = factor_panel.notna()
    paired = factor_valid & forward_return_20d.notna() & research_mask

    return_coverage[
        factor_name + "_factor_valid_count"
    ] = factor_valid.sum(axis=1)

    return_coverage[
        factor_name + "_factor_return_paired_count"
    ] = paired.sum(axis=1)

return_coverage.index.name = "signal_date"

coverage_summary_rows = []

for factor_name in FACTOR_TEST_POOL:
    factor_valid_column = factor_name + "_factor_valid_count"
    paired_column = factor_name + "_factor_return_paired_count"

    coverage_summary_rows.append({
        "factor_name": factor_name,
        "factor_valid_mean": return_coverage[
            factor_valid_column
        ].mean(),
        "factor_valid_min": return_coverage[
            factor_valid_column
        ].min(),
        "paired_mean": return_coverage[
            paired_column
        ].mean(),
        "paired_min": return_coverage[
            paired_column
        ].min(),
        "mean_pair_coverage_pct": (
            return_coverage[paired_column]
            / return_coverage[factor_valid_column]
            * 100
        ).mean(),
    })

return_coverage_summary = pd.DataFrame(
    coverage_summary_rows
).round(4)

# ---------------------------------------------------------------
# 5. 保存可开源汇总结果
# ---------------------------------------------------------------
schedule_path = TABLE_DIR / "05_return_schedule.csv"
coverage_path = TABLE_DIR / "05_forward_return_20d_coverage.csv"
summary_path = TABLE_DIR / "05_forward_return_20d_coverage_summary.csv"
metadata_path = META_DIR / "05_forward_return_20d_rules.json"
figure_path = FIG_DIR / "05_forward_return_20d_coverage.png"

return_schedule.to_csv(
    schedule_path,
    index=False,
    encoding="utf-8-sig",
)

return_coverage.to_csv(
    coverage_path,
    encoding="utf-8-sig",
)

return_coverage_summary.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig",
)

metadata = {
    "return_name": "forward_return_20d",
    "signal_timing": "Factor is formed after the close of signal date T.",
    "entry_rule": "Buy at the open of the first trading day after T.",
    "exit_rule": "Sell at the open of the 21st trading day after T.",
    "holding_intervals": RETURN_HORIZON,
    "price_field": PRICE_FIELD,
    "price_adjustment": PRICE_ADJUSTMENT,
    "missing_price_treatment": (
        "A stock is excluded for that signal date when entry or exit "
        "open price is missing or non-positive."
    ),
    "execution_constraints_included": False,
    "execution_constraint_note": (
        "Suspension, limit-up and tradability constraints are not "
        "included in this first factor-effectiveness test."
    ),
    "raw_price_or_stock_level_return_saved_to_repository": False,
}

with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)

# ---------------------------------------------------------------
# 6. 覆盖率图
# ---------------------------------------------------------------
plt.style.use("default")
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

plt.figure(figsize=(13, 5))

plt.plot(
    return_coverage.index,
    return_coverage["research_stock_count"],
    label="research_stock_count",
    linewidth=1.5,
)

plt.plot(
    return_coverage.index,
    return_coverage["forward_return_valid_in_research_universe"],
    label="forward_return_valid",
    linewidth=1.5,
)

for factor_name in FACTOR_TEST_POOL:
    plt.plot(
        return_coverage.index,
        return_coverage[
            factor_name + "_factor_return_paired_count"
        ],
        label=factor_name + "_paired",
        linewidth=1.2,
    )

plt.title("Forward 20-Day Return Coverage in Research Universe")
plt.xlabel("Signal Date")
plt.ylabel("Stock Count")
plt.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()

print("\n===== 收益标签时间表 =====")
print(
    "信号日期数：", len(return_schedule),
    "\n收益标签区间：",
    first_entry_date.strftime("%Y-%m-%d"),
    "至",
    last_exit_date.strftime("%Y-%m-%d"),
)

print("\n===== 因子—收益配对覆盖率摘要 =====")
print(return_coverage_summary.to_string(index=False))

print("\n已保存：", schedule_path)
print("已保存：", coverage_path)
print("已保存：", summary_path)
print("已保存：", metadata_path)
print("已保存：", figure_path)


In [ ]:
# 05.7 TTM 盈利因子的统一有效性检验
# 指标：IC、RankIC、五分组收益、Q5-Q1
# 分样本：全样本 / 2016-2022 样本内 / 2023-2025 样本外

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import t as student_t

PROJECT_ROOT = Path("/home/rice/notebook/a-share-profitability-factor-replication")
TABLE_DIR = PROJECT_ROOT / "artifacts" / "tables"
META_DIR = PROJECT_ROOT / "artifacts" / "metadata"
FIG_DIR = PROJECT_ROOT / "artifacts" / "figures"

for folder in [TABLE_DIR, META_DIR, FIG_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

required_variables = [
    "signal_dates",
    "forward_return_20d",
    "neutralized_ttm_factor_panels",
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "请先运行 05.0—05.6。缺少变量："
        + ", ".join(missing_variables)
    )

signal_dates = pd.DatetimeIndex(signal_dates)

FACTOR_TEST_POOL = [
    "roa_reconstructed_strict",
    "roa_vendor_reference",
    "roic_reconstructed_strict",
    "roic_vendor_reference",
    "fcffic_reconstructed_strict",
]

missing_factors = [
    factor_name for factor_name in FACTOR_TEST_POOL
    if factor_name not in neutralized_ttm_factor_panels
]

if missing_factors:
    raise RuntimeError(
        "neutralized_ttm_factor_panels 中缺少："
        + ", ".join(missing_factors)
    )

MIN_TEST_STOCKS = 100
QUINTILE_COUNT = 5

def evaluate_one_factor(factor_name, factor_panel):
    metric_records = []
    quintile_records = []

    factor_panel = factor_panel.reindex(
        index=signal_dates,
        columns=forward_return_20d.columns,
    )

    for signal_date in signal_dates:
        data = pd.DataFrame({
            "factor": factor_panel.loc[signal_date],
            "forward_return": forward_return_20d.loc[signal_date],
        })

        data = (
            data
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
        )

        valid_stocks = len(data)

        if valid_stocks < MIN_TEST_STOCKS:
            metric_records.append({
                "signal_date": signal_date,
                "factor_name": factor_name,
                "valid_stocks": valid_stocks,
                "status": "insufficient_paired_stocks",
                "ic": np.nan,
                "rank_ic": np.nan,
            })

            quintile_records.append({
                "signal_date": signal_date,
                "factor_name": factor_name,
                "valid_stocks": valid_stocks,
                "status": "insufficient_paired_stocks",
                "q1_return": np.nan,
                "q2_return": np.nan,
                "q3_return": np.nan,
                "q4_return": np.nan,
                "q5_return": np.nan,
                "high_minus_low_return": np.nan,
            })
            continue

        ic_value = data["factor"].corr(
            data["forward_return"],
            method="pearson",
        )

        rank_ic_value = data["factor"].corr(
            data["forward_return"],
            method="spearman",
        )

        # method="first" 只用于处理极少量并列值；
        # Q1 为因子最低组，Q5 为因子最高组。
        data["quintile"] = (
            pd.qcut(
                data["factor"].rank(method="first"),
                q=QUINTILE_COUNT,
                labels=False,
            )
            + 1
        )

        grouped_return = data.groupby(
            "quintile"
        )["forward_return"].mean()

        metric_records.append({
            "signal_date": signal_date,
            "factor_name": factor_name,
            "valid_stocks": valid_stocks,
            "status": "success",
            "ic": ic_value,
            "rank_ic": rank_ic_value,
        })

        quintile_records.append({
            "signal_date": signal_date,
            "factor_name": factor_name,
            "valid_stocks": valid_stocks,
            "status": "success",
            "q1_return": grouped_return.get(1, np.nan),
            "q2_return": grouped_return.get(2, np.nan),
            "q3_return": grouped_return.get(3, np.nan),
            "q4_return": grouped_return.get(4, np.nan),
            "q5_return": grouped_return.get(5, np.nan),
            "high_minus_low_return": (
                grouped_return.get(5, np.nan)
                - grouped_return.get(1, np.nan)
            ),
        })

    return (
        pd.DataFrame(metric_records),
        pd.DataFrame(quintile_records),
    )

# ---------------------------------------------------------------
# 1. 对预先定义的五个因子统一检验
# ---------------------------------------------------------------
metric_frames = []
quintile_frames = []

for factor_name in FACTOR_TEST_POOL:
    factor_metrics, factor_quintiles = evaluate_one_factor(
        factor_name=factor_name,
        factor_panel=neutralized_ttm_factor_panels[factor_name],
    )

    metric_frames.append(factor_metrics)
    quintile_frames.append(factor_quintiles)

factor_metrics = pd.concat(
    metric_frames,
    ignore_index=True,
)

factor_quintiles = pd.concat(
    quintile_frames,
    ignore_index=True,
)

# ---------------------------------------------------------------
# 2. 统计函数
# ---------------------------------------------------------------
def get_t_stat(series):
    series = series.dropna()

    if len(series) < 2 or series.std(ddof=1) == 0:
        return np.nan, np.nan

    t_stat = (
        series.mean()
        / (series.std(ddof=1) / np.sqrt(len(series)))
    )

    p_value = 2 * student_t.sf(
        np.abs(t_stat),
        df=len(series) - 1,
    )

    return t_stat, p_value

def annualized_ir(series):
    series = series.dropna()

    if len(series) < 2 or series.std(ddof=1) == 0:
        return np.nan

    return (
        series.mean()
        / series.std(ddof=1)
        * np.sqrt(12)
    )

def summarize_factor_period(
    factor_name,
    period_name,
    start_date,
    end_date,
):
    metrics = factor_metrics[
        (factor_metrics["factor_name"] == factor_name)
        & (factor_metrics["signal_date"] >= start_date)
        & (factor_metrics["signal_date"] <= end_date)
        & (factor_metrics["status"] == "success")
    ].copy()

    quintiles = factor_quintiles[
        (factor_quintiles["factor_name"] == factor_name)
        & (factor_quintiles["signal_date"] >= start_date)
        & (factor_quintiles["signal_date"] <= end_date)
        & (factor_quintiles["status"] == "success")
    ].copy()

    ic_t_stat, ic_p_value = get_t_stat(metrics["ic"])
    rank_ic_t_stat, rank_ic_p_value = get_t_stat(
        metrics["rank_ic"]
    )
    hml_t_stat, hml_p_value = get_t_stat(
        quintiles["high_minus_low_return"]
    )

    return {
        "factor_name": factor_name,
        "period": period_name,
        "observations": len(metrics),
        "average_valid_stocks": metrics["valid_stocks"].mean(),
        "mean_ic": metrics["ic"].mean(),
        "icir_annualized": annualized_ir(metrics["ic"]),
        "ic_t_stat": ic_t_stat,
        "ic_p_value": ic_p_value,
        "mean_rank_ic": metrics["rank_ic"].mean(),
        "rank_icir_annualized": annualized_ir(
            metrics["rank_ic"]
        ),
        "rank_ic_t_stat": rank_ic_t_stat,
        "rank_ic_p_value": rank_ic_p_value,
        "positive_rank_ic_ratio": (
            metrics["rank_ic"] > 0
        ).mean(),
        "mean_q1_return": quintiles["q1_return"].mean(),
        "mean_q2_return": quintiles["q2_return"].mean(),
        "mean_q3_return": quintiles["q3_return"].mean(),
        "mean_q4_return": quintiles["q4_return"].mean(),
        "mean_q5_return": quintiles["q5_return"].mean(),
        "mean_high_minus_low_return": (
            quintiles["high_minus_low_return"].mean()
        ),
        "high_minus_low_t_stat": hml_t_stat,
        "high_minus_low_p_value": hml_p_value,
        "high_minus_low_positive_ratio": (
            quintiles["high_minus_low_return"] > 0
        ).mean(),
    }

# ---------------------------------------------------------------
# 3. 固定样本划分，不依据结果调整
# ---------------------------------------------------------------
PERIODS = {
    "full_sample_2016_2025": (
        pd.Timestamp("2016-01-01"),
        pd.Timestamp("2025-12-31"),
    ),
    "in_sample_2016_2022": (
        pd.Timestamp("2016-01-01"),
        pd.Timestamp("2022-12-31"),
    ),
    "out_of_sample_2023_2025": (
        pd.Timestamp("2023-01-01"),
        pd.Timestamp("2025-12-31"),
    ),
}

summary_records = []

for factor_name in FACTOR_TEST_POOL:
    for period_name, (start_date, end_date) in PERIODS.items():
        summary_records.append(
            summarize_factor_period(
                factor_name=factor_name,
                period_name=period_name,
                start_date=start_date,
                end_date=end_date,
            )
        )

factor_effectiveness_summary = pd.DataFrame(
    summary_records
).round(6)

# ---------------------------------------------------------------
# 4. 保存可开源汇总产物
# ---------------------------------------------------------------
metrics_path = TABLE_DIR / "05_ttm_factor_ic_rankic_timeseries.csv"
quintile_path = TABLE_DIR / "05_ttm_factor_quintile_return_timeseries.csv"
summary_path = TABLE_DIR / "05_ttm_factor_effectiveness_summary.csv"
metadata_path = META_DIR / "05_ttm_factor_test_rules.json"
figure_path = FIG_DIR / "05_ttm_factor_effectiveness_summary.png"

factor_metrics.to_csv(
    metrics_path,
    index=False,
    encoding="utf-8-sig",
)

factor_quintiles.to_csv(
    quintile_path,
    index=False,
    encoding="utf-8-sig",
)

factor_effectiveness_summary.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig",
)

metadata = {
    "factor_test_pool": FACTOR_TEST_POOL,
    "return_label": "forward_return_20d",
    "ic_definition": (
        "Cross-sectional Pearson correlation between factor at T "
        "and forward 20-trading-day return."
    ),
    "rank_ic_definition": (
        "Cross-sectional Spearman correlation between factor at T "
        "and forward 20-trading-day return."
    ),
    "grouping": (
        "Equal-count quintiles. Q1 is the lowest factor group and "
        "Q5 is the highest factor group."
    ),
    "portfolio_return": (
        "Equal-weighted gross forward return within each quintile."
    ),
    "high_minus_low": "Q5 return minus Q1 return.",
    "sample_split": {
        "in_sample": "2016-01-01 to 2022-12-31",
        "out_of_sample": "2023-01-01 to 2025-12-31",
    },
    "transaction_costs_included": False,
    "execution_constraints_included": False,
    "raw_factor_or_stock_level_return_saved_to_repository": False,
}

with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)

# ---------------------------------------------------------------
# 5. 样本外汇总图
# ---------------------------------------------------------------
plt.style.use("default")
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

out_of_sample_summary = (
    factor_effectiveness_summary[
        factor_effectiveness_summary["period"]
        == "out_of_sample_2023_2025"
    ]
    .set_index("factor_name")
    .reindex(FACTOR_TEST_POOL)
)

rankic_values = out_of_sample_summary["mean_rank_ic"]
hml_values = out_of_sample_summary[
    "mean_high_minus_low_return"
]

rankic_colors = [
    "#54A24B" if value > 0 else "#E45756"
    for value in rankic_values
]

hml_colors = [
    "#54A24B" if value > 0 else "#E45756"
    for value in hml_values
]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].bar(
    out_of_sample_summary.index,
    rankic_values,
    color=rankic_colors,
)
axes[0].axhline(0, color="gray", linestyle="--", linewidth=1)
axes[0].set_title("Out-of-Sample Mean RankIC")
axes[0].set_xlabel("Factor")
axes[0].set_ylabel("Mean RankIC")
axes[0].tick_params(axis="x", rotation=25)

axes[1].bar(
    out_of_sample_summary.index,
    hml_values,
    color=hml_colors,
)
axes[1].axhline(0, color="gray", linestyle="--", linewidth=1)
axes[1].set_title("Out-of-Sample Q5 - Q1 Return")
axes[1].set_xlabel("Factor")
axes[1].set_ylabel("Mean 20-Day Gross Return")
axes[1].tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()

# ---------------------------------------------------------------
# 6. 控制台输出
# ---------------------------------------------------------------
display_columns = [
    "factor_name",
    "period",
    "observations",
    "mean_ic",
    "icir_annualized",
    "mean_rank_ic",
    "rank_icir_annualized",
    "positive_rank_ic_ratio",
    "mean_high_minus_low_return",
    "high_minus_low_t_stat",
]

print("\n===== TTM 盈利因子有效性汇总 =====")
print(
    factor_effectiveness_summary[
        display_columns
    ].round(4).to_string(index=False)
)

failure_count = int(
    (factor_metrics["status"] != "success").sum()
)

print("\n非成功的因子—日期检验数：", failure_count)

print("\n已保存：", metrics_path)
print("已保存：", quintile_path)
print("已保存：", summary_path)
print("已保存：", metadata_path)
print("已保存：", figure_path)

In [ ]:
# 05.8 TTM 盈利因子的研究决策日志
# 决策只使用既定样本外区间 2023-2025 的统计结果；
# 不据此重新构造因子、不调整参数、不设定组合权重。

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path("/home/rice/notebook/a-share-profitability-factor-replication")
TABLE_DIR = PROJECT_ROOT / "artifacts" / "tables"
META_DIR = PROJECT_ROOT / "artifacts" / "metadata"
FIG_DIR = PROJECT_ROOT / "artifacts" / "figures"

required_variables = ["factor_effectiveness_summary"]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "请先运行 05.7。缺少变量："
        + ", ".join(missing_variables)
    )

scope_path = TABLE_DIR / "05_ttm_factor_pool_scope.csv"

if not scope_path.exists():
    raise RuntimeError(
        "找不到因子池范围文件："
        + str(scope_path)
        + "。请先运行 05.3。"
    )

factor_scope = pd.read_csv(scope_path)

out_of_sample = factor_effectiveness_summary[
    factor_effectiveness_summary["period"]
    == "out_of_sample_2023_2025"
].copy()

decision_table = out_of_sample.merge(
    factor_scope[
        ["factor_name", "role", "decision"]
    ].rename(
        columns={"decision": "scope_decision"}
    ),
    on="factor_name",
    how="left",
)

def make_decision(row):
    """
    固定、保守的样本外归档规则：

    reject_as_standalone_factor：
        样本外 RankIC <= 0，或 Q5-Q1 平均收益 <= 0。

    candidate_not_confirmed：
        两项方向均为正，但至少一项统计检验尚未达到 5% 显著性。

    confirmed_standalone_candidate：
        RankIC 和 Q5-Q1 均为正，
        且 RankIC、Q5-Q1 的 p 值均小于 0.05，
        正 RankIC 比例不少于 55%。
    """
    rankic_positive = row["mean_rank_ic"] > 0
    hml_positive = row["mean_high_minus_low_return"] > 0

    both_significant = (
        row["rank_ic_p_value"] < 0.05
        and row["high_minus_low_p_value"] < 0.05
    )

    positive_ratio_ok = (
        row["positive_rank_ic_ratio"] >= 0.55
    )

    if (not rankic_positive) or (not hml_positive):
        return "reject_as_standalone_factor"

    if both_significant and positive_ratio_ok:
        return "confirmed_standalone_candidate"

    return "candidate_not_confirmed"

def make_reason(row):
    if row["research_decision"] == "reject_as_standalone_factor":
        return (
            "Out-of-sample RankIC or Q5-Q1 return is non-positive; "
            "do not retain as a standalone profitability factor."
        )

    if row["research_decision"] == "confirmed_standalone_candidate":
        return (
            "Out-of-sample RankIC and Q5-Q1 are positive and both "
            "pass the 5% significance threshold."
        )

    return (
        "Out-of-sample RankIC and Q5-Q1 are directionally positive, "
        "but statistical confirmation is insufficient."
    )

decision_table["research_decision"] = decision_table.apply(
    make_decision,
    axis=1,
)

decision_table["reason"] = decision_table.apply(
    make_reason,
    axis=1,
)

decision_table["decision_scope_note"] = np.where(
    decision_table["role"] == "reference_benchmark",
    "Reference benchmark only; never directly used in the future composite.",
    "Main or candidate factor according to the pre-return research scope.",
)

display_columns = [
    "factor_name",
    "role",
    "mean_rank_ic",
    "rank_icir_annualized",
    "rank_ic_p_value",
    "positive_rank_ic_ratio",
    "mean_high_minus_low_return",
    "high_minus_low_t_stat",
    "high_minus_low_p_value",
    "research_decision",
    "reason",
]

decision_log = decision_table[
    display_columns + ["decision_scope_note"]
].copy()

# ---------------------------------------------------------------
# 保存可开源决策日志
# ---------------------------------------------------------------
decision_path = TABLE_DIR / "05_ttm_factor_research_decision_log.csv"
metadata_path = META_DIR / "05_ttm_factor_research_decision_rules.json"
figure_path = FIG_DIR / "05_ttm_factor_out_of_sample_scorecard.png"

decision_log.to_csv(
    decision_path,
    index=False,
    encoding="utf-8-sig",
)

metadata = {
    "decision_period": "2023-01-01 to 2025-12-31",
    "decision_rule": {
        "reject_as_standalone_factor": (
            "Out-of-sample mean RankIC <= 0 OR mean Q5-Q1 return <= 0."
        ),
        "candidate_not_confirmed": (
            "Both directions are positive, but at least one of RankIC "
            "and Q5-Q1 does not pass the 5% significance threshold."
        ),
        "confirmed_standalone_candidate": (
            "Both directions are positive, both p-values < 0.05, and "
            "positive RankIC ratio >= 55%."
        ),
    },
    "important_restriction": (
        "The decision log does not alter factor definitions, "
        "parameters or future composite weights."
    ),
    "reference_factor_rule": (
        "Vendor ROA and vendor ROIC are reference benchmarks and are "
        "not directly eligible for the future composite."
    ),
}

with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)

# ---------------------------------------------------------------
# 样本外评分卡图
# ---------------------------------------------------------------
plt.style.use("default")
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

plot_data = decision_log.set_index("factor_name")

color_map = {
    "reject_as_standalone_factor": "#E45756",
    "candidate_not_confirmed": "#54A24B",
    "confirmed_standalone_candidate": "#4C78A8",
}

bar_colors = [
    color_map[decision]
    for decision in plot_data["research_decision"]
]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].bar(
    plot_data.index,
    plot_data["mean_rank_ic"],
    color=bar_colors,
)
axes[0].axhline(0, color="gray", linestyle="--", linewidth=1)
axes[0].set_title("Out-of-Sample Mean RankIC")
axes[0].set_xlabel("Factor")
axes[0].set_ylabel("Mean RankIC")
axes[0].tick_params(axis="x", rotation=25)

axes[1].bar(
    plot_data.index,
    plot_data["mean_high_minus_low_return"],
    color=bar_colors,
)
axes[1].axhline(0, color="gray", linestyle="--", linewidth=1)
axes[1].set_title("Out-of-Sample Q5 - Q1 Return")
axes[1].set_xlabel("Factor")
axes[1].set_ylabel("Mean 20-Day Gross Return")
axes[1].tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()

print("\n===== 样本外研究决策日志 =====")
print(
    decision_log[
        [
            "factor_name",
            "mean_rank_ic",
            "rank_icir_annualized",
            "positive_rank_ic_ratio",
            "mean_high_minus_low_return",
            "high_minus_low_t_stat",
            "research_decision",
        ]
    ].round(4).to_string(index=False)
)

print("\n已保存：", decision_path)
print("已保存：", metadata_path)
print("已保存：", figure_path)

In [ ]:
# 05.9 RONOA 严格核心口径：低覆盖敏感性检验
# 重要：本检验不改变 RONOA “不得进入主因子池”的既定决定。
# 原因是其严格原始覆盖率只有约 41%，该限制在收益检验前已经确定。

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import t as student_t

PROJECT_ROOT = Path("/home/rice/notebook/a-share-profitability-factor-replication")
TABLE_DIR = PROJECT_ROOT / "artifacts" / "tables"
META_DIR = PROJECT_ROOT / "artifacts" / "metadata"
FIG_DIR = PROJECT_ROOT / "artifacts" / "figures"

required_variables = [
    "stock_list",
    "signal_dates",
    "research_mask",
    "raw_factor_panels",
    "log_market_cap_panel",
    "industry_panel",
    "forward_return_20d",
    "neutralize_one_cross_section",
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "请先运行 05.0—05.7。缺少变量："
        + ", ".join(missing_variables)
    )

if "ronoa_core_strict" not in raw_factor_panels:
    raise RuntimeError(
        "raw_factor_panels 中没有 ronoa_core_strict，请重新运行 05.2。"
    )

signal_dates = pd.DatetimeIndex(signal_dates)
stock_list = list(stock_list)

research_mask = (
    research_mask
    .reindex(index=signal_dates, columns=stock_list)
    .fillna(False)
    .astype(bool)
)

# ---------------------------------------------------------------
# 1. 保持严格公式：不对投资收益、公允价值变动等缺失项补零
# ---------------------------------------------------------------
ronoa_core_raw = (
    raw_factor_panels["ronoa_core_strict"]
    .reindex(index=signal_dates, columns=stock_list)
    .astype(float)
    .where(research_mask)
)

# ---------------------------------------------------------------
# 2. 使用与 ROA / ROIC / FCFFIC 完全相同的中性化函数
# ---------------------------------------------------------------
ronoa_core_sensitivity_final = pd.DataFrame(
    np.nan,
    index=signal_dates,
    columns=stock_list,
    dtype=float,
)

neutralization_audit_records = []

for as_of_date in signal_dates:
    final_factor, audit = neutralize_one_cross_section(
        raw_factor=ronoa_core_raw.loc[as_of_date],
        log_market_cap=log_market_cap_panel.loc[as_of_date],
        industry=industry_panel.loc[as_of_date],
    )

    if len(final_factor) > 0:
        ronoa_core_sensitivity_final.loc[
            as_of_date,
            final_factor.index,
        ] = final_factor.values

    audit["signal_date"] = as_of_date
    audit["factor_name"] = "ronoa_core_strict_sensitivity"
    neutralization_audit_records.append(audit)

ronoa_neutralization_audit = pd.DataFrame(
    neutralization_audit_records
)

# ---------------------------------------------------------------
# 3. 与同一份 20 日远期收益标签配对并检验
# ---------------------------------------------------------------
MIN_TEST_STOCKS = 100
QUINTILE_COUNT = 5

metric_records = []
quintile_records = []

for signal_date in signal_dates:
    data = pd.DataFrame({
        "factor": ronoa_core_sensitivity_final.loc[signal_date],
        "forward_return": forward_return_20d.loc[signal_date],
    })

    data = (
        data
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
    )

    valid_stocks = len(data)

    if valid_stocks < MIN_TEST_STOCKS:
        metric_records.append({
            "signal_date": signal_date,
            "factor_name": "ronoa_core_strict_sensitivity",
            "valid_stocks": valid_stocks,
            "status": "insufficient_paired_stocks",
            "ic": np.nan,
            "rank_ic": np.nan,
        })

        quintile_records.append({
            "signal_date": signal_date,
            "factor_name": "ronoa_core_strict_sensitivity",
            "valid_stocks": valid_stocks,
            "status": "insufficient_paired_stocks",
            "q1_return": np.nan,
            "q2_return": np.nan,
            "q3_return": np.nan,
            "q4_return": np.nan,
            "q5_return": np.nan,
            "high_minus_low_return": np.nan,
        })
        continue

    ic_value = data["factor"].corr(
        data["forward_return"],
        method="pearson",
    )

    rank_ic_value = data["factor"].corr(
        data["forward_return"],
        method="spearman",
    )

    data["quintile"] = (
        pd.qcut(
            data["factor"].rank(method="first"),
            q=QUINTILE_COUNT,
            labels=False,
        )
        + 1
    )

    grouped_return = data.groupby(
        "quintile"
    )["forward_return"].mean()

    metric_records.append({
        "signal_date": signal_date,
        "factor_name": "ronoa_core_strict_sensitivity",
        "valid_stocks": valid_stocks,
        "status": "success",
        "ic": ic_value,
        "rank_ic": rank_ic_value,
    })

    quintile_records.append({
        "signal_date": signal_date,
        "factor_name": "ronoa_core_strict_sensitivity",
        "valid_stocks": valid_stocks,
        "status": "success",
        "q1_return": grouped_return.get(1, np.nan),
        "q2_return": grouped_return.get(2, np.nan),
        "q3_return": grouped_return.get(3, np.nan),
        "q4_return": grouped_return.get(4, np.nan),
        "q5_return": grouped_return.get(5, np.nan),
        "high_minus_low_return": (
            grouped_return.get(5, np.nan)
            - grouped_return.get(1, np.nan)
        ),
    })

ronoa_metrics = pd.DataFrame(metric_records)
ronoa_quintiles = pd.DataFrame(quintile_records)

# ---------------------------------------------------------------
# 4. 全样本、样本内、样本外汇总
# ---------------------------------------------------------------
def get_t_stat(series):
    series = series.dropna()

    if len(series) < 2 or series.std(ddof=1) == 0:
        return np.nan, np.nan

    t_stat = (
        series.mean()
        / (series.std(ddof=1) / np.sqrt(len(series)))
    )

    p_value = 2 * student_t.sf(
        np.abs(t_stat),
        df=len(series) - 1,
    )

    return t_stat, p_value

def annualized_ir(series):
    series = series.dropna()

    if len(series) < 2 or series.std(ddof=1) == 0:
        return np.nan

    return (
        series.mean()
        / series.std(ddof=1)
        * np.sqrt(12)
    )

PERIODS = {
    "full_sample_2016_2025": (
        pd.Timestamp("2016-01-01"),
        pd.Timestamp("2025-12-31"),
    ),
    "in_sample_2016_2022": (
        pd.Timestamp("2016-01-01"),
        pd.Timestamp("2022-12-31"),
    ),
    "out_of_sample_2023_2025": (
        pd.Timestamp("2023-01-01"),
        pd.Timestamp("2025-12-31"),
    ),
}

summary_records = []

for period_name, (start_date, end_date) in PERIODS.items():
    metrics = ronoa_metrics[
        (ronoa_metrics["signal_date"] >= start_date)
        & (ronoa_metrics["signal_date"] <= end_date)
        & (ronoa_metrics["status"] == "success")
    ]

    quintiles = ronoa_quintiles[
        (ronoa_quintiles["signal_date"] >= start_date)
        & (ronoa_quintiles["signal_date"] <= end_date)
        & (ronoa_quintiles["status"] == "success")
    ]

    rank_ic_t_stat, rank_ic_p_value = get_t_stat(
        metrics["rank_ic"]
    )
    hml_t_stat, hml_p_value = get_t_stat(
        quintiles["high_minus_low_return"]
    )

    summary_records.append({
        "factor_name": "ronoa_core_strict_sensitivity",
        "period": period_name,
        "observations": len(metrics),
        "average_valid_stocks": metrics["valid_stocks"].mean(),
        "min_valid_stocks": metrics["valid_stocks"].min(),
        "mean_ic": metrics["ic"].mean(),
        "icir_annualized": annualized_ir(metrics["ic"]),
        "mean_rank_ic": metrics["rank_ic"].mean(),
        "rank_icir_annualized": annualized_ir(
            metrics["rank_ic"]
        ),
        "rank_ic_t_stat": rank_ic_t_stat,
        "rank_ic_p_value": rank_ic_p_value,
        "positive_rank_ic_ratio": (
            metrics["rank_ic"] > 0
        ).mean(),
        "mean_q1_return": quintiles["q1_return"].mean(),
        "mean_q5_return": quintiles["q5_return"].mean(),
        "mean_high_minus_low_return": (
            quintiles["high_minus_low_return"].mean()
        ),
        "high_minus_low_t_stat": hml_t_stat,
        "high_minus_low_p_value": hml_p_value,
        "main_pool_eligibility": (
            "not_eligible_due_to_predefined_low_coverage"
        ),
    })

ronoa_summary = pd.DataFrame(summary_records).round(6)

# ---------------------------------------------------------------
# 5. 保存可开源汇总产物
# ---------------------------------------------------------------
audit_path = TABLE_DIR / "05_ronoa_sensitivity_neutralization_audit.csv"
metrics_path = TABLE_DIR / "05_ronoa_sensitivity_ic_rankic_timeseries.csv"
quintile_path = TABLE_DIR / "05_ronoa_sensitivity_quintile_return_timeseries.csv"
summary_path = TABLE_DIR / "05_ronoa_sensitivity_effectiveness_summary.csv"
metadata_path = META_DIR / "05_ronoa_sensitivity_rules.json"
figure_path = FIG_DIR / "05_ronoa_sensitivity_result.png"

ronoa_neutralization_audit.to_csv(
    audit_path,
    index=False,
    encoding="utf-8-sig",
)

ronoa_metrics.to_csv(
    metrics_path,
    index=False,
    encoding="utf-8-sig",
)

ronoa_quintiles.to_csv(
    quintile_path,
    index=False,
    encoding="utf-8-sig",
)

ronoa_summary.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig",
)

metadata = {
    "factor_name": "ronoa_core_strict_sensitivity",
    "formula_status": (
        "Strict core proxy using only observed fields; no missing input "
        "is imputed as zero."
    ),
    "main_pool_eligibility": (
        "No. This is determined before return testing because the "
        "strict factor coverage is insufficient for the main pool."
    ),
    "common_treatment": (
        "MAD winsorization, CITIC 2019 first-level industry "
        "neutralization, log-market-cap neutralization, z-score."
    ),
    "return_label": "forward_return_20d",
    "sample_split": {
        "in_sample": "2016-2022",
        "out_of_sample": "2023-2025",
    },
    "raw_factor_or_stock_level_return_saved_to_repository": False,
}

with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)

# ---------------------------------------------------------------
# 6. 图
# ---------------------------------------------------------------
plt.style.use("default")
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

successful_metrics = ronoa_metrics[
    ronoa_metrics["status"] == "success"
].copy()

successful_quintiles = ronoa_quintiles[
    ronoa_quintiles["status"] == "success"
].copy()

q_nav = (
    1
    + successful_quintiles[
        ["q1_return", "q2_return", "q3_return", "q4_return", "q5_return"]
    ]
).cumprod()

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].plot(
    successful_metrics["signal_date"],
    successful_metrics["ic"],
    label="IC",
    linewidth=1.2,
)

axes[0].plot(
    successful_metrics["signal_date"],
    successful_metrics["rank_ic"],
    label="RankIC",
    linewidth=1.2,
)

axes[0].axhline(0, color="gray", linestyle="--", linewidth=1)
axes[0].set_title("RONOA Sensitivity: Monthly IC and RankIC")
axes[0].set_xlabel("Signal Date")
axes[0].set_ylabel("Correlation")
axes[0].legend()

for column in q_nav.columns:
    axes[1].plot(
        successful_quintiles["signal_date"],
        q_nav[column],
        label=column.replace("_return", "").upper(),
        linewidth=1.3,
    )

axes[1].set_title("RONOA Sensitivity: Equal-Weighted Quintile NAV")
axes[1].set_xlabel("Signal Date")
axes[1].set_ylabel("Gross Net Value")
axes[1].legend()

plt.tight_layout()
plt.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()

print("\n===== RONOA 严格口径敏感性检验 =====")
print(
    ronoa_summary[
        [
            "period",
            "observations",
            "average_valid_stocks",
            "min_valid_stocks",
            "mean_rank_ic",
            "rank_icir_annualized",
            "positive_rank_ic_ratio",
            "mean_high_minus_low_return",
            "high_minus_low_t_stat",
            "main_pool_eligibility",
        ]
    ].round(4).to_string(index=False)
)

print(
    "\n中性化非成功期数：",
    int((ronoa_neutralization_audit["status"] != "success").sum()),
)

print(
    "收益检验非成功期数：",
    int((ronoa_metrics["status"] != "success").sum()),
)

print("\n已保存：", audit_path)
print("已保存：", metrics_path)
print("已保存：", quintile_path)
print("已保存：", summary_path)
print("已保存：", metadata_path)
print("已保存：", figure_path)

In [ ]:
# 05.10 稳定 ROIC / 稳定 RONOA 的构造覆盖率审计
# 研报思路：
# stable factor = 0.5 * zscore(current factor)
#               + 0.5 * zscore(-std(last 9 quarterly TTM observations))
#
# 实现说明：
# 月度信号中，每 3 个信号月视为一个季度间隔；
# 因此需要 0、3、6、...、24 月前共 9 个可得 TTM 截面。
# 缺少任一期时，不计算稳定性，不补零。

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path("/home/rice/notebook/a-share-profitability-factor-replication")
TABLE_DIR = PROJECT_ROOT / "artifacts" / "tables"
META_DIR = PROJECT_ROOT / "artifacts" / "metadata"
FIG_DIR = PROJECT_ROOT / "artifacts" / "figures"

required_variables = [
    "stock_list",
    "signal_dates",
    "research_mask",
    "raw_factor_panels",
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "请先运行 05.0—05.2。缺少变量："
        + ", ".join(missing_variables)
    )

required_raw_factors = [
    "roic_reconstructed_strict",
    "ronoa_core_strict",
]

missing_raw_factors = [
    name for name in required_raw_factors
    if name not in raw_factor_panels
]

if missing_raw_factors:
    raise RuntimeError(
        "raw_factor_panels 中缺少："
        + ", ".join(missing_raw_factors)
    )

signal_dates = pd.DatetimeIndex(signal_dates)
stock_list = list(stock_list)

research_mask = (
    research_mask
    .reindex(index=signal_dates, columns=stock_list)
    .fillna(False)
    .astype(bool)
)

STABILITY_PERIODS = 9
SIGNAL_MONTHS_PER_QUARTER = 3
MAX_LOOKBACK_MONTHS = (
    (STABILITY_PERIODS - 1)
    * SIGNAL_MONTHS_PER_QUARTER
)

def cross_section_zscore(series):
    clean = (
        series
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .astype(float)
    )

    result = pd.Series(
        np.nan,
        index=series.index,
        dtype=float,
    )

    if len(clean) == 0:
        return result

    std_value = clean.std(ddof=0)

    if (not np.isfinite(std_value)) or std_value == 0:
        return result

    result.loc[clean.index] = (
        clean - clean.mean()
    ) / std_value

    return result

def build_quarterly_spacing_volatility(raw_panel):
    """
    计算当前时点及 3、6、...、24 个信号月前的 9 个 TTM 截面标准差。
    必须九期全部存在才得到有效稳定性。
    """
    lagged_panels = [
        raw_panel.shift(
            periods=lag * SIGNAL_MONTHS_PER_QUARTER
        )
        for lag in range(STABILITY_PERIODS)
    ]

    lag_stack = np.stack(
        [
            panel.to_numpy(dtype=float)
            for panel in lagged_panels
        ],
        axis=0,
    )

    all_periods_available = np.isfinite(
        lag_stack
    ).all(axis=0)

    volatility_array = np.full(
        all_periods_available.shape,
        np.nan,
    )

    volatility_array[all_periods_available] = np.std(
        lag_stack[:, all_periods_available],
        axis=0,
        ddof=1,
    )

    return pd.DataFrame(
        volatility_array,
        index=raw_panel.index,
        columns=raw_panel.columns,
    )

def restrict_to_research_universe(raw_panel):
    return (
        raw_panel
        .reindex(index=signal_dates, columns=stock_list)
        .astype(float)
        .where(research_mask)
    )

# ---------------------------------------------------------------
# 1. 原始 ROIC / RONOA 严格因子
# ---------------------------------------------------------------
roic_raw_for_stability = restrict_to_research_universe(
    raw_factor_panels["roic_reconstructed_strict"]
)

ronoa_raw_for_stability = restrict_to_research_universe(
    raw_factor_panels["ronoa_core_strict"]
)

# ---------------------------------------------------------------
# 2. 9 个季度间隔的 TTM 波动率
# ---------------------------------------------------------------
roic_ttm_volatility = build_quarterly_spacing_volatility(
    roic_raw_for_stability
)

ronoa_ttm_volatility = build_quarterly_spacing_volatility(
    ronoa_raw_for_stability
)

# ---------------------------------------------------------------
# 3. 当前因子与负波动率等权融合
# 这里只得到“中性化前”的稳定因子；
# 下一步才会和其它因子一样做 MAD、行业和市值中性化。
# ---------------------------------------------------------------
roic_current_z = roic_raw_for_stability.apply(
    cross_section_zscore,
    axis=1,
)

roic_stability_z = (-roic_ttm_volatility).apply(
    cross_section_zscore,
    axis=1,
)

stable_roic_ttm_proxy_raw = (
    0.5 * roic_current_z
    + 0.5 * roic_stability_z
).where(research_mask)

ronoa_current_z = ronoa_raw_for_stability.apply(
    cross_section_zscore,
    axis=1,
)

ronoa_stability_z = (-ronoa_ttm_volatility).apply(
    cross_section_zscore,
    axis=1,
)

stable_ronoa_ttm_proxy_raw = (
    0.5 * ronoa_current_z
    + 0.5 * ronoa_stability_z
).where(research_mask)

# 后续 05.11 会使用这两个内存变量。
stable_factor_proxy_raw = {
    "stable_roic_ttm_proxy": stable_roic_ttm_proxy_raw,
    "stable_ronoa_ttm_proxy": stable_ronoa_ttm_proxy_raw,
}

# ---------------------------------------------------------------
# 4. 覆盖率审计
# ---------------------------------------------------------------
coverage = pd.DataFrame(index=signal_dates)

coverage["research_stock_count"] = research_mask.sum(axis=1)

coverage["roic_current_valid"] = (
    roic_raw_for_stability.notna().sum(axis=1)
)

coverage["roic_9q_volatility_valid"] = (
    roic_ttm_volatility.notna().sum(axis=1)
)

coverage["stable_roic_proxy_valid"] = (
    stable_roic_ttm_proxy_raw.notna().sum(axis=1)
)

coverage["ronoa_current_valid"] = (
    ronoa_raw_for_stability.notna().sum(axis=1)
)

coverage["ronoa_9q_volatility_valid"] = (
    ronoa_ttm_volatility.notna().sum(axis=1)
)

coverage["stable_ronoa_proxy_valid"] = (
    stable_ronoa_ttm_proxy_raw.notna().sum(axis=1)
)

coverage.index.name = "signal_date"

coverage_summary = (
    coverage
    .agg(["mean", "min", "max"])
    .T
    .reset_index()
    .rename(columns={"index": "metric"})
    .round(4)
)

first_valid_dates = []

for factor_name, column_name in {
    "stable_roic_ttm_proxy": "stable_roic_proxy_valid",
    "stable_ronoa_ttm_proxy": "stable_ronoa_proxy_valid",
}.items():
    valid_dates = coverage.index[
        coverage[column_name] >= 100
    ]

    first_valid_dates.append({
        "factor_name": factor_name,
        "first_date_with_at_least_100_stocks": (
            valid_dates.min() if len(valid_dates) > 0 else pd.NaT
        ),
        "periods_with_at_least_100_stocks": len(valid_dates),
        "mean_valid_stock_count": coverage[column_name].mean(),
        "min_valid_stock_count": coverage[column_name].min(),
    })

stability_readiness = pd.DataFrame(first_valid_dates)

# ---------------------------------------------------------------
# 5. 保存可开源汇总产物
# ---------------------------------------------------------------
coverage_path = TABLE_DIR / "05_stability_proxy_construction_coverage.csv"
readiness_path = TABLE_DIR / "05_stability_proxy_readiness.csv"
metadata_path = META_DIR / "05_stability_proxy_construction_rules.json"
figure_path = FIG_DIR / "05_stability_proxy_construction_coverage.png"

coverage.reset_index().to_csv(
    coverage_path,
    index=False,
    encoding="utf-8-sig",
)

stability_readiness.to_csv(
    readiness_path,
    index=False,
    encoding="utf-8-sig",
)

metadata = {
    "report_method": (
        "Current TTM profitability metric and negative volatility of "
        "the latest nine quarterly TTM observations are equally weighted."
    ),
    "implementation_proxy": (
        "Monthly signal panels are sampled at three-signal-month intervals "
        "to approximate consecutive quarterly TTM observations."
    ),
    "stability_periods": STABILITY_PERIODS,
    "signal_months_per_quarter": SIGNAL_MONTHS_PER_QUARTER,
    "maximum_lookback_months": MAX_LOOKBACK_MONTHS,
    "missing_value_rule": (
        "All nine historical observations must be present; no missing "
        "financial observation is filled with zero."
    ),
    "ronoa_role": (
        "Sensitivity-only. It remains ineligible for the main pool because "
        "strict RONOA raw coverage was insufficient before return testing."
    ),
    "raw_factor_values_saved_to_repository": False,
}

with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)

# ---------------------------------------------------------------
# 6. 图
# ---------------------------------------------------------------
plt.style.use("default")
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

plot_columns = [
    "research_stock_count",
    "roic_current_valid",
    "stable_roic_proxy_valid",
    "ronoa_current_valid",
    "stable_ronoa_proxy_valid",
]

plt.figure(figsize=(13, 5))

for column_name in plot_columns:
    plt.plot(
        coverage.index,
        coverage[column_name],
        label=column_name,
        linewidth=1.4,
    )

plt.title("Stable ROIC and RONOA Proxy Construction Coverage")
plt.xlabel("Signal Date")
plt.ylabel("Stock Count")
plt.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()

print("\n===== 稳定性构造覆盖率摘要 =====")
print(coverage_summary.to_string(index=False))

print("\n===== 稳定性因子可检验性 =====")
print(stability_readiness.to_string(index=False))

print("\n已保存：", coverage_path)
print("已保存：", readiness_path)
print("已保存：", metadata_path)
print("已保存：", figure_path)

In [ ]:
# 05.11 稳定 ROIC / 稳定 RONOA：中性化与有效性检验
# stable ROIC：研报方法代理，可正式报告
# stable RONOA：严格低覆盖敏感性附录，不进入主组合

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import t as student_t

PROJECT_ROOT = Path("/home/rice/notebook/a-share-profitability-factor-replication")
TABLE_DIR = PROJECT_ROOT / "artifacts" / "tables"
META_DIR = PROJECT_ROOT / "artifacts" / "metadata"
FIG_DIR = PROJECT_ROOT / "artifacts" / "figures"

required_variables = [
    "signal_dates",
    "stable_factor_proxy_raw",
    "log_market_cap_panel",
    "industry_panel",
    "forward_return_20d",
    "neutralize_one_cross_section",
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "请先运行 05.0—05.10。缺少变量："
        + ", ".join(missing_variables)
    )

signal_dates = pd.DatetimeIndex(signal_dates)

FACTOR_ROLE = {
    "stable_roic_ttm_proxy": "report_component_proxy",
    "stable_ronoa_ttm_proxy": "low_coverage_sensitivity_only",
}

missing_factors = [
    factor_name for factor_name in FACTOR_ROLE
    if factor_name not in stable_factor_proxy_raw
]

if missing_factors:
    raise RuntimeError(
        "stable_factor_proxy_raw 中缺少："
        + ", ".join(missing_factors)
    )

# ---------------------------------------------------------------
# 1. 对稳定因子执行与此前完全相同的中性化
# ---------------------------------------------------------------
stable_final_panels = {}
neutralization_audit_frames = []

for factor_name in FACTOR_ROLE:
    raw_panel = stable_factor_proxy_raw[factor_name]

    final_panel = pd.DataFrame(
        np.nan,
        index=signal_dates,
        columns=raw_panel.columns,
        dtype=float,
    )

    audit_records = []

    for as_of_date in signal_dates:
        final_factor, audit = neutralize_one_cross_section(
            raw_factor=raw_panel.loc[as_of_date],
            log_market_cap=log_market_cap_panel.loc[as_of_date],
            industry=industry_panel.loc[as_of_date],
        )

        if len(final_factor) > 0:
            final_panel.loc[
                as_of_date,
                final_factor.index,
            ] = final_factor.values

        audit["signal_date"] = as_of_date
        audit["factor_name"] = factor_name
        audit["component_role"] = FACTOR_ROLE[factor_name]
        audit_records.append(audit)

    stable_final_panels[factor_name] = final_panel
    neutralization_audit_frames.append(
        pd.DataFrame(audit_records)
    )

stable_neutralization_audit = pd.concat(
    neutralization_audit_frames,
    ignore_index=True,
)

# ---------------------------------------------------------------
# 2. 统一收益检验
# ---------------------------------------------------------------
MIN_TEST_STOCKS = 100
QUINTILE_COUNT = 5

metric_records = []
quintile_records = []

for factor_name, factor_panel in stable_final_panels.items():
    for signal_date in signal_dates:
        data = pd.DataFrame({
            "factor": factor_panel.loc[signal_date],
            "forward_return": forward_return_20d.loc[signal_date],
        })

        data = (
            data
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
        )

        valid_stocks = len(data)

        if valid_stocks < MIN_TEST_STOCKS:
            metric_records.append({
                "signal_date": signal_date,
                "factor_name": factor_name,
                "component_role": FACTOR_ROLE[factor_name],
                "valid_stocks": valid_stocks,
                "status": "insufficient_paired_stocks",
                "ic": np.nan,
                "rank_ic": np.nan,
            })

            quintile_records.append({
                "signal_date": signal_date,
                "factor_name": factor_name,
                "component_role": FACTOR_ROLE[factor_name],
                "valid_stocks": valid_stocks,
                "status": "insufficient_paired_stocks",
                "q1_return": np.nan,
                "q2_return": np.nan,
                "q3_return": np.nan,
                "q4_return": np.nan,
                "q5_return": np.nan,
                "high_minus_low_return": np.nan,
            })
            continue

        ic_value = data["factor"].corr(
            data["forward_return"],
            method="pearson",
        )

        rank_ic_value = data["factor"].corr(
            data["forward_return"],
            method="spearman",
        )

        data["quintile"] = (
            pd.qcut(
                data["factor"].rank(method="first"),
                q=QUINTILE_COUNT,
                labels=False,
            )
            + 1
        )

        grouped_return = data.groupby(
            "quintile"
        )["forward_return"].mean()

        metric_records.append({
            "signal_date": signal_date,
            "factor_name": factor_name,
            "component_role": FACTOR_ROLE[factor_name],
            "valid_stocks": valid_stocks,
            "status": "success",
            "ic": ic_value,
            "rank_ic": rank_ic_value,
        })

        quintile_records.append({
            "signal_date": signal_date,
            "factor_name": factor_name,
            "component_role": FACTOR_ROLE[factor_name],
            "valid_stocks": valid_stocks,
            "status": "success",
            "q1_return": grouped_return.get(1, np.nan),
            "q2_return": grouped_return.get(2, np.nan),
            "q3_return": grouped_return.get(3, np.nan),
            "q4_return": grouped_return.get(4, np.nan),
            "q5_return": grouped_return.get(5, np.nan),
            "high_minus_low_return": (
                grouped_return.get(5, np.nan)
                - grouped_return.get(1, np.nan)
            ),
        })

stable_metrics = pd.DataFrame(metric_records)
stable_quintiles = pd.DataFrame(quintile_records)

# ---------------------------------------------------------------
# 3. 汇总函数
# ---------------------------------------------------------------
def get_t_stat(series):
    series = series.dropna()

    if len(series) < 2 or series.std(ddof=1) == 0:
        return np.nan, np.nan

    t_stat = (
        series.mean()
        / (series.std(ddof=1) / np.sqrt(len(series)))
    )

    p_value = 2 * student_t.sf(
        np.abs(t_stat),
        df=len(series) - 1,
    )

    return t_stat, p_value

def annualized_ir(series):
    series = series.dropna()

    if len(series) < 2 or series.std(ddof=1) == 0:
        return np.nan

    return (
        series.mean()
        / series.std(ddof=1)
        * np.sqrt(12)
    )

PERIODS = {
    "all_available_periods": (
        pd.Timestamp("2016-01-01"),
        pd.Timestamp("2025-12-31"),
    ),
    "in_sample_2016_2022": (
        pd.Timestamp("2016-01-01"),
        pd.Timestamp("2022-12-31"),
    ),
    "out_of_sample_2023_2025": (
        pd.Timestamp("2023-01-01"),
        pd.Timestamp("2025-12-31"),
    ),
}

summary_records = []

for factor_name, role in FACTOR_ROLE.items():
    for period_name, (start_date, end_date) in PERIODS.items():
        metrics = stable_metrics[
            (stable_metrics["factor_name"] == factor_name)
            & (stable_metrics["signal_date"] >= start_date)
            & (stable_metrics["signal_date"] <= end_date)
            & (stable_metrics["status"] == "success")
        ]

        quintiles = stable_quintiles[
            (stable_quintiles["factor_name"] == factor_name)
            & (stable_quintiles["signal_date"] >= start_date)
            & (stable_quintiles["signal_date"] <= end_date)
            & (stable_quintiles["status"] == "success")
        ]

        rank_ic_t_stat, rank_ic_p_value = get_t_stat(
            metrics["rank_ic"]
        )

        hml_t_stat, hml_p_value = get_t_stat(
            quintiles["high_minus_low_return"]
        )

        summary_records.append({
            "factor_name": factor_name,
            "component_role": role,
            "period": period_name,
            "observations": len(metrics),
            "average_valid_stocks": metrics["valid_stocks"].mean(),
            "min_valid_stocks": metrics["valid_stocks"].min(),
            "mean_ic": metrics["ic"].mean(),
            "icir_annualized": annualized_ir(metrics["ic"]),
            "mean_rank_ic": metrics["rank_ic"].mean(),
            "rank_icir_annualized": annualized_ir(
                metrics["rank_ic"]
            ),
            "rank_ic_t_stat": rank_ic_t_stat,
            "rank_ic_p_value": rank_ic_p_value,
            "positive_rank_ic_ratio": (
                metrics["rank_ic"] > 0
            ).mean(),
            "mean_q1_return": quintiles["q1_return"].mean(),
            "mean_q5_return": quintiles["q5_return"].mean(),
            "mean_high_minus_low_return": (
                quintiles["high_minus_low_return"].mean()
            ),
            "high_minus_low_t_stat": hml_t_stat,
            "high_minus_low_p_value": hml_p_value,
        })

stable_factor_summary = pd.DataFrame(summary_records).round(6)

# ---------------------------------------------------------------
# 4. 与原始重构 ROIC 的可比摘要
# ---------------------------------------------------------------
base_summary_path = (
    TABLE_DIR / "05_ttm_factor_effectiveness_summary.csv"
)

if not base_summary_path.exists():
    raise FileNotFoundError(
        "缺少 05.7 保存的 ROIC 结果："
        + str(base_summary_path)
    )

base_summary = pd.read_csv(base_summary_path)

roic_base_comparison = base_summary[
    base_summary["factor_name"] == "roic_reconstructed_strict"
].copy()

roic_base_comparison["comparison_role"] = (
    "raw_roic_reconstructed"
)

roic_stable_comparison = stable_factor_summary[
    stable_factor_summary["factor_name"]
    == "stable_roic_ttm_proxy"
].copy()

roic_stable_comparison["comparison_role"] = (
    "stable_roic_ttm_proxy"
)

roic_stability_comparison = pd.concat(
    [
        roic_base_comparison,
        roic_stable_comparison,
    ],
    ignore_index=True,
    sort=False,
)

# ---------------------------------------------------------------
# 5. 保存可开源汇总结果
# ---------------------------------------------------------------
audit_path = (
    TABLE_DIR / "05_stable_profitability_neutralization_audit.csv"
)

metrics_path = (
    TABLE_DIR / "05_stable_profitability_ic_rankic_timeseries.csv"
)

quintile_path = (
    TABLE_DIR / "05_stable_profitability_quintile_return_timeseries.csv"
)

summary_path = (
    TABLE_DIR / "05_stable_profitability_effectiveness_summary.csv"
)

comparison_path = (
    TABLE_DIR / "05_roic_stability_comparison.csv"
)

metadata_path = (
    META_DIR / "05_stable_profitability_test_rules.json"
)

figure_path = (
    FIG_DIR / "05_stable_profitability_effectiveness.png"
)

stable_neutralization_audit.to_csv(
    audit_path,
    index=False,
    encoding="utf-8-sig",
)

stable_metrics.to_csv(
    metrics_path,
    index=False,
    encoding="utf-8-sig",
)

stable_quintiles.to_csv(
    quintile_path,
    index=False,
    encoding="utf-8-sig",
)

stable_factor_summary.to_csv(
    summary_path,
    index=False,
    encoding="utf-8-sig",
)

roic_stability_comparison.to_csv(
    comparison_path,
    index=False,
    encoding="utf-8-sig",
)

metadata = {
    "stable_roic_role": "report_component_proxy",
    "stable_ronoa_role": "low_coverage_sensitivity_only",
    "stability_formula": (
        "0.5 * zscore(current raw factor) + 0.5 * "
        "zscore(negative 9-quarter-spaced TTM volatility)"
    ),
    "stability_timing_proxy": (
        "Three monthly signal intervals approximate one quarter."
    ),
    "common_treatment": (
        "MAD winsorization, CITIC 2019 first-level industry "
        "neutralization, log-market-cap neutralization, z-score."
    ),
    "return_label": "forward_return_20d",
    "ronoa_restriction": (
        "Stable RONOA is not eligible for the main composite because "
        "the underlying strict RONOA data coverage was insufficient."
    ),
    "raw_factor_or_stock_level_return_saved_to_repository": False,
}

with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)

# ---------------------------------------------------------------
# 6. 图：样本外结果
# ---------------------------------------------------------------
plt.style.use("default")
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

plot_data = stable_factor_summary[
    stable_factor_summary["period"]
    == "out_of_sample_2023_2025"
].set_index("factor_name")

colors = [
    "#4C78A8"
    if role == "report_component_proxy"
    else "#B0B0B0"
    for role in plot_data["component_role"]
]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

axes[0].bar(
    plot_data.index,
    plot_data["mean_rank_ic"],
    color=colors,
)
axes[0].axhline(0, color="gray", linestyle="--", linewidth=1)
axes[0].set_title("Stable Factors: Out-of-Sample Mean RankIC")
axes[0].set_xlabel("Factor")
axes[0].set_ylabel("Mean RankIC")
axes[0].tick_params(axis="x", rotation=25)

axes[1].bar(
    plot_data.index,
    plot_data["mean_high_minus_low_return"],
    color=colors,
)
axes[1].axhline(0, color="gray", linestyle="--", linewidth=1)
axes[1].set_title("Stable Factors: Out-of-Sample Q5 - Q1 Return")
axes[1].set_xlabel("Factor")
axes[1].set_ylabel("Mean 20-Day Gross Return")
axes[1].tick_params(axis="x", rotation=25)

plt.tight_layout()
plt.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()

# ---------------------------------------------------------------
# 7. 输出
# ---------------------------------------------------------------
display_columns = [
    "factor_name",
    "component_role",
    "period",
    "observations",
    "average_valid_stocks",
    "mean_rank_ic",
    "rank_icir_annualized",
    "positive_rank_ic_ratio",
    "mean_high_minus_low_return",
    "high_minus_low_t_stat",
]

print("\n===== 稳定 ROIC / RONOA 有效性汇总 =====")
print(
    stable_factor_summary[
        display_columns
    ].round(4).to_string(index=False)
)

print("\n===== 原始 ROIC 与稳定 ROIC 对比 =====")
print(
    roic_stability_comparison[
        [
            "comparison_role",
            "period",
            "observations",
            "mean_rank_ic",
            "rank_icir_annualized",
            "mean_high_minus_low_return",
            "high_minus_low_t_stat",
        ]
    ].round(4).to_string(index=False)
)

print(
    "\n稳定 ROIC 中性化非成功期数：",
    int(
        (
            stable_neutralization_audit[
                (stable_neutralization_audit["factor_name"]
                 == "stable_roic_ttm_proxy")
                & (stable_neutralization_audit["status"] != "success")
            ]
        ).shape[0]
    ),
)

print(
    "稳定 RONOA 中性化非成功期数：",
    int(
        (
            stable_neutralization_audit[
                (stable_neutralization_audit["factor_name"]
                 == "stable_ronoa_ttm_proxy")
                & (stable_neutralization_audit["status"] != "success")
            ]
        ).shape[0]
    ),
)

print("\n已保存：", audit_path)
print("已保存：", metrics_path)
print("已保存：", quintile_path)
print("已保存：", summary_path)
print("已保存：", comparison_path)
print("已保存：", metadata_path)
print("已保存：", figure_path)

In [ ]:
# 05.12 重建稳定去杠杆 ROE，供后续综合盈利因子使用
# 公式：
# 1) 去杠杆 ROE = 横截面回归 ROE ~ 权益乘数 的残差
# 2) ROE 稳定性 = 过去 9 期 TTM ROE 的负标准差
# 3) 稳定去杠杆 ROE = 两者标准化后等权融合
# 4) 再执行 MAD 去极值、行业/市值中性化、z-score

from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from rqfactor import Factor, execute_factor

PROJECT_ROOT = Path("/home/rice/notebook/a-share-profitability-factor-replication")
TABLE_DIR = PROJECT_ROOT / "artifacts" / "tables"
META_DIR = PROJECT_ROOT / "artifacts" / "metadata"
FIG_DIR = PROJECT_ROOT / "artifacts" / "figures"

required_variables = [
    "stock_list",
    "signal_dates",
    "research_mask",
    "log_market_cap_panel",
    "industry_panel",
    "neutralize_one_cross_section",
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "请先运行 05.0—05.5。缺少变量："
        + ", ".join(missing_variables)
    )

signal_dates = pd.DatetimeIndex(signal_dates)
stock_list = list(stock_list)

research_mask = (
    research_mask
    .reindex(index=signal_dates, columns=stock_list)
    .fillna(False)
    .astype(bool)
)

# ---------------------------------------------------------------
# 1. 获取研报所需的点时 TTM ROE 与资产负债表输入
# ---------------------------------------------------------------
roe_input_fields = {
    "roe_lag_0": "return_on_equity_weighted_average_ttm_0",
    "roe_lag_1": "return_on_equity_weighted_average_ttm_1",
    "roe_lag_2": "return_on_equity_weighted_average_ttm_2",
    "roe_lag_3": "return_on_equity_weighted_average_ttm_3",
    "roe_lag_4": "return_on_equity_weighted_average_ttm_4",
    "roe_lag_5": "return_on_equity_weighted_average_ttm_5",
    "roe_lag_6": "return_on_equity_weighted_average_ttm_6",
    "roe_lag_7": "return_on_equity_weighted_average_ttm_7",
    "roe_lag_8": "return_on_equity_weighted_average_ttm_8",
    "total_assets_mrq": "total_assets_mrq_0",
    "total_equity_mrq": "total_equity_mrq_0",
}

roe_inputs = {}

for input_name, field_name in roe_input_fields.items():
    print("正在获取：", input_name)

    panel = execute_factor(
        Factor(field_name),
        stock_list,
        signal_dates.min(),
        signal_dates.max(),
    )

    panel.index = pd.to_datetime(panel.index)

    roe_inputs[input_name] = (
        panel
        .reindex(index=signal_dates, columns=stock_list)
        .astype(float)
        .where(research_mask)
    )

# ---------------------------------------------------------------
# 2. 工具函数
# ---------------------------------------------------------------
MAD_N = 3.5
MIN_LEVERAGE_REGRESSION_OBS = 100

def mad_winsorize(series, n=MAD_N):
    clean = (
        series
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .astype(float)
    )

    if len(clean) == 0:
        return clean

    median_value = clean.median()
    mad_value = (clean - median_value).abs().median()

    if (not np.isfinite(mad_value)) or mad_value == 0:
        return clean

    lower_bound = median_value - n * 1.4826 * mad_value
    upper_bound = median_value + n * 1.4826 * mad_value

    return clean.clip(lower=lower_bound, upper=upper_bound)

def cross_section_zscore(series):
    clean = (
        series
        .replace([np.inf, -np.inf], np.nan)
        .dropna()
        .astype(float)
    )

    result = pd.Series(
        np.nan,
        index=series.index,
        dtype=float,
    )

    if len(clean) == 0:
        return result

    std_value = clean.std(ddof=0)

    if (not np.isfinite(std_value)) or std_value == 0:
        return result

    result.loc[clean.index] = (
        clean - clean.mean()
    ) / std_value

    return result

def remove_leverage_effect(roe_panel, leverage_panel):
    result = pd.DataFrame(
        np.nan,
        index=signal_dates,
        columns=stock_list,
        dtype=float,
    )

    audit_rows = []

    for as_of_date in signal_dates:
        data = pd.DataFrame({
            "roe": roe_panel.loc[as_of_date],
            "equity_multiplier": leverage_panel.loc[as_of_date],
        })

        data = (
            data
            .replace([np.inf, -np.inf], np.nan)
            .dropna()
        )

        raw_valid_count = len(data)

        if raw_valid_count < MIN_LEVERAGE_REGRESSION_OBS:
            audit_rows.append({
                "signal_date": as_of_date,
                "raw_valid_count": raw_valid_count,
                "status": "insufficient_observations",
                "residual_valid_count": 0,
            })
            continue

        roe_clean = mad_winsorize(data["roe"])
        leverage_clean = mad_winsorize(
            data["equity_multiplier"]
        )

        data = pd.concat(
            [
                roe_clean.rename("roe"),
                leverage_clean.rename("equity_multiplier"),
            ],
            axis=1,
        ).dropna()

        x_matrix = np.column_stack([
            np.ones(len(data)),
            data["equity_multiplier"].to_numpy(dtype=float),
        ])

        y_vector = data["roe"].to_numpy(dtype=float)

        beta, _, _, _ = np.linalg.lstsq(
            x_matrix,
            y_vector,
            rcond=None,
        )

        residual = pd.Series(
            y_vector - x_matrix @ beta,
            index=data.index,
            dtype=float,
        )

        result.loc[as_of_date, residual.index] = residual.values

        audit_rows.append({
            "signal_date": as_of_date,
            "raw_valid_count": raw_valid_count,
            "status": "success",
            "residual_valid_count": len(residual),
            "equity_multiplier_beta": float(beta[1]),
        })

    return result, pd.DataFrame(audit_rows)

# ---------------------------------------------------------------
# 3. 构造去杠杆 ROE 与 9 期 TTM ROE 稳定性
# ---------------------------------------------------------------
roe_current_raw = roe_inputs["roe_lag_0"]

equity_multiplier_raw = (
    roe_inputs["total_assets_mrq"]
    / roe_inputs["total_equity_mrq"]
).where(
    (roe_inputs["total_assets_mrq"] > 0)
    & (roe_inputs["total_equity_mrq"] > 0)
)

delevered_roe_raw, leverage_audit = remove_leverage_effect(
    roe_current_raw,
    equity_multiplier_raw,
)

roe_lag_stack = np.stack(
    [
        roe_inputs[f"roe_lag_{lag}"].to_numpy(dtype=float)
        for lag in range(9)
    ],
    axis=0,
)

all_nine_periods_available = np.isfinite(
    roe_lag_stack
).all(axis=0)

roe_volatility_array = np.full(
    all_nine_periods_available.shape,
    np.nan,
)

roe_volatility_array[all_nine_periods_available] = np.std(
    roe_lag_stack[:, all_nine_periods_available],
    axis=0,
    ddof=1,
)

roe_ttm_volatility_raw = pd.DataFrame(
    roe_volatility_array,
    index=signal_dates,
    columns=stock_list,
)

stable_delevered_roe_pre_neutralization = (
    0.5 * delevered_roe_raw.apply(
        cross_section_zscore,
        axis=1,
    )
    + 0.5 * (-roe_ttm_volatility_raw).apply(
        cross_section_zscore,
        axis=1,
    )
).where(research_mask)

# ---------------------------------------------------------------
# 4. 与其他研报组件使用完全相同的中性化
# ---------------------------------------------------------------
stable_delevered_roe_for_composite_final = pd.DataFrame(
    np.nan,
    index=signal_dates,
    columns=stock_list,
    dtype=float,
)

neutralization_audit_rows = []

for as_of_date in signal_dates:
    final_factor, audit = neutralize_one_cross_section(
        raw_factor=stable_delevered_roe_pre_neutralization.loc[
            as_of_date
        ],
        log_market_cap=log_market_cap_panel.loc[as_of_date],
        industry=industry_panel.loc[as_of_date],
    )

    if len(final_factor) > 0:
        stable_delevered_roe_for_composite_final.loc[
            as_of_date,
            final_factor.index,
        ] = final_factor.values

    audit["signal_date"] = as_of_date
    audit["factor_name"] = "stable_delevered_roe"
    neutralization_audit_rows.append(audit)

stable_roe_neutralization_audit = pd.DataFrame(
    neutralization_audit_rows
)

# ---------------------------------------------------------------
# 5. 覆盖率审计与保存
# ---------------------------------------------------------------
coverage = pd.DataFrame(index=signal_dates)

coverage["research_stock_count"] = research_mask.sum(axis=1)
coverage["roe_current_valid"] = roe_current_raw.notna().sum(axis=1)
coverage["equity_multiplier_valid"] = (
    equity_multiplier_raw.notna().sum(axis=1)
)
coverage["delevered_roe_valid"] = (
    delevered_roe_raw.notna().sum(axis=1)
)
coverage["roe_9_period_volatility_valid"] = (
    roe_ttm_volatility_raw.notna().sum(axis=1)
)
coverage["stable_delevered_roe_pre_neutralization_valid"] = (
    stable_delevered_roe_pre_neutralization.notna().sum(axis=1)
)
coverage["stable_delevered_roe_final_valid"] = (
    stable_delevered_roe_for_composite_final.notna().sum(axis=1)
)

coverage.index.name = "signal_date"

coverage_summary = (
    coverage
    .agg(["mean", "min", "max"])
    .T
    .reset_index()
    .rename(columns={"index": "metric"})
    .round(4)
)

neutralization_summary = (
    stable_roe_neutralization_audit
    .agg({
        "raw_valid_count": ["mean", "min"],
        "model_input_count": ["mean", "min"],
        "final_valid_count": ["mean", "min"],
        "industry_groups": ["mean", "min"],
        "raw_cap_corr": lambda values: values.abs().mean(),
        "final_cap_corr": lambda values: values.abs().mean(),
    })
    .round(6)
)

coverage_path = (
    TABLE_DIR / "05_rebuilt_stable_delevered_roe_coverage.csv"
)

leverage_audit_path = (
    TABLE_DIR / "05_rebuilt_delevered_roe_audit.csv"
)

neutralization_audit_path = (
    TABLE_DIR / "05_rebuilt_stable_delevered_roe_neutralization_audit.csv"
)

metadata_path = (
    META_DIR / "05_rebuilt_stable_delevered_roe_rules.json"
)

figure_path = (
    FIG_DIR / "05_rebuilt_stable_delevered_roe_coverage.png"
)

coverage.reset_index().to_csv(
    coverage_path,
    index=False,
    encoding="utf-8-sig",
)

leverage_audit.to_csv(
    leverage_audit_path,
    index=False,
    encoding="utf-8-sig",
)

stable_roe_neutralization_audit.to_csv(
    neutralization_audit_path,
    index=False,
    encoding="utf-8-sig",
)

metadata = {
    "roe_input_fields": roe_input_fields,
    "equity_multiplier": "total_assets_mrq_0 / total_equity_mrq_0",
    "delevered_roe": (
        "Cross-sectional residual from ROE regressed on equity multiplier."
    ),
    "roe_stability": (
        "Negative sample standard deviation of the latest 9 TTM ROE observations."
    ),
    "stable_delevered_roe": (
        "0.5 * zscore(delevered ROE) + "
        "0.5 * zscore(negative ROE volatility)."
    ),
    "common_final_treatment": (
        "MAD winsorization, CITIC 2019 first-level industry "
        "neutralization, log-market-cap neutralization, z-score."
    ),
    "raw_factor_values_saved_to_repository": False,
}

with open(metadata_path, "w", encoding="utf-8") as file:
    json.dump(metadata, file, ensure_ascii=False, indent=2)

# ---------------------------------------------------------------
# 6. 图
# ---------------------------------------------------------------
plt.style.use("default")
plt.rcParams["axes.grid"] = True
plt.rcParams["grid.alpha"] = 0.25

plot_columns = [
    "research_stock_count",
    "delevered_roe_valid",
    "roe_9_period_volatility_valid",
    "stable_delevered_roe_final_valid",
]

plt.figure(figsize=(13, 5))

for column_name in plot_columns:
    plt.plot(
        coverage.index,
        coverage[column_name],
        label=column_name,
        linewidth=1.4,
    )

plt.title("Rebuilt Stable Delevered ROE Coverage")
plt.xlabel("Signal Date")
plt.ylabel("Stock Count")
plt.legend(fontsize=8)
plt.tight_layout()
plt.savefig(figure_path, dpi=160, bbox_inches="tight")
plt.show()

print("\n===== 重建稳定去杠杆 ROE 覆盖率 =====")
print(coverage_summary.to_string(index=False))

print("\n===== 重建稳定去杠杆 ROE 中性化摘要 =====")
print(neutralization_summary)

print(
    "\n去杠杆 ROE 回归非成功期数：",
    int((leverage_audit["status"] != "success").sum()),
)

print(
    "稳定去杠杆 ROE 中性化非成功期数：",
    int(
        (
            stable_roe_neutralization_audit["status"]
            != "success"
        ).sum()
    ),
)

print("\n已保存：", coverage_path)
print("已保存：", leverage_audit_path)
print("已保存：", neutralization_audit_path)
print("已保存：", metadata_path)
print("已保存：", figure_path)


In [ ]:
# 05.13.0 固化综合盈利因子的四个组件到本地缓存
# 目的：
# 1. 重启实验室后优先读取本地缓存，不重新请求财务字段；
# 2. 缓存仅保存在 cache/，不会进入 artifacts/ 或 GitHub 导出包；
# 3. 固化本次研究中已确定的“研报四个底层因子”。

from pathlib import Path
import hashlib
import json
from datetime import datetime

import numpy as np
import pandas as pd


# -------------------------------------------------------------------
# 0. 项目路径与本地缓存目录
# -------------------------------------------------------------------
PROJECT_ROOT = Path("/home/rice/notebook/a-share-profitability-factor-replication")
CACHE_DIR = PROJECT_ROOT / "cache"
CACHE_DATA_DIR = CACHE_DIR / "data"
CACHE_META_DIR = CACHE_DIR / "metadata"

for directory in [CACHE_DATA_DIR, CACHE_META_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# 防止本地缓存被 Git 追踪或误导出
gitignore_path = PROJECT_ROOT / ".gitignore"
gitignore_text = gitignore_path.read_text(encoding="utf-8") if gitignore_path.exists() else ""

if "cache/" not in gitignore_text.splitlines():
    with gitignore_path.open("a", encoding="utf-8") as f:
        if gitignore_text and not gitignore_text.endswith("\n"):
            f.write("\n")
        f.write("cache/\n")


# -------------------------------------------------------------------
# 1. 重启后变量完整性检查
# -------------------------------------------------------------------
required_variables = [
    "research_mask",
    "stable_delevered_roe_for_composite_final",
    "stable_final_panels",
    "neutralized_ttm_factor_panels",
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "当前会话缺少变量：" + ", ".join(missing_variables) + "\n\n"
        "请按缺少内容补跑，不需要全部从头运行：\n"
        "- 缺 research_mask：运行 05.0；\n"
        "- 缺 neutralized_ttm_factor_panels：运行 05.4、05.5；\n"
        "- 缺 stable_final_panels：运行 05.10、05.11；\n"
        "- 缺 stable_delevered_roe_for_composite_final：运行 05.12。\n\n"
        "补齐后重新运行本单元。"
    )

required_stable_keys = {
    "stable_roic_ttm_proxy",
    "stable_ronoa_ttm_proxy",
}
missing_stable_keys = required_stable_keys - set(stable_final_panels.keys())

if missing_stable_keys:
    raise RuntimeError(
        "stable_final_panels 中缺少："
        + ", ".join(sorted(missing_stable_keys))
        + "。请重新运行 05.10、05.11。"
    )

if "fcffic_reconstructed_strict" not in neutralized_ttm_factor_panels:
    raise RuntimeError(
        "neutralized_ttm_factor_panels 中缺少 fcffic_reconstructed_strict。"
        "请重新运行 05.4、05.5。"
    )


# -------------------------------------------------------------------
# 2. 对齐为研究券池的统一日期 × 股票矩阵
# -------------------------------------------------------------------
signal_index = research_mask.index
stock_columns = research_mask.columns

component_panels_in_memory = {
    "stable_delevered_roe": stable_delevered_roe_for_composite_final,
    "stable_roic_proxy": stable_final_panels["stable_roic_ttm_proxy"],
    "stable_ronoa_proxy": stable_final_panels["stable_ronoa_ttm_proxy"],
    "fcffic_reconstructed": neutralized_ttm_factor_panels[
        "fcffic_reconstructed_strict"
    ],
}

def align_component_panel(panel, name):
    if not isinstance(panel, pd.DataFrame):
        raise TypeError(f"{name} 不是 DataFrame，实际类型为 {type(panel)}。")

    aligned = panel.reindex(index=signal_index, columns=stock_columns).astype(float)

    # 防止缓存或上游结果包含研究券池外的值
    aligned = aligned.where(research_mask)
    return aligned

aligned_components = {
    name: align_component_panel(panel, name)
    for name, panel in component_panels_in_memory.items()
}


# -------------------------------------------------------------------
# 3. 为缓存设置“研究口径指纹”
#    若后续改了研究范围、组件定义或信号日期，会自动失效并重建。
# -------------------------------------------------------------------
cache_config = {
    "cache_schema_version": "2026-09-28-v1",
    "purpose": "local_only_four_component_profitability_structure_proxy",
    "universe": "dynamic_csi800_research_mask",
    "signal_dates": [pd.Timestamp(d).strftime("%Y-%m-%d") for d in signal_index],
    "components": {
        "stable_delevered_roe": (
            "9_period_ttm_roe_volatility_negative_plus_"
            "cross_sectional_delevered_roe_equal_weight"
        ),
        "stable_roic_proxy": (
            "current_roic_plus_negative_9_period_roic_volatility_equal_weight"
        ),
        "stable_ronoa_proxy": (
            "current_ronoa_plus_negative_9_period_ronoa_volatility_equal_weight"
        ),
        "fcffic_reconstructed": (
            "cfo_ttm_minus_capex_ttm_divided_by_equity_plus_interest_debt_minus_cash"
        ),
    },
    "component_state": "already_industry_and_log_market_cap_neutralized_zscore",
    "storage_policy": "local_cache_only_not_for_github_export",
    "shape": [int(len(signal_index)), int(len(stock_columns))],
}

fingerprint_text = json.dumps(
    cache_config,
    ensure_ascii=False,
    sort_keys=True,
)
fingerprint = hashlib.sha256(fingerprint_text.encode("utf-8")).hexdigest()

cache_data_path = CACHE_DATA_DIR / "05_integrated_profitability_components.pkl"
cache_meta_path = CACHE_META_DIR / "05_integrated_profitability_components.json"

cache_hit = False
component_panels = None

if cache_data_path.exists() and cache_meta_path.exists():
    try:
        cached_meta = json.loads(cache_meta_path.read_text(encoding="utf-8"))

        if cached_meta.get("fingerprint") == fingerprint:
            cached_panels = pd.read_pickle(cache_data_path)

            if (
                isinstance(cached_panels, dict)
                and set(cached_panels.keys()) == set(aligned_components.keys())
            ):
                component_panels = {
                    name: align_component_panel(panel, name)
                    for name, panel in cached_panels.items()
                }
                cache_hit = True
    except Exception as exc:
        print(f"发现旧缓存但无法安全读取，将使用当前内存结果重建。原因：{exc}")

if component_panels is None:
    component_panels = aligned_components

    data_tmp_path = CACHE_DATA_DIR / "05_integrated_profitability_components.tmp.pkl"
    meta_tmp_path = CACHE_META_DIR / "05_integrated_profitability_components.tmp.json"

    pd.to_pickle(component_panels, data_tmp_path)
    data_tmp_path.replace(cache_data_path)

    cache_meta = {
        "fingerprint": fingerprint,
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "config": cache_config,
        "contains_raw_individual_stock_factor_values": True,
        "github_export_allowed": False,
    }

    meta_tmp_path.write_text(
        json.dumps(cache_meta, ensure_ascii=False, indent=2),
        encoding="utf-8",
    )
    meta_tmp_path.replace(cache_meta_path)


# -------------------------------------------------------------------
# 4. 缓存结果审计
# -------------------------------------------------------------------
component_coverage_rows = []

for factor_name, panel in component_panels.items():
    valid_count = np.isfinite(panel).sum(axis=1)

    component_coverage_rows.append(
        {
            "factor_name": factor_name,
            "mean_valid_stock_count": float(valid_count.mean()),
            "min_valid_stock_count": int(valid_count.min()),
            "max_valid_stock_count": int(valid_count.max()),
            "periods_with_at_least_100_stocks": int((valid_count >= 100).sum()),
        }
    )

component_cache_coverage = (
    pd.DataFrame(component_coverage_rows)
    .sort_values("factor_name")
    .reset_index(drop=True)
)

print("===== 四个底层子因子本地缓存状态 =====")
print("缓存命中：", "是，直接读取已有本地缓存" if cache_hit else "否，已由当前内存结果新建缓存")
print("缓存数据：", cache_data_path)
print("缓存规则：", cache_meta_path)
print()
print("===== 四个底层子因子覆盖率审计 =====")
display(component_cache_coverage)

print(
    "\n说明：cache/ 仅用于本地断点续跑，已写入 .gitignore，"
    "不得加入 artifacts/ 或 GitHub 导出包。"
)
print("下一步：运行 05.13.1，构造基于研报四个底层子因子的固定等权合成对照版本。")

In [ ]:
# 05.13.1 构造综合盈利因子等权合成对照版本
# 结构：
# 25% 稳定去杠杆 ROE
# 25% 稳定 ROIC 代理
# 25% 稳定 RONOA 代理
# 25% 重构 FCFFIC
#
# 注意：
# - 稳定 RONOA 覆盖率较低，因此当前版本是“等权合成对照 / 权重敏感性对照”，不是研报原始两层权重公式的复现；
# - 不依据样本外结果优化权重；
# - 个股层面的组件和综合因子仅进入本地 cache/，不进入 GitHub 导出目录。

from pathlib import Path
import hashlib
import json
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt


# -------------------------------------------------------------------
# 0. 路径与读取 05.13.0 的本地组件缓存
# -------------------------------------------------------------------
PROJECT_ROOT = Path("/home/rice/notebook/a-share-profitability-factor-replication")
TABLE_DIR = PROJECT_ROOT / "artifacts" / "tables"
FIGURE_DIR = PROJECT_ROOT / "artifacts" / "figures"
META_DIR = PROJECT_ROOT / "artifacts" / "metadata"

CACHE_DATA_DIR = PROJECT_ROOT / "cache" / "data"
CACHE_META_DIR = PROJECT_ROOT / "cache" / "metadata"

for directory in [TABLE_DIR, FIGURE_DIR, META_DIR, CACHE_DATA_DIR, CACHE_META_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

component_cache_path = CACHE_DATA_DIR / "05_integrated_profitability_components.pkl"
component_meta_path = CACHE_META_DIR / "05_integrated_profitability_components.json"

if not component_cache_path.exists() or not component_meta_path.exists():
    raise RuntimeError(
        "未找到 05.13.0 的组件缓存。请先运行 05.13.0，再运行本单元。"
    )

component_panels = pd.read_pickle(component_cache_path)
component_meta = json.loads(component_meta_path.read_text(encoding="utf-8"))

expected_components = {
    "stable_delevered_roe",
    "stable_roic_proxy",
    "stable_ronoa_proxy",
    "fcffic_reconstructed",
}

if set(component_panels.keys()) != expected_components:
    raise RuntimeError(
        "组件缓存内容与预期不一致。请重新运行 05.13.0。\n"
        f"当前组件：{sorted(component_panels.keys())}"
    )

signal_index = component_panels["stable_delevered_roe"].index
stock_columns = component_panels["stable_delevered_roe"].columns

for name, panel in component_panels.items():
    if not isinstance(panel, pd.DataFrame):
        raise TypeError(f"缓存中的 {name} 不是 DataFrame。")

    if not panel.index.equals(signal_index) or not panel.columns.equals(stock_columns):
        raise RuntimeError(f"组件 {name} 的日期或股票维度不一致，请重新运行 05.13.0。")


# -------------------------------------------------------------------
# 1. 缓存中性化上下文
#    本次优先从内存保存；将来重启后可直接读取此缓存。
# -------------------------------------------------------------------
context_cache_path = CACHE_DATA_DIR / "05_neutralization_context.pkl"

context_in_memory_available = all(
    name in globals()
    for name in ["research_mask", "industry_panel", "log_market_cap_panel"]
)

if context_in_memory_available:
    neutralization_context = {
        "research_mask": (
            research_mask
            .reindex(index=signal_index, columns=stock_columns)
            .fillna(False)
            .astype(bool)
        ),
        "industry_panel": industry_panel.reindex(
            index=signal_index,
            columns=stock_columns,
        ),
        "log_market_cap_panel": log_market_cap_panel.reindex(
            index=signal_index,
            columns=stock_columns,
        ).astype(float),
    }

    pd.to_pickle(neutralization_context, context_cache_path)

elif context_cache_path.exists():
    neutralization_context = pd.read_pickle(context_cache_path)

else:
    raise RuntimeError(
        "缺少中性化上下文，且没有本地上下文缓存。\n"
        "请运行 05.0、05.4 后重新运行本单元。"
    )

for name in ["research_mask", "industry_panel", "log_market_cap_panel"]:
    if name not in neutralization_context:
        raise RuntimeError(f"中性化上下文缺少 {name}，请运行 05.0、05.4 后重试。")

research_mask_for_composite = (
    neutralization_context["research_mask"]
    .reindex(index=signal_index, columns=stock_columns)
    .fillna(False)
    .astype(bool)
)
industry_for_composite = neutralization_context["industry_panel"].reindex(
    index=signal_index,
    columns=stock_columns,
)
log_cap_for_composite = neutralization_context["log_market_cap_panel"].reindex(
    index=signal_index,
    columns=stock_columns,
).astype(float)


# -------------------------------------------------------------------
# 2. 固定研究设计：当前等权合成对照版本中四个底层子因子各 25%，不允许事后调权
# -------------------------------------------------------------------
COMPONENT_WEIGHTS = {
    "stable_delevered_roe": 0.25,
    "stable_roic_proxy": 0.25,
    "stable_ronoa_proxy": 0.25,
    "fcffic_reconstructed": 0.25,
}

if not np.isclose(sum(COMPONENT_WEIGHTS.values()), 1.0):
    raise RuntimeError("组件权重之和不为 1，停止计算。")

COMPOSITE_NAME = "integrated_profitability_structure_proxy_strict"
MIN_STOCKS_FOR_FACTOR_TEST = 100
WINSOR_MAD_MULTIPLIER = 3.5

component_source_fingerprint = component_meta.get("fingerprint", "unknown")

composite_config = {
    "cache_schema_version": "2026-09-28-v1",
    "factor_name": COMPOSITE_NAME,
    "component_source_fingerprint": component_source_fingerprint,
    "weights": COMPONENT_WEIGHTS,
    "combination_rule": (
        "equal_weight_average_of_four_pre_neutralized_component_panels"
    ),
    "missing_value_rule": (
        "strict_intersection_all_four_components_must_be_finite_no_imputation"
    ),
    "post_combination_treatment": (
        "MAD_winsorize_then_CITIC2019_level1_industry_and_log_market_cap_"
        "neutralize_then_zscore"
    ),
    "minimum_stock_count_for_test": MIN_STOCKS_FOR_FACTOR_TEST,
    "signal_dates": [pd.Timestamp(d).strftime("%Y-%m-%d") for d in signal_index],
    "universe": "dynamic_csi800_research_mask",
    "github_export_allowed_for_individual_stock_panels": False,
}

composite_fingerprint = hashlib.sha256(
    json.dumps(
        composite_config,
        ensure_ascii=False,
        sort_keys=True,
    ).encode("utf-8")
).hexdigest()

composite_cache_path = (
    CACHE_DATA_DIR / "05_integrated_profitability_structure_proxy.pkl"
)
composite_meta_path = (
    CACHE_META_DIR / "05_integrated_profitability_structure_proxy.json"
)


# -------------------------------------------------------------------
# 3. 通用函数：计算相关系数、行业均值离散度、最终中性化
# -------------------------------------------------------------------
def safe_correlation(left, right):
    pair = pd.concat(
        [
            pd.to_numeric(left, errors="coerce"),
            pd.to_numeric(right, errors="coerce"),
        ],
        axis=1,
    ).dropna()

    if len(pair) < 3:
        return np.nan

    if pair.iloc[:, 0].std(ddof=0) == 0 or pair.iloc[:, 1].std(ddof=0) == 0:
        return np.nan

    return float(pair.iloc[:, 0].corr(pair.iloc[:, 1]))


def industry_mean_dispersion(values, industries):
    frame = pd.DataFrame(
        {
            "value": pd.to_numeric(values, errors="coerce"),
            "industry": industries,
        }
    ).dropna()

    if frame.empty:
        return np.nan, 0

    industry_means = frame.groupby("industry")["value"].mean()

    if len(industry_means) < 2:
        return np.nan, int(len(industry_means))

    return float(industry_means.std(ddof=0)), int(len(industry_means))


def neutralize_composite_cross_section(
    raw_factor,
    eligible_mask,
    log_market_cap,
    industry,
):
    """
    输出：
    - 最终 z-score 因子值；
    - 本期审计字典。
    """
    final_factor = pd.Series(np.nan, index=raw_factor.index, dtype=float)

    raw_factor = pd.to_numeric(raw_factor, errors="coerce")
    log_market_cap = pd.to_numeric(log_market_cap, errors="coerce")

    valid = (
        eligible_mask.fillna(False).astype(bool)
        & raw_factor.notna()
        & np.isfinite(raw_factor)
        & log_market_cap.notna()
        & np.isfinite(log_market_cap)
        & industry.notna()
    )

    raw_valid_count = int(valid.sum())

    audit = {
        "raw_valid_count": raw_valid_count,
        "model_input_count": 0,
        "final_valid_count": 0,
        "industry_groups": 0,
        "raw_cap_corr": np.nan,
        "final_cap_corr": np.nan,
        "raw_industry_dispersion": np.nan,
        "final_industry_dispersion": np.nan,
        "status": "not_run",
        "failure_reason": "",
    }

    if raw_valid_count < MIN_STOCKS_FOR_FACTOR_TEST:
        audit["status"] = "warmup_or_insufficient_coverage"
        audit["failure_reason"] = (
            f"valid_stock_count_below_{MIN_STOCKS_FOR_FACTOR_TEST}"
        )
        return final_factor, audit

    factor_valid = raw_factor.loc[valid].copy()
    cap_valid = log_market_cap.loc[valid].copy()
    industry_valid = industry.loc[valid].astype(str).copy()

    # MAD 去极值
    factor_median = factor_valid.median()
    factor_mad = (factor_valid - factor_median).abs().median()
    robust_sigma = 1.4826 * factor_mad

    if not np.isfinite(robust_sigma) or robust_sigma <= 0:
        audit["status"] = "failed"
        audit["failure_reason"] = "non_positive_or_invalid_mad"
        return final_factor, audit

    lower_bound = factor_median - WINSOR_MAD_MULTIPLIER * robust_sigma
    upper_bound = factor_median + WINSOR_MAD_MULTIPLIER * robust_sigma
    factor_winsorized = factor_valid.clip(lower=lower_bound, upper=upper_bound)

    # 常数项 + 对数市值 + CITIC 2019 一级行业虚拟变量
    industry_dummy = pd.get_dummies(
        industry_valid,
        prefix="industry",
    ).astype(float)

    # 去掉一个行业虚拟变量，避免与常数项完全共线
    if industry_dummy.shape[1] > 1:
        industry_dummy = industry_dummy.iloc[:, 1:]

    design_matrix = pd.concat(
        [
            pd.Series(1.0, index=factor_winsorized.index, name="const"),
            cap_valid.rename("log_market_cap"),
            industry_dummy,
        ],
        axis=1,
    ).astype(float)

    audit["model_input_count"] = int(len(factor_winsorized))
    audit["industry_groups"] = int(industry_valid.nunique())

    # 样本数需要明显大于回归参数数
    if len(factor_winsorized) <= design_matrix.shape[1] + 5:
        audit["status"] = "failed"
        audit["failure_reason"] = "insufficient_degrees_of_freedom"
        return final_factor, audit

    try:
        coefficients = np.linalg.lstsq(
            design_matrix.to_numpy(dtype=float),
            factor_winsorized.to_numpy(dtype=float),
            rcond=None,
        )[0]

        fitted = design_matrix.to_numpy(dtype=float) @ coefficients
        residual = pd.Series(
            factor_winsorized.to_numpy(dtype=float) - fitted,
            index=factor_winsorized.index,
            dtype=float,
        )

        residual_std = residual.std(ddof=0)

        if not np.isfinite(residual_std) or residual_std <= 0:
            audit["status"] = "failed"
            audit["failure_reason"] = "non_positive_or_invalid_residual_std"
            return final_factor, audit

        standardized_residual = (residual - residual.mean()) / residual_std
        final_factor.loc[standardized_residual.index] = standardized_residual

        raw_dispersion, _ = industry_mean_dispersion(
            factor_winsorized,
            industry_valid,
        )
        final_dispersion, _ = industry_mean_dispersion(
            standardized_residual,
            industry_valid,
        )

        audit.update(
            {
                "final_valid_count": int(standardized_residual.notna().sum()),
                "raw_cap_corr": safe_correlation(
                    factor_winsorized,
                    cap_valid,
                ),
                "final_cap_corr": safe_correlation(
                    standardized_residual,
                    cap_valid,
                ),
                "raw_industry_dispersion": raw_dispersion,
                "final_industry_dispersion": final_dispersion,
                "status": "success",
                "failure_reason": "",
            }
        )

        return final_factor, audit

    except Exception as exc:
        audit["status"] = "failed"
        audit["failure_reason"] = f"{type(exc).__name__}: {exc}"
        return final_factor, audit


# -------------------------------------------------------------------
# 4. 优先读取已完成的综合因子缓存；没有则构建并缓存
# -------------------------------------------------------------------
composite_cache_hit = False

if composite_cache_path.exists() and composite_meta_path.exists():
    try:
        cached_composite_meta = json.loads(
            composite_meta_path.read_text(encoding="utf-8")
        )

        if cached_composite_meta.get("fingerprint") == composite_fingerprint:
            composite_cache_payload = pd.read_pickle(composite_cache_path)

            integrated_profitability_pre_neutralization = (
                composite_cache_payload["pre_neutralization"]
            )
            integrated_profitability_structure_proxy_final = (
                composite_cache_payload["final_factor"]
            )
            composite_construction_audit = (
                composite_cache_payload["construction_audit"]
            )
            component_date_coverage = (
                composite_cache_payload["component_date_coverage"]
            )

            composite_cache_hit = True

    except Exception as exc:
        print(f"综合因子缓存无法安全读取，将重新构建。原因：{exc}")

if not composite_cache_hit:
    aligned_components = {
        name: panel.reindex(index=signal_index, columns=stock_columns).astype(float)
        for name, panel in component_panels.items()
    }

    component_valid_masks = {
        name: np.isfinite(panel)
        for name, panel in aligned_components.items()
    }

    # 严格交集：四个组件任一个缺失，该股票在该期不参与综合因子
    all_components_available = pd.DataFrame(
        True,
        index=signal_index,
        columns=stock_columns,
    )

    for valid_mask in component_valid_masks.values():
        all_components_available &= valid_mask

    all_components_available &= research_mask_for_composite

    integrated_profitability_pre_neutralization = pd.DataFrame(
        np.nan,
        index=signal_index,
        columns=stock_columns,
        dtype=float,
    )

    weighted_sum = pd.DataFrame(
        0.0,
        index=signal_index,
        columns=stock_columns,
        dtype=float,
    )

    for component_name, weight in COMPONENT_WEIGHTS.items():
        weighted_sum = weighted_sum + weight * aligned_components[component_name]

    integrated_profitability_pre_neutralization = weighted_sum.where(
        all_components_available
    )

    integrated_profitability_structure_proxy_final = pd.DataFrame(
        np.nan,
        index=signal_index,
        columns=stock_columns,
        dtype=float,
    )

    coverage_rows = []
    audit_rows = []

    for signal_date in signal_index:
        component_counts = {
            f"{name}_valid_count": int(
                component_valid_masks[name].loc[signal_date].sum()
            )
            for name in COMPONENT_WEIGHTS
        }

        all_component_count = int(
            all_components_available.loc[signal_date].sum()
        )

        final_values, audit = neutralize_composite_cross_section(
            raw_factor=integrated_profitability_pre_neutralization.loc[signal_date],
            eligible_mask=research_mask_for_composite.loc[signal_date],
            log_market_cap=log_cap_for_composite.loc[signal_date],
            industry=industry_for_composite.loc[signal_date],
        )

        integrated_profitability_structure_proxy_final.loc[
            signal_date,
            final_values.index,
        ] = final_values

        coverage_rows.append(
            {
                "signal_date": pd.Timestamp(signal_date),
                "research_stock_count": int(
                    research_mask_for_composite.loc[signal_date].sum()
                ),
                **component_counts,
                "all_component_intersection_count": all_component_count,
                "pre_neutralization_valid_count": int(
                    integrated_profitability_pre_neutralization.loc[
                        signal_date
                    ].notna().sum()
                ),
                "final_factor_valid_count": int(final_values.notna().sum()),
            }
        )

        audit_rows.append(
            {
                "signal_date": pd.Timestamp(signal_date),
                **audit,
            }
        )

    component_date_coverage = pd.DataFrame(coverage_rows)
    composite_construction_audit = pd.DataFrame(audit_rows)

    composite_cache_payload = {
        "pre_neutralization": integrated_profitability_pre_neutralization,
        "final_factor": integrated_profitability_structure_proxy_final,
        "construction_audit": composite_construction_audit,
        "component_date_coverage": component_date_coverage,
    }

    tmp_composite_path = (
        CACHE_DATA_DIR / "05_integrated_profitability_structure_proxy.tmp.pkl"
    )
    pd.to_pickle(composite_cache_payload, tmp_composite_path)
    tmp_composite_path.replace(composite_cache_path)

    composite_cache_meta = {
        "fingerprint": composite_fingerprint,
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "config": composite_config,
        "contains_raw_individual_stock_factor_values": True,
        "github_export_allowed": False,
    }

    tmp_composite_meta_path = (
        CACHE_META_DIR / "05_integrated_profitability_structure_proxy.tmp.json"
    )
    tmp_composite_meta_path.write_text(
        json.dumps(
            composite_cache_meta,
            ensure_ascii=False,
            indent=2,
        ),
        encoding="utf-8",
    )
    tmp_composite_meta_path.replace(composite_meta_path)


# -------------------------------------------------------------------
# 5. 生成可公开的覆盖率与质量审计
# -------------------------------------------------------------------
component_date_coverage["signal_date"] = pd.to_datetime(
    component_date_coverage["signal_date"]
)
composite_construction_audit["signal_date"] = pd.to_datetime(
    composite_construction_audit["signal_date"]
)

ready_mask = (
    component_date_coverage["final_factor_valid_count"]
    >= MIN_STOCKS_FOR_FACTOR_TEST
)

ready_dates = component_date_coverage.loc[
    ready_mask,
    "signal_date",
]

success_audit = composite_construction_audit.loc[
    composite_construction_audit["status"] == "success"
].copy()

construction_summary = pd.DataFrame(
    [
        {
            "factor_name": COMPOSITE_NAME,
            "total_signal_dates": int(len(signal_index)),
            "dates_with_at_least_100_stocks": int(ready_mask.sum()),
            "first_date_with_at_least_100_stocks": (
                ready_dates.min().strftime("%Y-%m-%d")
                if not ready_dates.empty
                else None
            ),
            "mean_final_valid_count_all_dates": float(
                component_date_coverage["final_factor_valid_count"].mean()
            ),
            "mean_final_valid_count_ready_dates": (
                float(
                    component_date_coverage.loc[
                        ready_mask,
                        "final_factor_valid_count",
                    ].mean()
                )
                if ready_mask.any()
                else np.nan
            ),
            "min_final_valid_count_ready_dates": (
                int(
                    component_date_coverage.loc[
                        ready_mask,
                        "final_factor_valid_count",
                    ].min()
                )
                if ready_mask.any()
                else np.nan
            ),
            "mean_abs_pre_neutralization_cap_corr": float(
                success_audit["raw_cap_corr"].abs().mean()
            ),
            "mean_abs_final_cap_corr": float(
                success_audit["final_cap_corr"].abs().mean()
            ),
            "mean_pre_neutralization_industry_dispersion": float(
                success_audit["raw_industry_dispersion"].mean()
            ),
            "mean_final_industry_dispersion": float(
                success_audit["final_industry_dispersion"].mean()
            ),
            "successful_neutralization_dates": int(
                (composite_construction_audit["status"] == "success").sum()
            ),
            "warmup_or_insufficient_coverage_dates": int(
                (
                    composite_construction_audit["status"]
                    == "warmup_or_insufficient_coverage"
                ).sum()
            ),
            "failed_neutralization_dates": int(
                (composite_construction_audit["status"] == "failed").sum()
            ),
        }
    ]
)

component_weight_table = pd.DataFrame(
    [
        {
            "component_name": name,
            "fixed_weight": weight,
            "weight_selection_rule": (
                "fixed_equal_weight_comparator_no_oos_optimization"
            ),
        }
        for name, weight in COMPONENT_WEIGHTS.items()
    ]
)


# -------------------------------------------------------------------
# 6. 保存 GitHub 安全的审计输出
# -------------------------------------------------------------------
coverage_output_path = (
    TABLE_DIR / "05_integrated_profitability_component_coverage.csv"
)
audit_output_path = (
    TABLE_DIR / "05_integrated_profitability_construction_audit.csv"
)
summary_output_path = (
    TABLE_DIR / "05_integrated_profitability_construction_summary.csv"
)
weight_output_path = (
    TABLE_DIR / "05_integrated_profitability_component_weights.csv"
)
metadata_output_path = (
    META_DIR / "05_integrated_profitability_structure_rules.json"
)
figure_output_path = (
    FIGURE_DIR / "05_integrated_profitability_structure_coverage.png"
)

component_date_coverage.to_csv(
    coverage_output_path,
    index=False,
    encoding="utf-8-sig",
)
composite_construction_audit.to_csv(
    audit_output_path,
    index=False,
    encoding="utf-8-sig",
)
construction_summary.to_csv(
    summary_output_path,
    index=False,
    encoding="utf-8-sig",
)
component_weight_table.to_csv(
    weight_output_path,
    index=False,
    encoding="utf-8-sig",
)

metadata_output_path.write_text(
    json.dumps(
        {
            "factor_name": COMPOSITE_NAME,
            "research_design": composite_config,
            "interpretation": (
                "This is a transparent structural proxy for the report's "
                "integrated profitability factor. It is not claimed to be "
                "an exact point-by-point reproduction."
            ),
            "coverage_caveat": (
                "The strict all-component intersection inherits the lower "
                "availability of the stable RONOA proxy. Results must be "
                "reported as a low-coverage structural sensitivity test."
            ),
            "anti_overfitting_rule": (
                "All four weights are fixed at 25%; no out-of-sample "
                "optimization or component selection is allowed."
            ),
            "export_policy": {
                "individual_stock_factor_values_saved_to_artifacts": False,
                "individual_stock_factor_values_saved_only_to_local_cache": True,
            },
        },
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)


# -------------------------------------------------------------------
# 7. 可公开图：覆盖率与最终中性化质量
# -------------------------------------------------------------------
try:
    plt.style.use("seaborn-v0_8-whitegrid")
except OSError:
    plt.style.use("default")

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

coverage_plot_columns = {
    "stable_delevered_roe_valid_count": "Stable delevered ROE",
    "stable_roic_proxy_valid_count": "Stable ROIC proxy",
    "stable_ronoa_proxy_valid_count": "Stable RONOA proxy",
    "fcffic_reconstructed_valid_count": "Reconstructed FCFFIC",
    "final_factor_valid_count": "Final composite",
}

for column, label in coverage_plot_columns.items():
    axes[0].plot(
        component_date_coverage["signal_date"],
        component_date_coverage[column],
        label=label,
        linewidth=1.8 if column == "final_factor_valid_count" else 1.2,
    )

axes[0].axhline(
    MIN_STOCKS_FOR_FACTOR_TEST,
    color="gray",
    linestyle="--",
    linewidth=1,
    label="Minimum test threshold",
)
axes[0].set_title("Integrated Profitability Structure Proxy Coverage")
axes[0].set_xlabel("Signal Date")
axes[0].set_ylabel("Valid Stock Count")
axes[0].legend(fontsize=8, loc="best")

quality_plot = pd.DataFrame(
    {
        "Before final neutralization": [
            success_audit["raw_cap_corr"].abs().mean(),
            success_audit["raw_industry_dispersion"].mean(),
        ],
        "After final neutralization": [
            success_audit["final_cap_corr"].abs().mean(),
            success_audit["final_industry_dispersion"].mean(),
        ],
    },
    index=[
        "Abs. correlation with log market cap",
        "Industry-mean dispersion",
    ],
)

quality_plot.plot(
    kind="bar",
    ax=axes[1],
    color=["#4C78A8", "#F58518"],
    rot=15,
)
axes[1].set_title("Post-Combination Neutralization Quality")
axes[1].set_xlabel("")
axes[1].set_ylabel("Mean value across successful dates")
axes[1].legend(fontsize=8)

fig.tight_layout()
fig.savefig(figure_output_path, dpi=160, bbox_inches="tight")
plt.show()


# -------------------------------------------------------------------
# 8. 控制台结论
# -------------------------------------------------------------------
print("===== 综合盈利因子等权合成对照版本构建完成 =====")
print(
    "综合因子缓存：",
    "命中，直接读取已有本地缓存"
    if composite_cache_hit
    else "未命中，已完成构建并写入本地缓存",
)
print("本地综合因子缓存：", composite_cache_path)
print()
display(component_weight_table)
display(construction_summary)

print("\n已保存：", coverage_output_path)
print("已保存：", audit_output_path)
print("已保存：", summary_output_path)
print("已保存：", weight_output_path)
print("已保存：", metadata_output_path)
print("已保存：", figure_output_path)

print(
    "\n下一步：使用固定的可用期间检验该综合因子。"
    "预期可用期约为 84 个信号日，其中样本内约 48 期、样本外 36 期；"
    "不会为了提高结果而删除或挑选样本外日期。"
)

In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/home/rice/notebook/a-share-profitability-factor-replication")
(PROJECT_ROOT / "lib").mkdir(parents=True, exist_ok=True)

print("库目录已就绪：", PROJECT_ROOT / "lib")

In [ ]:
%%writefile /home/rice/notebook/a-share-profitability-factor-replication/lib/profitability_research_utils.py
"""
A 股盈利能力因子研究：可复用基础库

设计原则：
1. cache/ 只存本地个股级中间数据，不进入 GitHub；
2. artifacts/ 只存可公开的汇总表、规则文件和图；
3. 研究公式、样本范围、固定权重仍留在 Notebook 显式展示；
4. 所有缓存均使用配置指纹，研究口径变动时自动失效重建。
"""

from __future__ import annotations

from dataclasses import dataclass
from datetime import datetime
from pathlib import Path
from typing import Any, Callable, Mapping

import hashlib
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


LIBRARY_VERSION = "0.1.0"


@dataclass(frozen=True)
class ProjectPaths:
    """项目中所有标准目录。"""

    root: Path
    table_dir: Path
    figure_dir: Path
    metadata_dir: Path
    cache_data_dir: Path
    cache_metadata_dir: Path

    @classmethod
    def from_root(cls, root: str | Path) -> "ProjectPaths":
        root = Path(root)
        return cls(
            root=root,
            table_dir=root / "artifacts" / "tables",
            figure_dir=root / "artifacts" / "figures",
            metadata_dir=root / "artifacts" / "metadata",
            cache_data_dir=root / "cache" / "data",
            cache_metadata_dir=root / "cache" / "metadata",
        )

    def ensure_directories(self) -> None:
        for directory in [
            self.table_dir,
            self.figure_dir,
            self.metadata_dir,
            self.cache_data_dir,
            self.cache_metadata_dir,
        ]:
            directory.mkdir(parents=True, exist_ok=True)

    def ensure_cache_ignored(self) -> None:
        """确保本地个股级缓存不会被 Git 跟踪。"""
        gitignore_path = self.root / ".gitignore"
        current = (
            gitignore_path.read_text(encoding="utf-8")
            if gitignore_path.exists()
            else ""
        )

        if "cache/" not in current.splitlines():
            with gitignore_path.open("a", encoding="utf-8") as f:
                if current and not current.endswith("\n"):
                    f.write("\n")
                f.write("cache/\n")


def cache_fingerprint(config: Mapping[str, Any]) -> str:
    """根据研究口径生成稳定的 SHA-256 指纹。"""
    text = json.dumps(
        dict(config),
        ensure_ascii=False,
        sort_keys=True,
        default=str,
    )
    return hashlib.sha256(text.encode("utf-8")).hexdigest()


def load_or_build_pickle_cache(
    data_path: str | Path,
    metadata_path: str | Path,
    config: Mapping[str, Any],
    builder: Callable[[], Any],
    contains_individual_stock_values: bool = True,
) -> tuple[Any, bool]:
    """
    读取与当前配置一致的本地缓存；没有命中时才执行 builder 并保存。

    返回：
        payload, cache_hit
    """
    data_path = Path(data_path)
    metadata_path = Path(metadata_path)
    data_path.parent.mkdir(parents=True, exist_ok=True)
    metadata_path.parent.mkdir(parents=True, exist_ok=True)

    fingerprint = cache_fingerprint(config)

    if data_path.exists() and metadata_path.exists():
        try:
            metadata = json.loads(metadata_path.read_text(encoding="utf-8"))

            if metadata.get("fingerprint") == fingerprint:
                return pd.read_pickle(data_path), True

        except Exception as exc:
            print(f"缓存读取失败，将重建：{type(exc).__name__}: {exc}")

    payload = builder()

    temporary_data_path = data_path.with_suffix(data_path.suffix + ".tmp")
    temporary_metadata_path = metadata_path.with_suffix(
        metadata_path.suffix + ".tmp"
    )

    pd.to_pickle(payload, temporary_data_path)
    temporary_data_path.replace(data_path)

    metadata = {
        "fingerprint": fingerprint,
        "created_at": datetime.now().isoformat(timespec="seconds"),
        "config": dict(config),
        "contains_individual_stock_values": contains_individual_stock_values,
        "github_export_allowed": False,
    }

    temporary_metadata_path.write_text(
        json.dumps(metadata, ensure_ascii=False, indent=2, default=str),
        encoding="utf-8",
    )
    temporary_metadata_path.replace(metadata_path)

    return payload, False


def save_public_csv(
    frame: pd.DataFrame,
    path: str | Path,
) -> Path:
    """保存可公开的汇总表；调用者不得传入个股级原始因子或价格。"""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    frame.to_csv(path, index=False, encoding="utf-8-sig")
    return path


def save_public_json(
    content: Mapping[str, Any],
    path: str | Path,
) -> Path:
    """保存可公开的规则和研究元数据。"""
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(
        json.dumps(content, ensure_ascii=False, indent=2, default=str),
        encoding="utf-8",
    )
    return path


def configure_matplotlib_style() -> None:
    """兼容不同 matplotlib 版本。"""
    try:
        plt.style.use("seaborn-v0_8-whitegrid")
    except OSError:
        plt.style.use("default")


def safe_correlation(left: pd.Series, right: pd.Series) -> float:
    """计算两序列的 Pearson 相关系数；样本不足时返回 NaN。"""
    paired = pd.concat(
        [
            pd.to_numeric(left, errors="coerce"),
            pd.to_numeric(right, errors="coerce"),
        ],
        axis=1,
    ).dropna()

    if len(paired) < 3:
        return np.nan

    if paired.iloc[:, 0].std(ddof=0) == 0:
        return np.nan

    if paired.iloc[:, 1].std(ddof=0) == 0:
        return np.nan

    return float(paired.iloc[:, 0].corr(paired.iloc[:, 1]))


def mad_winsorize(
    values: pd.Series,
    multiplier: float = 3.5,
) -> pd.Series:
    """使用 median ± multiplier × 1.4826 × MAD 去极值。"""
    values = pd.to_numeric(values, errors="coerce").copy()
    valid = values.dropna()

    if valid.empty:
        return values

    median = valid.median()
    mad = (valid - median).abs().median()
    robust_sigma = 1.4826 * mad

    if not np.isfinite(robust_sigma) or robust_sigma <= 0:
        return values

    return values.clip(
        lower=median - multiplier * robust_sigma,
        upper=median + multiplier * robust_sigma,
    )


def industry_mean_dispersion(
    values: pd.Series,
    industries: pd.Series,
) -> tuple[float, int]:
    """返回一级行业均值的标准差和有效行业数。"""
    frame = pd.DataFrame(
        {
            "value": pd.to_numeric(values, errors="coerce"),
            "industry": industries,
        }
    ).dropna()

    if frame.empty:
        return np.nan, 0

    industry_means = frame.groupby("industry")["value"].mean()

    if len(industry_means) < 2:
        return np.nan, int(len(industry_means))

    return float(industry_means.std(ddof=0)), int(len(industry_means))


def neutralize_cross_section(
    raw_factor: pd.Series,
    eligible_mask: pd.Series,
    log_market_cap: pd.Series,
    industry: pd.Series,
    min_stocks: int = 100,
    winsor_multiplier: float = 3.5,
) -> tuple[pd.Series, dict[str, Any]]:
    """
    单期标准横截面处理：
    MAD 去极值 → 一级行业 + log 市值回归 → 残差 z-score。

    返回：
        final_factor: 与 raw_factor 同索引的 Series
        audit: 覆盖率与中性化质量审计
    """
    final_factor = pd.Series(np.nan, index=raw_factor.index, dtype=float)

    raw_factor = pd.to_numeric(raw_factor, errors="coerce")
    log_market_cap = pd.to_numeric(log_market_cap, errors="coerce")

    valid = (
        eligible_mask.fillna(False).astype(bool)
        & raw_factor.notna()
        & np.isfinite(raw_factor)
        & log_market_cap.notna()
        & np.isfinite(log_market_cap)
        & industry.notna()
    )

    raw_valid_count = int(valid.sum())

    audit = {
        "raw_valid_count": raw_valid_count,
        "model_input_count": 0,
        "final_valid_count": 0,
        "industry_groups": 0,
        "raw_cap_corr": np.nan,
        "final_cap_corr": np.nan,
        "raw_industry_dispersion": np.nan,
        "final_industry_dispersion": np.nan,
        "status": "not_run",
        "failure_reason": "",
    }

    if raw_valid_count < min_stocks:
        audit["status"] = "insufficient_coverage"
        audit["failure_reason"] = f"valid_stock_count_below_{min_stocks}"
        return final_factor, audit

    factor_valid = mad_winsorize(
        raw_factor.loc[valid],
        multiplier=winsor_multiplier,
    )
    cap_valid = log_market_cap.loc[valid]
    industry_valid = industry.loc[valid].astype(str)

    industry_dummy = pd.get_dummies(
        industry_valid,
        prefix="industry",
    ).astype(float)

    if industry_dummy.shape[1] > 1:
        industry_dummy = industry_dummy.iloc[:, 1:]

    design_matrix = pd.concat(
        [
            pd.Series(1.0, index=factor_valid.index, name="const"),
            cap_valid.rename("log_market_cap"),
            industry_dummy,
        ],
        axis=1,
    ).astype(float)

    audit["model_input_count"] = int(len(factor_valid))
    audit["industry_groups"] = int(industry_valid.nunique())

    if len(factor_valid) <= design_matrix.shape[1] + 5:
        audit["status"] = "failed"
        audit["failure_reason"] = "insufficient_degrees_of_freedom"
        return final_factor, audit

    try:
        beta = np.linalg.lstsq(
            design_matrix.to_numpy(dtype=float),
            factor_valid.to_numpy(dtype=float),
            rcond=None,
        )[0]

        residual = pd.Series(
            factor_valid.to_numpy(dtype=float)
            - design_matrix.to_numpy(dtype=float) @ beta,
            index=factor_valid.index,
        )

        residual_std = residual.std(ddof=0)

        if not np.isfinite(residual_std) or residual_std <= 0:
            audit["status"] = "failed"
            audit["failure_reason"] = "invalid_residual_std"
            return final_factor, audit

        final_values = (residual - residual.mean()) / residual_std
        final_factor.loc[final_values.index] = final_values

        raw_dispersion, _ = industry_mean_dispersion(
            factor_valid,
            industry_valid,
        )
        final_dispersion, _ = industry_mean_dispersion(
            final_values,
            industry_valid,
        )

        audit.update(
            {
                "final_valid_count": int(final_values.notna().sum()),
                "raw_cap_corr": safe_correlation(factor_valid, cap_valid),
                "final_cap_corr": safe_correlation(final_values, cap_valid),
                "raw_industry_dispersion": raw_dispersion,
                "final_industry_dispersion": final_dispersion,
                "status": "success",
            }
        )

    except Exception as exc:
        audit["status"] = "failed"
        audit["failure_reason"] = f"{type(exc).__name__}: {exc}"

    return final_factor, audit


def _one_sample_t_stat(values: pd.Series) -> float:
    values = pd.to_numeric(values, errors="coerce").dropna()

    if len(values) < 2:
        return np.nan

    standard_deviation = values.std(ddof=1)

    if not np.isfinite(standard_deviation) or standard_deviation == 0:
        return np.nan

    return float(values.mean() / (standard_deviation / np.sqrt(len(values))))


def _annualized_ir(values: pd.Series, periods_per_year: int = 12) -> float:
    values = pd.to_numeric(values, errors="coerce").dropna()

    if len(values) < 2:
        return np.nan

    standard_deviation = values.std(ddof=1)

    if not np.isfinite(standard_deviation) or standard_deviation == 0:
        return np.nan

    return float(values.mean() / standard_deviation * np.sqrt(periods_per_year))


def evaluate_factor_panel(
    factor_panel: pd.DataFrame,
    forward_return_panel: pd.DataFrame,
    min_stocks: int = 100,
    quantiles: int = 5,
) -> pd.DataFrame:
    """
    对月度横截面因子做 IC、RankIC 和等权五分组检验。

    返回的表仅含逐期汇总结果，不含个股级因子或收益。
    """
    forward_return_panel = forward_return_panel.reindex(
        index=factor_panel.index,
        columns=factor_panel.columns,
    )

    records = []

    for signal_date in factor_panel.index:
        paired = pd.DataFrame(
            {
                "factor": pd.to_numeric(
                    factor_panel.loc[signal_date],
                    errors="coerce",
                ),
                "forward_return": pd.to_numeric(
                    forward_return_panel.loc[signal_date],
                    errors="coerce",
                ),
            }
        ).dropna()

        record = {
            "signal_date": pd.Timestamp(signal_date),
            "paired_stock_count": int(len(paired)),
            "ic": np.nan,
            "rank_ic": np.nan,
            "q1_return": np.nan,
            "q2_return": np.nan,
            "q3_return": np.nan,
            "q4_return": np.nan,
            "q5_return": np.nan,
            "high_minus_low_return": np.nan,
            "status": "insufficient_coverage",
        }

        if len(paired) < min_stocks:
            records.append(record)
            continue

        if paired["factor"].std(ddof=0) == 0:
            record["status"] = "zero_factor_variance"
            records.append(record)
            continue

        record["ic"] = float(
            paired["factor"].corr(paired["forward_return"], method="pearson")
        )
        record["rank_ic"] = float(
            paired["factor"].corr(paired["forward_return"], method="spearman")
        )

        try:
            ranks = paired["factor"].rank(method="first")
            quintile = pd.qcut(
                ranks,
                q=quantiles,
                labels=False,
            ) + 1

            quintile_return = paired.groupby(quintile)["forward_return"].mean()

            for group_number in range(1, quantiles + 1):
                record[f"q{group_number}_return"] = quintile_return.get(
                    group_number,
                    np.nan,
                )

            record["high_minus_low_return"] = (
                record[f"q{quantiles}_return"] - record["q1_return"]
            )
            record["status"] = "success"

        except Exception as exc:
            record["status"] = f"quintile_failure: {type(exc).__name__}"

        records.append(record)

    return pd.DataFrame(records)


def summarize_factor_evaluation(
    factor_name: str,
    monthly_metrics: pd.DataFrame,
    periods: Mapping[str, tuple[str | None, str | None]],
    periods_per_year: int = 12,
) -> pd.DataFrame:
    """
    按预先指定的时间窗口汇总因子表现。

    periods 示例：
    {
        "all_available_periods": (None, None),
        "in_sample_2016_2022": ("2016-01-01", "2022-12-31"),
        "out_of_sample_2023_2025": ("2023-01-01", "2025-12-31"),
    }
    """
    metrics = monthly_metrics.copy()
    metrics["signal_date"] = pd.to_datetime(metrics["signal_date"])

    rows = []

    for period_name, (start_date, end_date) in periods.items():
        selected = metrics.copy()

        if start_date is not None:
            selected = selected[
                selected["signal_date"] >= pd.Timestamp(start_date)
            ]

        if end_date is not None:
            selected = selected[
                selected["signal_date"] <= pd.Timestamp(end_date)
            ]

        selected = selected[selected["status"] == "success"].copy()

        rows.append(
            {
                "factor_name": factor_name,
                "period": period_name,
                "observations": int(len(selected)),
                "mean_paired_stock_count": (
                    float(selected["paired_stock_count"].mean())
                    if not selected.empty
                    else np.nan
                ),
                "min_paired_stock_count": (
                    int(selected["paired_stock_count"].min())
                    if not selected.empty
                    else np.nan
                ),
                "mean_ic": (
                    float(selected["ic"].mean())
                    if not selected.empty
                    else np.nan
                ),
                "icir_annualized": _annualized_ir(
                    selected["ic"],
                    periods_per_year=periods_per_year,
                ),
                "mean_rank_ic": (
                    float(selected["rank_ic"].mean())
                    if not selected.empty
                    else np.nan
                ),
                "rank_icir_annualized": _annualized_ir(
                    selected["rank_ic"],
                    periods_per_year=periods_per_year,
                ),
                "positive_rank_ic_ratio": (
                    float((selected["rank_ic"] > 0).mean())
                    if not selected.empty
                    else np.nan
                ),
                "mean_q1_return": (
                    float(selected["q1_return"].mean())
                    if not selected.empty
                    else np.nan
                ),
                "mean_q5_return": (
                    float(selected["q5_return"].mean())
                    if not selected.empty
                    else np.nan
                ),
                "mean_high_minus_low_return": (
                    float(selected["high_minus_low_return"].mean())
                    if not selected.empty
                    else np.nan
                ),
                "high_minus_low_t_stat": _one_sample_t_stat(
                    selected["high_minus_low_return"]
                ),
            }
        )

    return pd.DataFrame(rows)

In [ ]:
from pathlib import Path
import importlib
import sys

PROJECT_ROOT = Path("/home/rice/notebook/a-share-profitability-factor-replication")

if str(PROJECT_ROOT / "lib") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "lib"))

import profitability_research_utils as pru
pru = importlib.reload(pru)

PATHS = pru.ProjectPaths.from_root(PROJECT_ROOT)
PATHS.ensure_directories()
PATHS.ensure_cache_ignored()

print("基础库导入成功。")
print("版本：", pru.LIBRARY_VERSION)
print("缓存目录：", PATHS.cache_data_dir)

In [ ]:
# 05.13.2a 缓存既有的 20 日前瞻收益标签
# 第一次运行需要内存中已有 forward_return_20d；
# 若当前会话没有它，只需先运行原来的 05.6 一次，再回来运行本单元。

import json
import pandas as pd

composite_payload = pd.read_pickle(
    PATHS.cache_data_dir / "05_integrated_profitability_structure_proxy.pkl"
)
integrated_profitability_structure_proxy_final = composite_payload["final_factor"]

return_cache_config = {
    "cache_schema_version": "2026-09-28-v1",
    "label_definition": "signal_date_T_plus_1_open_to_T_plus_21_open_20_trading_days",
    "signal_dates": [
        pd.Timestamp(d).strftime("%Y-%m-%d")
        for d in integrated_profitability_structure_proxy_final.index
    ],
    "universe_columns": int(
        len(integrated_profitability_structure_proxy_final.columns)
    ),
    "storage_policy": "local_only_not_for_github_export",
}

def build_forward_return_cache():
    if "forward_return_20d" not in globals():
        raise RuntimeError(
            "当前内存没有 forward_return_20d，且这是第一次建立收益标签缓存。\n"
            "请先运行原来的 05.6 一次；运行完成后，再重新运行本单元。"
        )

    return (
        forward_return_20d
        .reindex(
            index=integrated_profitability_structure_proxy_final.index,
            columns=integrated_profitability_structure_proxy_final.columns,
        )
        .astype(float)
    )

forward_return_20d_cached, return_cache_hit = pru.load_or_build_pickle_cache(
    data_path=PATHS.cache_data_dir / "05_forward_return_20d.pkl",
    metadata_path=PATHS.cache_metadata_dir / "05_forward_return_20d.json",
    config=return_cache_config,
    builder=build_forward_return_cache,
    contains_individual_stock_values=True,
)

print(
    "收益标签缓存：",
    "命中，直接读取本地缓存" if return_cache_hit else "未命中，已由当前内存结果新建缓存",
)
print("本地缓存：", PATHS.cache_data_dir / "05_forward_return_20d.pkl")
print(
    "收益标签非空数：",
    int(forward_return_20d_cached.notna().sum().sum()),
)

In [ ]:
%%writefile -a /home/rice/notebook/a-share-profitability-factor-replication/lib/profitability_research_utils.py


def run_integrated_profitability_factor_test(
    project_root: str | Path,
    factor_name: str = "integrated_profitability_structure_proxy_strict",
    min_stocks: int = 100,
) -> dict[str, Any]:
    """
    从本地缓存读取综合盈利因子与 20 日收益标签，完成全套检验并保存公开产物。

    不重新请求数据。
    不把个股级因子值、个股收益或价格写入 artifacts/。
    """
    paths = ProjectPaths.from_root(project_root)
    paths.ensure_directories()
    paths.ensure_cache_ignored()

    composite_cache_path = (
        paths.cache_data_dir
        / "05_integrated_profitability_structure_proxy.pkl"
    )
    forward_return_cache_path = (
        paths.cache_data_dir
        / "05_forward_return_20d.pkl"
    )

    if not composite_cache_path.exists():
        raise FileNotFoundError(
            "缺少综合因子缓存，请先运行 05.13.1。"
        )

    if not forward_return_cache_path.exists():
        raise FileNotFoundError(
            "缺少收益标签缓存，请先运行 05.13.2a。"
        )

    composite_payload = pd.read_pickle(composite_cache_path)

    if "final_factor" not in composite_payload:
        raise RuntimeError(
            "综合因子缓存中缺少 final_factor，请重新运行 05.13.1。"
        )

    factor_panel = composite_payload["final_factor"]
    forward_return_panel = pd.read_pickle(
        forward_return_cache_path
    ).reindex(
        index=factor_panel.index,
        columns=factor_panel.columns,
    )

    monthly_metrics = evaluate_factor_panel(
        factor_panel=factor_panel,
        forward_return_panel=forward_return_panel,
        min_stocks=min_stocks,
        quantiles=5,
    )

    successful_metrics = monthly_metrics.loc[
        monthly_metrics["status"] == "success"
    ].copy()

    if successful_metrics.empty:
        raise RuntimeError(
            f"没有日期满足最低 {min_stocks} 只股票的检验门槛。"
        )

    first_available_date = successful_metrics["signal_date"].min()
    last_available_date = successful_metrics["signal_date"].max()

    periods = {
        (
            f"all_available_"
            f"{first_available_date.year}_{last_available_date.year}"
        ): (
            first_available_date.strftime("%Y-%m-%d"),
            last_available_date.strftime("%Y-%m-%d"),
        ),
        f"in_sample_available_{first_available_date.year}_2022": (
            first_available_date.strftime("%Y-%m-%d"),
            "2022-12-31",
        ),
        "out_of_sample_2023_2025": (
            "2023-01-01",
            "2025-12-31",
        ),
    }

    effectiveness_summary = summarize_factor_evaluation(
        factor_name=factor_name,
        monthly_metrics=monthly_metrics,
        periods=periods,
        periods_per_year=12,
    )

    test_failures = monthly_metrics.loc[
        monthly_metrics["status"] != "success"
    ].copy()

    ic_rankic_path = (
        paths.table_dir
        / "05_integrated_profitability_ic_rankic_timeseries.csv"
    )
    quintile_path = (
        paths.table_dir
        / "05_integrated_profitability_quintile_return_timeseries.csv"
    )
    summary_path = (
        paths.table_dir
        / "05_integrated_profitability_effectiveness_summary.csv"
    )
    failures_path = (
        paths.table_dir
        / "05_integrated_profitability_factor_test_failures.csv"
    )
    metadata_path = (
        paths.metadata_dir
        / "05_integrated_profitability_factor_test_rules.json"
    )
    figure_path = (
        paths.figure_dir
        / "05_integrated_profitability_factor_effectiveness.png"
    )

    # 仅保存日期级汇总数据，不保存个股级数据
    save_public_csv(
        monthly_metrics[
            [
                "signal_date",
                "paired_stock_count",
                "ic",
                "rank_ic",
                "status",
            ]
        ],
        ic_rankic_path,
    )

    save_public_csv(
        monthly_metrics[
            [
                "signal_date",
                "paired_stock_count",
                "q1_return",
                "q2_return",
                "q3_return",
                "q4_return",
                "q5_return",
                "high_minus_low_return",
                "status",
            ]
        ],
        quintile_path,
    )

    save_public_csv(effectiveness_summary, summary_path)
    save_public_csv(test_failures, failures_path)

    save_public_json(
        {
            "factor_name": factor_name,
            "fixed_component_weights": {
                "stable_delevered_roe": 0.25,
                "stable_roic_proxy": 0.25,
                "stable_ronoa_proxy": 0.25,
                "fcffic_reconstructed": 0.25,
            },
            "component_missing_value_rule": (
                "All four components must be available; "
                "no stock-level imputation is allowed."
            ),
            "post_combination_treatment": (
                "MAD winsorization, CITIC 2019 level-1 industry "
                "neutralization, log-market-cap neutralization, z-score."
            ),
            "return_label": (
                "T+1 open to T+21 open, "
                "20 trading-day holding return."
            ),
            "minimum_stocks_for_factor_test": min_stocks,
            "first_available_date": first_available_date.strftime(
                "%Y-%m-%d"
            ),
            "last_available_date": last_available_date.strftime(
                "%Y-%m-%d"
            ),
            "in_sample_period": (
                f"{first_available_date.strftime('%Y-%m-%d')} to 2022-12-31"
            ),
            "out_of_sample_period": "2023-01-01 to 2025-12-31",
            "anti_overfitting_rule": (
                "No component weights, components, or out-of-sample dates "
                "are changed after observing results."
            ),
            "individual_stock_level_data_saved_to_artifacts": False,
        },
        metadata_path,
    )

    configure_matplotlib_style()

    plot_metrics = successful_metrics.sort_values("signal_date").copy()

    fig, axes = plt.subplots(1, 2, figsize=(16, 5))

    axes[0].plot(
        plot_metrics["signal_date"],
        plot_metrics["ic"],
        label="IC",
        linewidth=1.2,
    )
    axes[0].plot(
        plot_metrics["signal_date"],
        plot_metrics["rank_ic"],
        label="RankIC",
        linewidth=1.2,
    )
    axes[0].axhline(0, color="gray", linestyle="--", linewidth=1)
    axes[0].set_title("Integrated Profitability: Monthly IC and RankIC")
    axes[0].set_xlabel("Signal Date")
    axes[0].set_ylabel("Correlation")
    axes[0].legend()

    nav_frame = pd.DataFrame(
        {
            "signal_date": plot_metrics["signal_date"],
            "Q1": (1 + plot_metrics["q1_return"]).cumprod(),
            "Q2": (1 + plot_metrics["q2_return"]).cumprod(),
            "Q3": (1 + plot_metrics["q3_return"]).cumprod(),
            "Q4": (1 + plot_metrics["q4_return"]).cumprod(),
            "Q5": (1 + plot_metrics["q5_return"]).cumprod(),
        }
    ).set_index("signal_date")

    for quintile_name in ["Q1", "Q2", "Q3", "Q4", "Q5"]:
        axes[1].plot(
            nav_frame.index,
            nav_frame[quintile_name],
            label=quintile_name,
            linewidth=1.2,
        )

    axes[1].set_title(
        "Integrated Profitability: Equal-Weighted Quintile NAV"
    )
    axes[1].set_xlabel("Signal Date")
    axes[1].set_ylabel("Gross Net Value")
    axes[1].legend()

    fig.tight_layout()
    fig.savefig(figure_path, dpi=160, bbox_inches="tight")
    plt.show()

    result = {
        "summary": effectiveness_summary,
        "monthly_metrics": monthly_metrics,
        "first_available_date": first_available_date,
        "last_available_date": last_available_date,
        "successful_periods": int(len(successful_metrics)),
        "failed_periods": int(len(test_failures)),
        "paths": {
            "ic_rankic": ic_rankic_path,
            "quintile_returns": quintile_path,
            "summary": summary_path,
            "failures": failures_path,
            "metadata": metadata_path,
            "figure": figure_path,
        },
    }

    print("===== 综合盈利因子等权合成对照版本有效性汇总 =====")
    print(effectiveness_summary.round(4).to_string(index=False))
    print()
    print("可检验首日：", first_available_date.strftime("%Y-%m-%d"))
    print("可检验末日：", last_available_date.strftime("%Y-%m-%d"))
    print("可检验期数：", len(successful_metrics))
    print("未通过覆盖率门槛的期数：", len(test_failures))
    print("已保存：", summary_path)
    print("已保存：", figure_path)

    return result

In [ ]:
#5.13.2
import importlib

pru = importlib.reload(pru)
integrated_factor_test = pru.run_integrated_profitability_factor_test(PROJECT_ROOT)

In [ ]:
%%writefile -a /home/rice/notebook/a-share-profitability-factor-replication/lib/profitability_research_utils.py


def run_out_of_sample_decision_from_summary(
    project_root: str | Path,
    summary_filename: str,
    output_prefix: str,
    factor_name: str,
    oos_period: str = "out_of_sample_2023_2025",
    decision_scope: str = "standard_factor_test",
    min_observations: int = 24,
    min_mean_paired_stocks: int = 100,
    min_positive_rank_ic_ratio: float = 0.55,
    min_rank_icir_annualized: float = 0.50,
    min_hml_t_stat: float = 1.96,
) -> dict[str, Any]:
    """
    基于已保存的样本外汇总表，生成统一、可复用的研究决策日志。

    判定规则：
    - 数据不足：inconclusive_insufficient_coverage；
    - RankIC 或 Q5-Q1 非正：reject_as_standalone_factor；
    - RankICIR、正 RankIC 比例、HML t 值均达阈值：confirmed_candidate；
    - 其余正向但证据不足：candidate_not_confirmed。

    该函数不会根据结果修改阈值、权重或样本区间。
    """
    paths = ProjectPaths.from_root(project_root)
    paths.ensure_directories()

    summary_path = paths.table_dir / summary_filename

    if not summary_path.exists():
        raise FileNotFoundError(
            f"找不到汇总表：{summary_path}"
        )

    summary = pd.read_csv(summary_path)

    required_columns = {
        "factor_name",
        "period",
        "observations",
        "mean_paired_stock_count",
        "mean_rank_ic",
        "rank_icir_annualized",
        "positive_rank_ic_ratio",
        "mean_high_minus_low_return",
        "high_minus_low_t_stat",
    }

    missing_columns = required_columns - set(summary.columns)

    if missing_columns:
        raise RuntimeError(
            "汇总表缺少字段：" + ", ".join(sorted(missing_columns))
        )

    selected = summary.loc[
        (summary["factor_name"] == factor_name)
        & (summary["period"] == oos_period)
    ].copy()

    if len(selected) != 1:
        raise RuntimeError(
            f"未能唯一定位 {factor_name} 在 {oos_period} 的样本外记录。"
        )

    row = selected.iloc[0]

    observations = int(row["observations"])
    mean_paired_stocks = float(row["mean_paired_stock_count"])
    mean_rank_ic = float(row["mean_rank_ic"])
    rank_icir = float(row["rank_icir_annualized"])
    positive_ratio = float(row["positive_rank_ic_ratio"])
    mean_hml = float(row["mean_high_minus_low_return"])
    hml_t_stat = float(row["high_minus_low_t_stat"])

    if (
        observations < min_observations
        or mean_paired_stocks < min_mean_paired_stocks
    ):
        decision = "inconclusive_insufficient_coverage"
        decision_reason = (
            "样本外观测数或平均配对股票数未达到预先设定的最低门槛。"
        )

    elif mean_rank_ic <= 0 or mean_hml <= 0:
        decision = "reject_as_standalone_factor"
        decision_reason = (
            "样本外平均 RankIC 或 Q5-Q1 收益为非正值。"
        )

    elif (
        rank_icir >= min_rank_icir_annualized
        and positive_ratio >= min_positive_rank_ic_ratio
        and hml_t_stat >= min_hml_t_stat
    ):
        decision = "confirmed_candidate"
        decision_reason = (
            "样本外方向、稳定性和高低组收益显著性均达到预设门槛。"
        )

    else:
        decision = "candidate_not_confirmed"
        decision_reason = (
            "样本外 RankIC 与 Q5-Q1 为正，但稳定性或显著性"
            "尚未达到预设确认门槛。"
        )

    decision_log = pd.DataFrame(
        [
            {
                "factor_name": factor_name,
                "period": oos_period,
                "observations": observations,
                "mean_paired_stock_count": mean_paired_stocks,
                "mean_rank_ic": mean_rank_ic,
                "rank_icir_annualized": rank_icir,
                "positive_rank_ic_ratio": positive_ratio,
                "mean_high_minus_low_return": mean_hml,
                "high_minus_low_t_stat": hml_t_stat,
                "decision": decision,
                "decision_scope": decision_scope,
                "decision_reason": decision_reason,
            }
        ]
    )

    decision_path = (
        paths.table_dir / f"{output_prefix}_research_decision_log.csv"
    )
    metadata_path = (
        paths.metadata_dir / f"{output_prefix}_research_decision_rules.json"
    )
    figure_path = (
        paths.figure_dir / f"{output_prefix}_out_of_sample_scorecard.png"
    )

    save_public_csv(decision_log, decision_path)

    save_public_json(
        {
            "factor_name": factor_name,
            "out_of_sample_period": oos_period,
            "decision_scope": decision_scope,
            "predefined_decision_thresholds": {
                "minimum_observations": min_observations,
                "minimum_mean_paired_stocks": min_mean_paired_stocks,
                "minimum_positive_rank_ic_ratio": min_positive_rank_ic_ratio,
                "minimum_rank_icir_annualized": min_rank_icir_annualized,
                "minimum_high_minus_low_t_stat": min_hml_t_stat,
            },
            "decision_rule": (
                "Reject if mean RankIC <= 0 or mean Q5-Q1 <= 0; "
                "confirm only when all predefined stability and "
                "significance thresholds are met; otherwise retain "
                "as candidate_not_confirmed."
            ),
            "anti_overfitting_rule": (
                "No weights, components, thresholds, or sample dates "
                "are modified after observing the result."
            ),
        },
        metadata_path,
    )

    configure_matplotlib_style()

    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    axes[0].bar(
        ["Mean RankIC"],
        [mean_rank_ic],
        color="#54A24B" if mean_rank_ic > 0 else "#E45756",
    )
    axes[0].axhline(0, color="gray", linestyle="--", linewidth=1)
    axes[0].set_title("Out-of-Sample Mean RankIC")
    axes[0].set_ylabel("RankIC")

    axes[1].bar(
        ["Q5 - Q1"],
        [mean_hml],
        color="#54A24B" if mean_hml > 0 else "#E45756",
    )
    axes[1].axhline(0, color="gray", linestyle="--", linewidth=1)
    axes[1].set_title("Out-of-Sample Mean Q5 - Q1 Return")
    axes[1].set_ylabel("20-Day Gross Return")

    fig.tight_layout()
    fig.savefig(figure_path, dpi=160, bbox_inches="tight")
    plt.show()

    print("===== 样本外研究决策 =====")
    print(decision_log.round(4).to_string(index=False))
    print("已保存：", decision_path)
    print("已保存：", metadata_path)
    print("已保存：", figure_path)

    return {
        "decision_log": decision_log,
        "decision_path": decision_path,
        "metadata_path": metadata_path,
        "figure_path": figure_path,
    }

In [ ]:
import importlib

pru = importlib.reload(pru)
integrated_factor_decision = pru.run_out_of_sample_decision_from_summary(
    project_root=PROJECT_ROOT,
    summary_filename="05_integrated_profitability_effectiveness_summary.csv",
    output_prefix="05_integrated_profitability",
    factor_name="integrated_profitability_structure_proxy_strict",
    decision_scope="low_coverage_structural_sensitivity_only",
)

In [ ]:
#5.14
component_attribution = pdiag.run_common_sample_component_attribution(PROJECT_ROOT)

In [ ]:
#5.15汇总
pdiag = importlib.reload(pdiag)
final_research_synthesis = pdiag.run_profitability_research_synthesis(PROJECT_ROOT)